Modelo funcional sin mejoras

In [ ]:
"""
================================================================================
  SURROGATE MODEL — CALIDAD DEL AIRE  |  Interfaz Web Gradio
  Versión : 10.3
  ─── Correcciones aplicadas ─────────────────────────────────────────────────
  [FIX-1]  feature_importances_ ausente en HistGB algunas versiones de sklearn
           → Reemplazado por RandomForestRegressor como estimador base para
             SelectFromModel (siempre tiene feature_importances_). A prueba
             de fallos: si falla, se usan todas las features disponibles.
  [FIX-2]  Checkboxes no clickeables por conflicto de CSS
           → CSS reescrito. Los checkboxes usan appearance:checkbox nativo,
             sin reglas que oculten el input subyacente.
  [FIX-3]  Eliminado el explorador de archivos locales (Dropdown + botón
           Detectar + botón Refrescar). Solo queda gr.File para subir CSV.
  [FIX-4]  Todos los parámetros de la UI conectados correctamente a entrenar().
  ─── Arquitectura ──────────────────────────────────────────────────────────
  · TimeSeriesSplit(K ajustable) sobre el 80 % de entrenamiento.
  · Test ciego = último 20 %, nunca expuesto al modelo.
  · Sub-split interno 80/20 del bloque train para early stopping de XGB/LGB.
  · Algoritmos: HistGradientBoosting, XGBoost, LightGBM (auto GPU/CPU).
  · Feature Engineering opcional: rolling stats, diffs, interacciones,
    variables cíclicas de día de semana. Selección top-40 con RandomForest.
================================================================================
  Dependencias:
      pip install gradio scikit-learn pandas numpy matplotlib seaborn
      pip install xgboost lightgbm   # opcionales
      pip install torch              # opcional, solo para detección CUDA
================================================================================
"""

# ──────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTACIONES
# ──────────────────────────────────────────────────────────────────────────────
import os
import warnings
import logging
import traceback
import subprocess
import pickle
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

import gradio as gr
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# ──────────────────────────────────────────────────────────────────────────────
# 1.  CONSTANTES
# ──────────────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("resultados_surrogate")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SESION: dict = {
    "modelo":     None,
    "scaler":     None,
    "feat_cols":  [],
    "target":     "",
    "parroquia":  "",
    "feat_stats": {},
}

ANOS_PANDEMIA   = [2020, 2021]
CONTAMINANTES_Y = ["PM25", "PM10", "O3", "CO", "NO2", "SO2"]

FEATURES_X_BASE = [
    "Temperatura", "Humedad", "Viento_Velocidad", "Viento_Direccion", "Precipitacion",
    "hora_sin", "hora_cos", "mes_sin", "mes_cos",
    "PM25_lag_1h",  "PM25_lag_24h",
    "PM10_lag_1h",  "PM10_lag_24h",
    "O3_lag_1h",    "O3_lag_24h",
    "CO_lag_1h",    "CO_lag_24h",
    "NO2_lag_1h",   "NO2_lag_24h",
    "SO2_lag_1h",   "SO2_lag_24h",
]

COLOR_REAL = "#3B82F6"
COLOR_PRED = "#F97316"
COLOR_POS  = "#22C55E"
COLOR_NEG  = "#EF4444"
BG_PLOT    = "#0F172A"
TEXT_PLOT  = "#E2E8F0"
GRID_PLOT  = "#1E293B"

MAX_FEATURES_SEL = 40   # top-N features en Feature Engineering


# ──────────────────────────────────────────────────────────────────────────────
# 2.  DETECCIÓN AUTOMÁTICA DE GPU
# ──────────────────────────────────────────────────────────────────────────────

def detectar_gpu() -> tuple[bool, str]:
    try:
        r = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=8,
        )
        if r.returncode == 0 and r.stdout.strip():
            return True, f"GPU detectada (nvidia-smi): {r.stdout.strip().split(chr(10))[0]}"
    except Exception:
        pass
    try:
        import torch
        if torch.cuda.is_available():
            return True, f"GPU detectada (torch.cuda): {torch.cuda.get_device_name(0)}"
    except ImportError:
        pass
    try:
        import cupy  # noqa: F401
        return True, "GPU detectada (cupy disponible)"
    except ImportError:
        pass
    return False, "No se detectó GPU — se usará CPU"


GPU_DISPONIBLE, GPU_MSG = detectar_gpu()
log.info(GPU_MSG)


def _xgb_tree_method() -> dict:
    return {"tree_method": "hist", "device": "cuda"} if GPU_DISPONIBLE else {"tree_method": "hist", "device": "cpu"}


def _lgbm_device() -> str:
    return "gpu" if GPU_DISPONIBLE else "cpu"


# ──────────────────────────────────────────────────────────────────────────────
# 3.  CARGA Y DETECCIÓN DE TIMESTAMP
# ──────────────────────────────────────────────────────────────────────────────

def _cargar_csv(ruta: str) -> pd.DataFrame:
    try:
        return pd.read_csv(ruta, comment="#", low_memory=False, on_bad_lines="warn")
    except Exception as e:
        raise RuntimeError(f"Error al leer '{ruta}': {e}") from e


def _detectar_timestamp(df: pd.DataFrame) -> str | None:
    keywords = ("time", "fecha", "date", "hora", "datetime", "timestamp")
    candidatos = [c for c in df.columns if any(k in c.lower() for k in keywords)]
    if candidatos:
        return candidatos[0]
    for c in df.columns:
        try:
            pd.to_datetime(df[c].dropna().astype(str).iloc[:10], infer_datetime_format=True)
            return c
        except Exception:
            continue
    return None


# ──────────────────────────────────────────────────────────────────────────────
# 4.  PREPROCESAMIENTO
# ──────────────────────────────────────────────────────────────────────────────

def _preprocesar(df: pd.DataFrame, target_col: str, timestamp_col: str,
                 excluir_pandemia: bool = True) -> pd.DataFrame:
    df = df.copy()
    df[timestamp_col] = pd.to_datetime(
        df[timestamp_col], infer_datetime_format=True, errors="coerce"
    )
    df = df.dropna(subset=[timestamp_col]).set_index(timestamp_col).sort_index()
    if excluir_pandemia:
        df = df[~df.index.year.isin(ANOS_PANDEMIA)]
    obj_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
    if obj_cols:
        df[obj_cols] = df[obj_cols].apply(pd.to_numeric, errors="coerce")
    df = df.dropna(subset=[target_col])
    return df


def _dividir_cronologico(X, y, ratio: float = 0.80):
    """División temporal estricta — el (1-ratio) % final es test ciego."""
    corte = int(len(X) * ratio)
    return X.iloc[:corte], X.iloc[corte:], y.iloc[:corte], y.iloc[corte:]


def _resolver_features_x(df: pd.DataFrame, target_col: str) -> list[str]:
    """Columnas numéricas que no sean contaminantes ni el target."""
    cols = set(df.select_dtypes(include=[np.number]).columns)
    excluir = set(CONTAMINANTES_Y)
    return [c for c in cols if c not in excluir and c != target_col]


# ──────────────────────────────────────────────────────────────────────────────
# 5.  FEATURE ENGINEERING
# ──────────────────────────────────────────────────────────────────────────────

def _agregar_features_avanzadas(df: pd.DataFrame, target_col: str) -> pd.DataFrame:
    """
    Genera rolling stats, diferencias, interacciones y variables cíclicas
    de día de semana. Lógica no modificada respecto a v10.2.
    """
    df = df.copy()
    contaminantes = [c for c in CONTAMINANTES_Y if c in df.columns]

    for col in contaminantes:
        for window in [3, 6, 12, 24]:
            df[f"{col}_roll_mean_{window}h"] = df[col].rolling(window, min_periods=1).mean()
            if window >= 6:
                df[f"{col}_roll_std_{window}h"] = df[col].rolling(window, min_periods=2).std()
        for lag in [1, 3, 6, 12, 24]:
            df[f"{col}_diff_{lag}h"] = df[col].diff(lag)

    if "Temperatura" in df.columns and "Humedad" in df.columns:
        df["temp_hum"] = df["Temperatura"] * df["Humedad"]
    if "Temperatura" in df.columns and "Viento_Velocidad" in df.columns:
        df["temp_wind"] = df["Temperatura"] * df["Viento_Velocidad"]
    if "Viento_Direccion" in df.columns and "Viento_Velocidad" in df.columns:
        dir_rad = np.radians(df["Viento_Direccion"])
        df["wind_u"] = df["Viento_Velocidad"] * np.cos(dir_rad)
        df["wind_v"] = df["Viento_Velocidad"] * np.sin(dir_rad)

    if hasattr(df.index, "weekday"):
        df["dia_semana"]     = df.index.weekday
        df["dia_semana_sin"] = np.sin(2 * np.pi * df["dia_semana"] / 7)
        df["dia_semana_cos"] = np.cos(2 * np.pi * df["dia_semana"] / 7)
        df["es_finde"]       = (df["dia_semana"] >= 5).astype(int)

    return df.dropna()


def _seleccionar_top_features(
    X_tr: pd.DataFrame,
    y_tr: pd.Series,
    n_features: int = MAX_FEATURES_SEL,
    log_fn=None,
) -> list[str]:
    """
    [FIX-1] Selección robusta de las N mejores características.

    Usa RandomForestRegressor (siempre tiene feature_importances_) como
    estimador base de SelectFromModel. Esto evita el fallo de
    HistGradientBoostingRegressor.feature_importances_ que no existe en
    algunas versiones de scikit-learn.

    Fallback: si la selección falla por cualquier razón, devuelve todas
    las columnas disponibles (comportamiento seguro).
    """
    if log_fn is None:
        log_fn = log.info

    all_cols = X_tr.columns.tolist()
    if len(all_cols) <= n_features:
        log_fn(f"  Todas las {len(all_cols)} features disponibles (≤ {n_features}) — sin selección.")
        return all_cols

    try:
        # Usar solo filas sin NaN para el estimador de selección
        mask_ok = X_tr.notna().all(axis=1) & y_tr.notna()
        X_ok = X_tr[mask_ok].values
        y_ok = y_tr[mask_ok].values

        if len(X_ok) < 50:
            log_fn("  ⚠️  Datos insuficientes para selección — usando todas las features.")
            return all_cols

        # RandomForest con n_estimators reducido para rapidez
        rf_sel = RandomForestRegressor(
            n_estimators=80,
            max_depth=6,
            min_samples_leaf=20,
            n_jobs=-1,
            random_state=42,
        )
        rf_sel.fit(X_ok, y_ok)

        # SelectFromModel con threshold=-np.inf fuerza selección por top-N
        selector = SelectFromModel(
            rf_sel,
            threshold=-np.inf,
            max_features=n_features,
            prefit=True,
        )
        selected_mask = selector.get_support()
        selected = [col for col, keep in zip(all_cols, selected_mask) if keep]

        log_fn(f"  SelectFromModel (RandomForest): {len(all_cols)} → {len(selected)} features.")
        return selected if selected else all_cols

    except Exception as exc:
        log_fn(f"  ⚠️  Selección de features falló ({exc}) — usando todas las features.")
        return all_cols


# ──────────────────────────────────────────────────────────────────────────────
# 6.  CONSTRUCCIÓN DEL MODELO
# ──────────────────────────────────────────────────────────────────────────────

def _construir_modelo(algoritmo: str, params: dict | None = None):
    if params is None:
        params = {}

    if "XGBoost" in algoritmo:
        import xgboost as xgb
        return xgb.XGBRegressor(
            n_estimators=2000,
            max_depth=int(params.get("max_depth", 5)),
            learning_rate=float(params.get("learning_rate", 0.05)),
            subsample=float(params.get("subsample", 0.8)),
            colsample_bytree=float(params.get("colsample_bytree", 0.8)),
            reg_lambda=float(params.get("reg_lambda", 2.0)),
            reg_alpha=float(params.get("reg_alpha", 0.5)),
            gamma=float(params.get("gamma", 0.0)),
            min_child_weight=float(params.get("min_child_weight", 1)),
            early_stopping_rounds=50,
            eval_metric="rmse",
            random_state=42,
            verbosity=0,
            **_xgb_tree_method(),
        )

    elif "LightGBM" in algoritmo:
        import lightgbm as lgb
        return lgb.LGBMRegressor(
            n_estimators=2000,
            max_depth=int(params.get("max_depth", 5)),
            learning_rate=float(params.get("learning_rate", 0.05)),
            subsample=float(params.get("subsample", 0.8)),
            colsample_bytree=float(params.get("colsample_bytree", 0.8)),
            reg_lambda=float(params.get("reg_lambda", 2.0)),
            reg_alpha=float(params.get("reg_alpha", 0.5)),
            min_child_samples=int(params.get("min_child_weight", 30)),
            metric="rmse",
            device=_lgbm_device(),
            random_state=42,
            verbose=-1,
        )

    else:  # HistGradientBoosting (default)
        return HistGradientBoostingRegressor(
            max_iter=1000,
            early_stopping=True,
            n_iter_no_change=30,
            validation_fraction=0.1,
            max_depth=int(params.get("max_depth", 5)),
            min_samples_leaf=int(params.get("min_child_weight", 30)),
            learning_rate=float(params.get("learning_rate", 0.05)),
            l2_regularization=float(params.get("reg_lambda", 1.0)),
            random_state=42,
        )


# ──────────────────────────────────────────────────────────────────────────────
# 7.  PIPELINE TimeSeriesSplit
# ──────────────────────────────────────────────────────────────────────────────

def _extraer_curvas_fold(modelo, algoritmo: str) -> tuple[list, list, str]:
    train_hist, val_hist, metric_name = [], [], "Score"
    try:
        if "XGBoost" in algoritmo:
            evals = modelo.evals_result()
            ks = list(evals.keys())
            met = list(evals[ks[0]].keys())[0]
            metric_name = met.upper()
            train_hist  = list(evals[ks[0]][met])
            val_hist    = list(evals[ks[1]][met]) if len(ks) > 1 else []
        elif "LightGBM" in algoritmo:
            evals = modelo.evals_result_
            ks = list(evals.keys())
            met = list(evals[ks[0]].keys())[0]
            metric_name = met.upper()
            train_hist  = list(evals[ks[0]][met])
            val_hist    = list(evals[ks[1]][met]) if len(ks) > 1 else []
        else:
            if hasattr(modelo, "train_score_") and modelo.train_score_ is not None:
                train_hist  = list(modelo.train_score_)
                metric_name = "R²"
            if hasattr(modelo, "validation_score_") and modelo.validation_score_ is not None:
                val_hist = list(modelo.validation_score_)
    except Exception as exc:
        log.warning(f"_extraer_curvas_fold: {exc}")
    return train_hist, val_hist, metric_name


def _entrenar_kfold(
    df: pd.DataFrame,
    algoritmo: str,
    n_splits: int = 5,
    log_fn=None,
    xgb_params: dict | None = None,
) -> tuple[pd.DataFrame, dict]:
    if log_fn is None:
        log_fn = log.info

    tscv       = TimeSeriesSplit(n_splits=n_splits)
    filas      = []
    cols_df    = set(df.columns)
    all_curves = {}

    for contaminante in CONTAMINANTES_Y:
        if contaminante not in cols_df:
            log_fn(f"  ⏭️  {contaminante} no disponible — omitido.")
            continue

        feat_cols = _resolver_features_x(df, contaminante)
        y_vals    = df[contaminante].dropna()
        X_vals    = df.loc[y_vals.index, feat_cols]
        mask_ok   = X_vals.notna().all(axis=1) & y_vals.notna()
        X_vals, y_vals = X_vals[mask_ok], y_vals[mask_ok]

        if len(X_vals) < n_splits * 20:
            log_fn(f"  ⚠️  {contaminante}: datos insuficientes ({len(X_vals)} filas) — omitido.")
            continue

        X_arr = X_vals.values
        y_arr = y_vals.values

        fold_metrics   = []
        fold_curves_tr = []
        fold_curves_va = []
        metric_name_cv = "Score"

        for fold_idx, (tr_idx, va_idx) in enumerate(tscv.split(X_arr), 1):
            X_tr, X_va = X_arr[tr_idx], X_arr[va_idx]
            y_tr, y_va = y_arr[tr_idx], y_arr[va_idx]

            scaler  = StandardScaler()
            X_tr_sc = scaler.fit_transform(X_tr)
            X_va_sc = scaler.transform(X_va)

            modelo = _construir_modelo(algoritmo, xgb_params)

            if "XGBoost" in algoritmo:
                modelo.fit(
                    X_tr_sc, y_tr,
                    eval_set=[(X_tr_sc, y_tr), (X_va_sc, y_va)],
                    verbose=False,
                )
            elif "LightGBM" in algoritmo:
                import lightgbm as lgb
                modelo.fit(
                    X_tr_sc, y_tr,
                    eval_set=[(X_tr_sc, y_tr), (X_va_sc, y_va)],
                    callbacks=[
                        lgb.early_stopping(30, verbose=False),
                        lgb.log_evaluation(-1),
                    ],
                )
            else:
                modelo.fit(X_tr_sc, y_tr)

            tr_h, va_h, met = _extraer_curvas_fold(modelo, algoritmo)
            if tr_h:
                fold_curves_tr.append(tr_h)
            if va_h:
                fold_curves_va.append(va_h)
            metric_name_cv = met

            y_pred = modelo.predict(X_va_sc)
            fold_metrics.append({
                "MAE":  mean_absolute_error(y_va, y_pred),
                "RMSE": float(np.sqrt(mean_squared_error(y_va, y_pred))),
                "R2":   r2_score(y_va, y_pred),
            })
            log_fn(
                f"  {contaminante} | Fold {fold_idx}/{n_splits} → "
                f"MAE={fold_metrics[-1]['MAE']:.3f}  "
                f"RMSE={fold_metrics[-1]['RMSE']:.3f}  "
                f"R²={fold_metrics[-1]['R2']:.3f}"
            )

        all_curves[contaminante] = {
            "train": fold_curves_tr,
            "val":   fold_curves_va,
            "metric": metric_name_cv,
        }

        mf = pd.DataFrame(fold_metrics)
        filas.append({
            "Contaminante": contaminante,
            "Features_X":   len(feat_cols),
            "MAE_mean":     mf["MAE"].mean(),
            "MAE_std":      mf["MAE"].std(),
            "RMSE_mean":    mf["RMSE"].mean(),
            "RMSE_std":     mf["RMSE"].std(),
            "R2_mean":      mf["R2"].mean(),
            "R2_std":       mf["R2"].std(),
        })

    df_out = pd.DataFrame(filas) if filas else pd.DataFrame()
    return df_out, all_curves


# ──────────────────────────────────────────────────────────────────────────────
# 8.  FIGURAS
# ──────────────────────────────────────────────────────────────────────────────

def _aplicar_estilo_ax(ax, titulo: str, xlabel: str, ylabel: str) -> None:
    ax.set_title(titulo, fontsize=11, fontweight="bold", color=TEXT_PLOT, pad=12)
    if xlabel:
        ax.set_xlabel(xlabel, color=TEXT_PLOT, fontsize=10)
    if ylabel:
        ax.set_ylabel(ylabel, color=TEXT_PLOT, fontsize=10)
    ax.tick_params(colors=TEXT_PLOT, labelsize=9)
    ax.grid(True, linestyle="--", alpha=0.18, color=TEXT_PLOT)
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)


def _fig_prediccion(y_test, y_pred, target_col: str, days: int, parroquia: str = ""):
    df_p = pd.DataFrame({"Real": y_test.values, "Predicho": y_pred}, index=y_test.index)
    df_p = df_p[df_p.index >= df_p.index.max() - pd.Timedelta(days=days)]

    titulo = f"Surrogate Model — {target_col}  ·  Últimos {days} días (Test)"
    if parroquia:
        titulo = f"[{parroquia}]  {titulo}"

    fig, ax = plt.subplots(figsize=(13, 4.5), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    ax.plot(df_p.index, df_p["Real"],     label="Real",     color=COLOR_REAL, lw=1.8, alpha=0.95)
    ax.plot(df_p.index, df_p["Predicho"], label="Predicho", color=COLOR_PRED, lw=1.5,
            linestyle="--", alpha=0.90)
    ax.fill_between(df_p.index, df_p["Real"], df_p["Predicho"], alpha=0.07, color=COLOR_PRED)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    ax.xaxis.set_major_locator(mdates.DayLocator())
    plt.xticks(rotation=28, ha="right", color=TEXT_PLOT, fontsize=9)
    plt.yticks(color=TEXT_PLOT, fontsize=9)
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)
    ax.set_title(titulo, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=12)
    ax.set_ylabel(target_col, color=TEXT_PLOT)
    ax.set_xlabel("Fecha", color=TEXT_PLOT)
    ax.grid(True, linestyle="--", alpha=0.18, color=TEXT_PLOT)
    ax.legend(framealpha=0.15, labelcolor=TEXT_PLOT, facecolor=BG_PLOT,
              edgecolor=GRID_PLOT, fontsize=10)
    plt.tight_layout()
    return fig


def _fig_feature_importance(fi_df, target_col: str, parroquia: str = ""):
    n = len(fi_df)
    fig, ax = plt.subplots(figsize=(9, max(4, n * 0.52)), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    colores = [COLOR_POS if v >= 0 else COLOR_NEG for v in fi_df["Importance"]]
    ax.barh(
        fi_df["Feature"][::-1], fi_df["Importance"][::-1],
        xerr=fi_df["Std"][::-1], color=colores[::-1],
        align="center", alpha=0.85, ecolor="#94A3B8", capsize=3, height=0.65,
    )
    ax.axvline(0, color="#475569", linewidth=0.9, linestyle="--")
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_PLOT)
    titulo_fi = f"Feature Importance  ·  {target_col}  (Permutation Δ R²)"
    if parroquia:
        titulo_fi = f"[{parroquia}]  {titulo_fi}"
    plt.yticks(color=TEXT_PLOT, fontsize=9)
    plt.xticks(color=TEXT_PLOT, fontsize=9)
    ax.set_title(titulo_fi, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=12)
    ax.set_xlabel("Importancia media (Δ R²)", color=TEXT_PLOT)
    ax.grid(True, axis="x", linestyle="--", alpha=0.14, color=TEXT_PLOT)
    plt.tight_layout()
    return fig


def _fig_heatmap(df_full, target_col: str, parroquia: str = ""):
    num_df = df_full.select_dtypes(include=[np.number])
    if num_df.shape[1] < 2:
        return None
    corr = num_df.corr(method="pearson")
    n    = len(corr)
    fig, ax = plt.subplots(figsize=(max(8, n * 0.60), max(7, n * 0.55)), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    mask = np.zeros_like(corr, dtype=bool)
    mask[np.triu_indices_from(mask, k=1)] = True
    cmap = sns.diverging_palette(230, 20, as_cmap=True)
    sns.heatmap(
        corr, mask=mask, cmap=cmap, vmin=-1, vmax=1, center=0,
        annot=True, fmt=".2f", annot_kws={"size": 7.5, "color": TEXT_PLOT},
        linewidths=0.4, linecolor=GRID_PLOT, square=True, ax=ax,
        cbar_kws={"shrink": 0.7},
    )
    if target_col in corr.columns:
        idx = list(corr.columns).index(target_col)
        ax.add_patch(plt.Rectangle((idx, 0), 1, n, fill=False,
                                   edgecolor="#F97316", lw=2.5, clip_on=False))
        ax.add_patch(plt.Rectangle((0, idx), n, 1, fill=False,
                                   edgecolor="#F97316", lw=2.5, clip_on=False))
    titulo_hm = f"Correlación de Pearson — {target_col}"
    if parroquia:
        titulo_hm = f"[{parroquia}]  {titulo_hm}"
    ax.set_title(titulo_hm, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=14)
    ax.tick_params(colors=TEXT_PLOT, labelsize=8)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    plt.setp(ax.get_yticklabels(), rotation=0)
    ax.collections[0].colorbar.ax.tick_params(colors=TEXT_PLOT, labelsize=8)
    plt.tight_layout()
    return fig


def _fig_curva_aprendizaje(
    curvas: dict, target_col: str, algoritmo: str, parroquia: str = "", k_splits: int = 5
):
    fig, ax = plt.subplots(figsize=(13, 5), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)

    if not curvas or not curvas.get("train"):
        ax.text(0.5, 0.5, "No hay datos de curva de aprendizaje.",
                ha="center", va="center", color=TEXT_PLOT, fontsize=11, transform=ax.transAxes)
        _aplicar_estilo_ax(ax, f"Curva de Aprendizaje — {target_col}", "", "")
        plt.tight_layout()
        return fig

    train_curves = curvas["train"]
    val_curves   = curvas.get("val", [])
    metric_name  = curvas.get("metric", "Score")
    is_r2        = metric_name == "R²"

    all_raw = train_curves + (val_curves if val_curves else [])
    lengths = [len(c) for c in all_raw if c]
    if not lengths or min(lengths) < 2:
        ax.text(0.5, 0.5, "Historial demasiado corto.",
                ha="center", va="center", color=TEXT_PLOT, transform=ax.transAxes)
        _aplicar_estilo_ax(ax, f"Curva de Aprendizaje — {target_col}", "", "")
        plt.tight_layout()
        return fig

    min_len  = min(lengths)
    x        = np.arange(min_len)
    tr_arr   = np.array([c[:min_len] for c in train_curves])
    tr_mean  = tr_arr.mean(axis=0)
    tr_std   = tr_arr.std(axis=0)

    for curve in tr_arr:
        ax.plot(x, curve, color=COLOR_REAL, alpha=0.10, lw=0.75, zorder=2)
    ax.plot(x, tr_mean, color=COLOR_REAL, lw=2.2,
            label=f"Train — {metric_name}  (μ TSS-Fold)", zorder=5)
    ax.fill_between(x, tr_mean - tr_std, tr_mean + tr_std,
                    alpha=0.14, color=COLOR_REAL, zorder=3)

    if val_curves:
        va_arr  = np.array([c[:min_len] for c in val_curves])
        va_mean = va_arr.mean(axis=0)
        va_std  = va_arr.std(axis=0)
        for curve in va_arr:
            ax.plot(x, curve, color=COLOR_PRED, alpha=0.10, lw=0.75, zorder=2)
        ax.plot(x, va_mean, color=COLOR_PRED, lw=2.2, linestyle="--",
                label=f"Validación — {metric_name}  (μ TSS-Fold)", zorder=5)
        ax.fill_between(x, va_mean - va_std, va_mean + va_std,
                        alpha=0.14, color=COLOR_PRED, zorder=3)

        best_iter = int(np.argmax(va_mean) if is_r2 else np.argmin(va_mean))
        best_val  = va_mean[best_iter]
        ax.axvline(best_iter, color="#F59E0B", lw=1.6, linestyle=":", alpha=0.88,
                   label=f"Mejor iteración: {best_iter}  ({metric_name}={best_val:.4f})",
                   zorder=6)
        offset = max(1, int(min_len * 0.02))
        ax.annotate(f" iter {best_iter}", xy=(best_iter, best_val),
                    xytext=(best_iter + offset, best_val),
                    color="#F59E0B", fontsize=8.5, va="center", zorder=7)

    ax.axhline(tr_mean[-1], color=COLOR_REAL, lw=0.8, linestyle=":", alpha=0.40, zorder=1)

    algo_label = algoritmo.split("(")[0].strip()
    titulo = (
        f"Curva de Aprendizaje — {target_col}  ·  "
        f"{algo_label}  ·  TimeSeriesSplit (K={k_splits})"
    )
    if parroquia:
        titulo = f"[{parroquia}]  {titulo}"

    ylabel = "R²  (↑ mejor)" if is_r2 else f"{metric_name}  (↓ mejor)"
    nota   = (
        "Nota HGB: Validación = fracción interna 10% de early stopping."
        if is_r2 else
        "Nota: banda sombreada = ±1σ entre pliegues temporales (TimeSeriesSplit)."
    )

    _aplicar_estilo_ax(ax, titulo, "Iteración / Época (Boosting Round)", ylabel)
    ax.legend(framealpha=0.20, labelcolor=TEXT_PLOT, facecolor=BG_PLOT,
              edgecolor=GRID_PLOT, fontsize=9, loc="best")
    fig.text(0.012, 0.012, nota, fontsize=7.5, color="#64748B", ha="left", va="bottom")
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    return fig


# ──────────────────────────────────────────────────────────────────────────────
# 9.  PIPELINE PRINCIPAL
# ──────────────────────────────────────────────────────────────────────────────

def entrenar(
    csv_upload,           # gr.File — único origen de datos [FIX-3]
    target_col: str,
    nombre_modelo: str,
    algoritmo: str,
    plot_days: int,
    train_ratio: float,
    excluir_pandemia: bool,
    k_splits: int,
    max_depth: float,
    learning_rate: float,
    subsample: float,
    colsample_bytree: float,
    reg_lambda: float,
    reg_alpha: float,
    gamma: float,
    min_child_weight: float,
    usar_feature_engineering: bool,
):
    logs: list[str] = []

    def info(m): log.info(m);    logs.append(f"✅ {m}")
    def warn(m): log.warning(m); logs.append(f"⚠️  {m}")
    def err(m):  log.error(m);   logs.append(f"❌ {m}")
    def _estado(): return "**Registro:**  " + "  ·  ".join(logs[-30:])

    VACIO = (None, None, None, None)

    try:
        # ── 9.1  Resolver archivo ─────────────────────────────────────────────
        if csv_upload is None:
            err("No se subió ningún archivo CSV.")
            return "### ❌ Sube un archivo CSV antes de entrenar.", *VACIO, _estado()

        ruta_csv = csv_upload if isinstance(csv_upload, str) else csv_upload.name

        if not Path(ruta_csv).exists():
            err(f"Archivo no encontrado: {ruta_csv}")
            return f"### ❌ Archivo no encontrado: `{ruta_csv}`", *VACIO, _estado()

        target_col    = target_col.strip()
        nombre_modelo = nombre_modelo.strip() or "surrogate_model"
        parroquia     = Path(ruta_csv).stem.replace("_", " ").title()
        info(f"Archivo: {Path(ruta_csv).name}")

        # ── 9.2  Carga ────────────────────────────────────────────────────────
        df = _cargar_csv(ruta_csv)
        info(f"CSV cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")

        ts_col = _detectar_timestamp(df)
        if ts_col is None:
            err("No se detectó columna de tiempo.")
            return "### ❌ No se encontró columna Timestamp/Date/Fecha.", *VACIO, _estado()
        info(f"Timestamp: '{ts_col}'")

        if target_col not in df.columns:
            cols_disp = ", ".join(df.columns.tolist())
            err(f"Target '{target_col}' no existe.")
            return (
                f"### ❌ Target **`{target_col}`** no encontrado.\n\n"
                f"**Columnas disponibles:** `{cols_disp}`",
                *VACIO, _estado()
            )

        # ── 9.3  Preprocesamiento ─────────────────────────────────────────────
        df_prep = _preprocesar(df, target_col, ts_col, excluir_pandemia)
        if excluir_pandemia:
            info(f"Pandemia excluida: años {ANOS_PANDEMIA} eliminados.")

        y = df_prep[target_col]
        X = df_prep.drop(columns=[target_col])

        # ── 9.4  Feature Engineering + selección robusta ──────────────────────
        if usar_feature_engineering:
            info("Aplicando Feature Engineering avanzado…")
            df_full = _agregar_features_avanzadas(pd.concat([X, y], axis=1), target_col)
            y = df_full[target_col]
            X = df_full.drop(columns=[target_col])

            # Usar el bloque train para seleccionar features (sin ver el test)
            X_tr_sel, _, y_tr_sel, _ = _dividir_cronologico(X, y, train_ratio)
            info(f"Seleccionando top-{MAX_FEATURES_SEL} features entre {X_tr_sel.shape[1]}…")

            selected = _seleccionar_top_features(X_tr_sel, y_tr_sel, MAX_FEATURES_SEL, log_fn=info)
            X = X[selected]
            feat_cols = list(X.columns)
            info(f"Features seleccionadas: {len(feat_cols)}")
        else:
            num_cols  = X.select_dtypes(include=[np.number]).columns.tolist()
            feat_cols = [c for c in num_cols if c != target_col]
            X         = X[feat_cols]
            info(f"Usando {len(feat_cols)} features originales.")

        # ── 9.5  Limpiar NaN ──────────────────────────────────────────────────
        n_nulos = int(X.isna().sum().sum())
        if n_nulos:
            warn(f"{n_nulos:,} NaN en features — se eliminan filas afectadas.")
        mask = X.notna().all(axis=1)
        X    = X.loc[mask]
        y    = y.loc[mask]

        df_clean = pd.concat([X, y], axis=1)
        info(f"Dataset limpio: {df_clean.shape[0]:,} filas × {df_clean.shape[1]} columnas")

        # ── 9.6  Hiperparámetros ──────────────────────────────────────────────
        xgb_params = {
            "max_depth":       int(max_depth),
            "learning_rate":   float(learning_rate),
            "subsample":       float(subsample),
            "colsample_bytree": float(colsample_bytree),
            "reg_lambda":      float(reg_lambda),
            "reg_alpha":       float(reg_alpha),
            "gamma":           float(gamma),
            "min_child_weight": float(min_child_weight),
        }
        info("Hiperparámetros: " + ", ".join(f"{k}={v}" for k, v in xgb_params.items()))

        # ── 9.7  TimeSeriesSplit K-Fold ───────────────────────────────────────
        info(f"TimeSeriesSplit (K={k_splits}) · Algoritmo: {algoritmo}")
        info(f"GPU disponible: {GPU_DISPONIBLE} — {GPU_MSG}")

        kfold_logs = []
        def kf_log(m): log.info(m); kfold_logs.append(m)

        kf_resumen, kf_curvas = _entrenar_kfold(
            df_clean, algoritmo, n_splits=k_splits,
            log_fn=kf_log, xgb_params=xgb_params,
        )
        for l in kfold_logs:
            logs.append(l)

        if kf_resumen.empty:
            warn("TSS K-Fold no produjo resultados.")

        fig_lc = _fig_curva_aprendizaje(
            kf_curvas.get(target_col, {}), target_col, algoritmo, parroquia, k_splits
        )

        # ── 9.8  Modelo final con sub-split interno ───────────────────────────
        X_tr, X_te, y_tr, y_te = _dividir_cronologico(X, y, train_ratio)
        sub_corte  = int(len(X_tr) * 0.80)
        X_sub_tr   = X_tr.iloc[:sub_corte]
        X_val_sub  = X_tr.iloc[sub_corte:]
        y_sub_tr   = y_tr.iloc[:sub_corte]
        y_val_sub  = y_tr.iloc[sub_corte:]

        scaler_final  = StandardScaler()
        X_sub_tr_sc   = scaler_final.fit_transform(X_sub_tr)
        X_val_sub_sc  = scaler_final.transform(X_val_sub)
        X_te_sc       = scaler_final.transform(X_te)
        X_tr_sc       = np.vstack([X_sub_tr_sc, X_val_sub_sc])
        y_tr_np       = y_tr.values

        modelo_final = _construir_modelo(algoritmo, xgb_params)

        if "XGBoost" in algoritmo:
            modelo_final.fit(
                X_sub_tr_sc, y_sub_tr.values,
                eval_set=[(X_val_sub_sc, y_val_sub.values)],
                verbose=False,
            )
        elif "LightGBM" in algoritmo:
            import lightgbm as lgb
            modelo_final.fit(
                X_sub_tr_sc, y_sub_tr.values,
                eval_set=[(X_val_sub_sc, y_val_sub.values)],
                callbacks=[
                    lgb.early_stopping(30, verbose=False),
                    lgb.log_evaluation(-1),
                ],
            )
        else:
            modelo_final.fit(X_sub_tr_sc, y_sub_tr.values)

        y_pred_tr = modelo_final.predict(X_tr_sc)
        y_pred_te = modelo_final.predict(X_te_sc)

        # ── 9.9  Métricas ─────────────────────────────────────────────────────
        def _m(yt, yp):
            return dict(
                MAE  = mean_absolute_error(yt, yp),
                RMSE = float(np.sqrt(mean_squared_error(yt, yp))),
                R2   = r2_score(yt, yp),
            )

        m_tr = _m(y_tr_np, y_pred_tr)
        m_te = _m(y_te.values, y_pred_te)
        gap  = m_tr["R2"] - m_te["R2"]
        if gap > 0.15:
            warn(f"Posible overfitting: ΔR² = {gap:.3f}")

        # ── 9.10  Markdown métricas ────────────────────────────────────────────
        hw_label       = f"{'🟢 GPU' if GPU_DISPONIBLE else '🔵 CPU'} — {GPU_MSG}"
        pandemia_label = "🚫 2020-2021 excluidos" if excluir_pandemia else "⚠️ Pandemia incluida"

        if not kf_resumen.empty:
            filas_kf = []
            for _, row in kf_resumen.iterrows():
                ico = "🟢" if row["R2_mean"] >= 0.7 else ("🟡" if row["R2_mean"] >= 0.5 else "🔴")
                filas_kf.append(
                    f"| **{row['Contaminante']}** "
                    f"| `{int(row['Features_X'])}` "
                    f"| `{row['MAE_mean']:.3f} ± {row['MAE_std']:.3f}` "
                    f"| `{row['RMSE_mean']:.3f} ± {row['RMSE_std']:.3f}` "
                    f"| {ico} `{row['R2_mean']:.3f} ± {row['R2_std']:.3f}` |"
                )
            tabla_kf = "\n".join(filas_kf)
        else:
            tabla_kf = "| — | — | — | — | — |"

        metricas_md = f"""
## 📊 Surrogate Model v10.3 — *{parroquia}*

### Validación Cronológica TimeSeriesSplit (K={k_splits}) — 6 Contaminantes

| Contaminante | Features X | MAE (μ ± σ) | RMSE (μ ± σ) | R² (μ ± σ) |
|:------------:|:----------:|:-----------:|:------------:|:----------:|
{tabla_kf}

> `TimeSeriesSplit` garantiza que cada fold de validación sea **posterior** al bloque de entrenamiento.

---

### Modelo Final — `{target_col}` (Split {int(train_ratio*100)}/{int((1-train_ratio)*100)})

> **Test set 100 % ciego** — el {int((1-train_ratio)*100)} % final nunca fue expuesto al modelo.

| Métrica | 🟦 Train | 🟧 Test |
|---------|:--------:|:-------:|
| **MAE** | `{m_tr['MAE']:.4f}` | `{m_te['MAE']:.4f}` |
| **RMSE** | `{m_tr['RMSE']:.4f}` | `{m_te['RMSE']:.4f}` |
| **R²** | `{m_tr['R2']:.4f}` | `{m_te['R2']:.4f}` |
| **Precisión (R² %)** | `{m_tr['R2']*100:.2f}%` | `{m_te['R2']*100:.2f}%` |

{"⚠️ **Posible overfitting** — ΔR² = `" + f"{gap:.3f}`" if gap > 0.15 else "✅ Sin señales de overfitting."}

| Parámetro | Valor |
|-----------|-------|
| Parroquia | `{parroquia}` |
| Algoritmo | `{algoritmo}` |
| Hardware | {hw_label} |
| Features | `{len(feat_cols)}` columnas |
| Sub-train / Val interno / Test ciego | `{len(X_sub_tr):,}` / `{len(X_val_sub):,}` / `{len(X_te):,}` |
| Pandemia | {pandemia_label} |
| Feature Engineering | {'✅ Activado (top-' + str(MAX_FEATURES_SEL) + ')' if usar_feature_engineering else '❌ Desactivado'} |
| Regularización | L2={xgb_params['reg_lambda']}  α={xgb_params['reg_alpha']}  depth={xgb_params['max_depth']} |
"""

        # ── 9.11  Feature Importance ──────────────────────────────────────────
        info("Calculando Permutation Importance en test ciego…")
        perm  = permutation_importance(
            modelo_final, X_te_sc, y_te.values,
            n_repeats=8, random_state=42, scoring="r2",
        )
        fi_df = (
            pd.DataFrame({
                "Feature":    feat_cols,
                "Importance": perm.importances_mean,
                "Std":        perm.importances_std,
            })
            .sort_values("Importance", ascending=False)
            .reset_index(drop=True)
        )

        # ── 9.12  Persistencia ────────────────────────────────────────────────
        tag      = f"{nombre_modelo}_{parroquia.replace(' ', '_')}"
        pkl_path = OUTPUT_DIR / f"{tag}.pkl"
        with open(pkl_path, "wb") as fh:
            pickle.dump({
                "modelo":       modelo_final,
                "scaler":       scaler_final,
                "features":     feat_cols,
                "target":       target_col,
                "parroquia":    parroquia,
                "kfold_resumen": kf_resumen,
                "kf_curvas":    kf_curvas,
            }, fh)
        info(f"Modelo guardado: {pkl_path}")

        fi_df.to_csv(OUTPUT_DIR / f"{tag}_feature_importance.csv", index=False)
        if not kf_resumen.empty:
            kf_resumen.to_csv(OUTPUT_DIR / f"{tag}_kfold_resumen.csv", index=False)
        pd.DataFrame([{
            "Parroquia": parroquia, "Archivo": ruta_csv, "Target": target_col,
            **{f"Train_{k}": v for k, v in m_tr.items()},
            **{f"Test_{k}":  v for k, v in m_te.items()},
            "GPU": GPU_DISPONIBLE, "Algoritmo": algoritmo,
        }]).to_csv(OUTPUT_DIR / f"{tag}_metricas.csv", index=False)

        # ── 9.13  Actualizar sesión ───────────────────────────────────────────
        SESION.update({
            "modelo":     modelo_final,
            "scaler":     scaler_final,
            "feat_cols":  feat_cols,
            "target":     target_col,
            "parroquia":  parroquia,
            "feat_stats": {
                col: {
                    "min":  float(X[col].min()),
                    "max":  float(X[col].max()),
                    "mean": float(X[col].mean()),
                }
                for col in feat_cols
            },
        })
        info("Sesión actualizada → pestaña Predicción lista.")

        # ── 9.14  Figuras ─────────────────────────────────────────────────────
        y_te_series = pd.Series(y_te.values, index=X_te.index, name=target_col)
        fig_pred    = _fig_prediccion(y_te_series, y_pred_te, target_col, plot_days, parroquia)
        fig_fi      = _fig_feature_importance(fi_df, target_col, parroquia)
        fig_hm      = _fig_heatmap(pd.concat([X, y], axis=1), target_col, parroquia)
        info("Figuras generadas.")

        return metricas_md, fig_pred, fig_fi, fig_hm, fig_lc, _estado()

    except ImportError as e:
        err(str(e))
        pkg = str(e).split("'")[-2] if "'" in str(e) else str(e).split()[-1]
        return (
            f"### ❌ Librería faltante\n```\n{e}\n```\nInstala con: `pip install {pkg}`",
            *VACIO, _estado()
        )
    except Exception as e:
        err(str(e))
        return (
            f"### ❌ Error\n```\n{traceback.format_exc()}\n```",
            *VACIO, _estado()
        )


# ──────────────────────────────────────────────────────────────────────────────
# 10.  PREDICCIÓN DESDE SESIÓN
# ──────────────────────────────────────────────────────────────────────────────

def predecir_desde_sesion(valores_json: str) -> str:
    if SESION["modelo"] is None:
        return "### ⚠️ No hay modelo entrenado.\nEntrena primero un modelo."
    try:
        vals      = json.loads(valores_json)
        feat_cols = SESION["feat_cols"]
        row       = {col: float(vals.get(col, SESION["feat_stats"][col]["mean"]))
                     for col in feat_cols}
        X_input   = pd.DataFrame([row])
        scaler    = SESION.get("scaler")
        X_sc      = scaler.transform(X_input) if scaler else X_input.values
        pred      = float(SESION["modelo"].predict(X_sc)[0])
        target    = SESION["target"]
        parroquia = SESION["parroquia"]

        extra = ""
        if target in ("PM25", "PM2.5"):
            if pred <= 12:    cal = "🟢 **Buena**"
            elif pred <= 35:  cal = "🟡 **Moderada**"
            elif pred <= 55:  cal = "🟠 **Insalubre GS**"
            elif pred <= 150: cal = "🔴 **Insalubre**"
            else:             cal = "🟣 **Muy insalubre**"
            extra = f"\n\n**Índice AQI:** {cal}"

        return (
            f"## 🔮 Predicción — `{target}` · *{parroquia}*\n\n"
            f"| Campo | Valor |\n|-------|-------|\n"
            f"| **{target} estimado** | `{pred:.3f} µg/m³` |\n"
            f"| Parroquia | `{parroquia}` |\n"
            f"| Features usadas | `{len(feat_cols)}` |"
            f"{extra}"
        )
    except Exception:
        return f"### ❌ Error\n```\n{traceback.format_exc()}\n```"


# ──────────────────────────────────────────────────────────────────────────────
# 11.  CSS  [FIX-2]  Checkboxes funcionales + tema oscuro
# ──────────────────────────────────────────────────────────────────────────────
#
# Regla clave para los checkboxes:
#   · NO se oculta el <input type="checkbox"> nativo (no hay visibility:hidden
#     ni display:none sobre él).
#   · Se fuerza -webkit-appearance/appearance: checkbox para que el navegador
#     lo dibuje visualmente.
#   · El contenedor label usa display:flex para alinear el tick y el texto.
# ──────────────────────────────────────────────────────────────────────────────

CSS = """
/* ── Variables del tema ── */
:root {
    --bg:      #0F172A;
    --card:    #1E293B;
    --input:   #0D1525;
    --border:  #334155;
    --accent:  #6366F1;
    --accentH: #818CF8;
    --text:    #F1F5F9;
    --muted:   #94A3B8;
    --ok:      #22C55E;
    --warn:    #F59E0B;
    --err:     #EF4444;
    --r:       10px;
}

/* ── Layout general ── */
body, .gradio-container {
    background: var(--bg) !important;
    color: var(--text) !important;
    font-family: 'Inter', 'Segoe UI', sans-serif !important;
}
.gr-group, .gr-box {
    background: var(--card) !important;
    border: 1px solid var(--border) !important;
    border-radius: var(--r) !important;
    padding: 16px !important;
}

/* ── Inputs de texto / select ── */
input[type="text"],
input[type="number"],
input[type="email"],
textarea,
select {
    background: var(--input) !important;
    color: var(--text) !important;
    border: 1px solid var(--border) !important;
    border-radius: 6px !important;
}

/* ── Labels ── */
label, .gr-label {
    color: var(--muted) !important;
    font-size: .8rem !important;
    font-weight: 700 !important;
    text-transform: uppercase;
    letter-spacing: .05em !important;
}

/* ── [FIX-2] Checkboxes: apariencia nativa, sin ocultar el input ── */
input[type="checkbox"] {
    -webkit-appearance: checkbox !important;
    appearance:         checkbox !important;
    width:     18px !important;
    height:    18px !important;
    min-width: 18px !important;
    margin:    0 8px 0 0 !important;
    cursor:    pointer !important;
    vertical-align: middle !important;
    /* Acento azul-índigo cuando está marcado */
    accent-color: var(--accent);
}
/* El label del checkbox debe ser flex para que el tick y el texto se alineen */
.gr-checkbox > label,
.gr-checkbox label,
[data-testid="checkbox"] label {
    display:     flex !important;
    align-items: center !important;
    color:       var(--text) !important;
    font-size:   .9rem !important;
    font-weight: 500 !important;
    text-transform: none !important;
    cursor: pointer !important;
    gap: 4px;
}

/* ── Botones ── */
button.primary {
    background: linear-gradient(135deg, var(--accent), #7C3AED) !important;
    color: #fff !important;
    border: none !important;
    border-radius: 8px !important;
    font-weight: 800 !important;
    font-size: 1rem !important;
    padding: 12px 28px !important;
    box-shadow: 0 4px 20px rgba(99,102,241,.45);
    transition: all .15s;
}
button.primary:hover {
    transform: translateY(-2px);
    box-shadow: 0 6px 28px rgba(99,102,241,.6);
}
button.secondary {
    background: var(--card) !important;
    color: var(--text) !important;
    border: 1px solid var(--border) !important;
    border-radius: 8px !important;
}

/* ── Markdown / tablas ── */
.gr-markdown { color: var(--text) !important; }
.gr-markdown table { border-collapse: collapse; width: 100%; }
.gr-markdown th {
    background: #1E293B;
    color: var(--accentH);
    padding: 8px 14px;
    border: 1px solid var(--border);
}
.gr-markdown td {
    color: var(--text);
    padding: 7px 14px;
    border: 1px solid var(--border);
}
.gr-markdown tr:nth-child(even) td { background: #19253a; }

/* ── Área de carga de archivo ── */
.gr-file {
    border: 2px dashed var(--accent) !important;
    border-radius: var(--r) !important;
    background: rgba(99,102,241,.04) !important;
}

/* ── Hero ── */
.hero { text-align: center; padding: 24px 0 6px; }
.hero h1 {
    font-size: 2rem; font-weight: 900;
    background: linear-gradient(90deg,#6366F1,#38BDF8);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}
.hero p { color: var(--muted); font-size: .88rem; }

/* ── Badge GPU ── */
.gpu-badge {
    display: inline-block;
    padding: 4px 12px;
    border-radius: 20px;
    font-size: .75rem;
    font-weight: 700;
    margin-top: 4px;
}
.gpu-on  { background: rgba(34,197,94,.18);  color: #4ADE80; border: 1px solid #22C55E; }
.gpu-off { background: rgba(99,102,241,.15); color: #A5B4FC; border: 1px solid #6366F1; }

/* ── Caja de logs ── */
.logs-box {
    background: #0B1527 !important;
    border: 1px solid #1D3557 !important;
    border-radius: 8px;
    padding: 10px 14px;
    font-family: 'JetBrains Mono', monospace;
    font-size: .76rem;
    color: #7DD3FC;
    max-height: 140px;
    overflow-y: auto;
    line-height: 1.6;
}

/* ── Sección Feature Engineering ── */
.fe-badge {
    background: rgba(99,102,241,.10);
    border: 1px solid #4338CA;
    border-radius: 8px;
    padding: 8px 14px;
    font-size: .78rem;
    color: #C7D2FE;
    margin-top: 4px;
}
"""


# ──────────────────────────────────────────────────────────────────────────────
# 12.  UI — rediseño limpio, sin explorador de archivos locales [FIX-3]
# ──────────────────────────────────────────────────────────────────────────────

MODELOS = [
    "HistGradientBoosting (CPU — sin GPU requerida)",
    "XGBoost  (auto GPU/CPU)",
    "LightGBM (auto GPU/CPU)",
]

gpu_badge_html = (
    f'<span class="gpu-badge gpu-on">🟢 GPU: {GPU_MSG.split(": ")[-1]}</span>'
    if GPU_DISPONIBLE else
    f'<span class="gpu-badge gpu-off">🔵 CPU mode — {GPU_MSG}</span>'
)


def construir_app() -> gr.Blocks:
    with gr.Blocks(
        theme=gr.themes.Base(
            primary_hue="indigo",
            secondary_hue="sky",
            neutral_hue="slate",
            font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif"],
        ),
        css=CSS,
        title="Surrogate Model — Calidad del Aire v10.3",
    ) as app:

        # ── Hero ──────────────────────────────────────────────────────────────
        gr.HTML(f"""
        <div class="hero">
            <h1>🌬️ Surrogate Model — Calidad del Aire</h1>
            <p>Feature Engineering · XGBoost · LightGBM · HistGB · Blind Test · v10.3</p>
            {gpu_badge_html}
        </div>
        """)

        with gr.Row(equal_height=False):

            # ── PANEL IZQUIERDO — CONTROLES ────────────────────────────────────
            with gr.Column(scale=1, min_width=360):

                # ── Dataset: solo File upload [FIX-3] ────────────────────────
                with gr.Group():
                    gr.Markdown("### 📂 Dataset")
                    csv_upload = gr.File(
                        label="Arrastra o sube tu archivo CSV",
                        file_types=[".csv"],
                        type="filepath",
                    )
                    gr.Markdown(
                        "_El nombre del archivo se usará como identificador de parroquia._",
                        elem_classes=[],
                    )

                gr.HTML("<div style='height:8px'/>")

                # ── Configuración del modelo ──────────────────────────────────
                with gr.Group():
                    gr.Markdown("### ⚙️ Configuración del Modelo")

                    with gr.Row():
                        target_input = gr.Dropdown(
                            label="Variable objetivo (Target)",
                            choices=["PM25", "PM10", "NO2", "O3", "CO", "SO2"],
                            value="PM25",
                            allow_custom_value=True,
                            scale=3,
                        )
                        nombre_modelo_input = gr.Textbox(
                            label="Nombre del modelo (.pkl)",
                            value="surrogate_calidad_aire",
                            scale=3,
                        )

                    algoritmo_radio = gr.Radio(
                        label="Algoritmo de entrenamiento",
                        choices=MODELOS,
                        value=MODELOS[0],   # HistGB por defecto (sin dependencias extras)
                    )

                    # Checkboxes [FIX-2] — visibles y funcionales
                    with gr.Row():
                        excluir_pandemia_chk = gr.Checkbox(
                            label="🚫 Excluir pandemia (2020–2021)",
                            value=True,
                            scale=1,
                        )
                        usar_fe_ck = gr.Checkbox(
                            label="🧪 Feature Engineering",
                            value=False,
                            scale=1,
                        )

                    gr.HTML(
                        '<div class="fe-badge">'
                        '<b>Feature Engineering</b>: rolling stats, diferencias, '
                        'interacciones meteo, día de semana → top-40 con RandomForest.'
                        '</div>'
                    )

                    with gr.Row():
                        train_ratio_slider = gr.Slider(
                            label="Train / Test split",
                            minimum=0.6, maximum=0.95, step=0.05, value=0.80, scale=3,
                        )
                        k_splits_slider = gr.Slider(
                            label="K — pliegues TSCV",
                            minimum=2, maximum=15, step=1, value=5, scale=2,
                        )
                        plot_days_slider = gr.Slider(
                            label="Días a graficar",
                            minimum=3, maximum=30, step=1, value=7, scale=2,
                        )

                gr.HTML("<div style='height:8px'/>")

                # ── Hiperparámetros XGBoost ───────────────────────────────────
                with gr.Accordion("🧪 Hiperparámetros XGBoost / LightGBM / HistGB", open=False):
                    gr.Markdown(
                        "_Estos controles aplican a XGBoost y LightGBM. "
                        "Para HistGB se usan `learning_rate`, `max_depth` y `reg_lambda`._"
                    )
                    with gr.Row():
                        max_depth_slider       = gr.Slider(label="max_depth",        minimum=2,   maximum=10,  step=1,    value=5)
                        learning_rate_slider   = gr.Slider(label="learning_rate",    minimum=0.01, maximum=0.3, step=0.01, value=0.05)
                    with gr.Row():
                        subsample_slider       = gr.Slider(label="subsample",        minimum=0.5, maximum=1.0, step=0.05, value=0.8)
                        colsample_bytree_slider = gr.Slider(label="colsample_bytree", minimum=0.5, maximum=1.0, step=0.05, value=0.8)
                    with gr.Row():
                        reg_lambda_slider      = gr.Slider(label="reg_lambda (L2)",  minimum=0.0, maximum=10.0, step=0.5,  value=2.0)
                        reg_alpha_slider       = gr.Slider(label="reg_alpha (L1)",   minimum=0.0, maximum=5.0,  step=0.1,  value=0.5)
                    with gr.Row():
                        gamma_slider           = gr.Slider(label="gamma",            minimum=0.0, maximum=5.0,  step=0.1,  value=0.0)
                        min_child_weight_slider = gr.Slider(label="min_child_weight", minimum=1,   maximum=20,   step=1,    value=1)

                gr.HTML("<div style='height:10px'/>")

                # ── Botón entrenar + logs ─────────────────────────────────────
                btn_train = gr.Button(
                    "🚀  Iniciar Entrenamiento", variant="primary", size="lg"
                )
                gr.HTML("<div style='height:6px'/>")
                gr.Markdown("##### 🖥️ Registro de ejecución")
                estado_output = gr.Markdown(
                    value="_Esperando ejecución…_",
                    elem_classes=["logs-box"],
                )

            # ── PANEL DERECHO — RESULTADOS ─────────────────────────────────────
            with gr.Column(scale=2, min_width=580):
                with gr.Tabs():

                    with gr.Tab("📊 Métricas"):
                        metricas_output = gr.Markdown(
                            value="*Las métricas aparecerán aquí tras entrenar.*"
                        )

                    with gr.Tab("📈 Real vs Predicho"):
                        fig_pred_output = gr.Plot(
                            label="Serie temporal — últimos N días del test ciego"
                        )

                    with gr.Tab("📉 Curva de Aprendizaje"):
                        gr.Markdown(
                            "**Progreso por época / boosting round** con TimeSeriesSplit.\n\n"
                            "Línea continua = Train · Línea discontinua = Validación. "
                            "Banda = ±1σ entre pliegues."
                        )
                        fig_lc_output = gr.Plot(
                            label="Curva de Aprendizaje — Error vs Iteración (TSS)"
                        )

                    with gr.Tab("🔍 Feature Importance"):
                        gr.Markdown(
                            "**Importancia por permutación** calculada sobre el test ciego."
                        )
                        fig_fi_output = gr.Plot(
                            label="Permutation Importance (Δ R²)"
                        )

                    with gr.Tab("📊 Correlaciones"):
                        gr.Markdown(
                            "**Correlación de Pearson** entre todas las variables. "
                            "La columna del target está resaltada en naranja."
                        )
                        fig_hm_output = gr.Plot(
                            label="Mapa de calor — Correlación de Pearson"
                        )

                    with gr.Tab("🔮 Predicción Instantánea"):
                        gr.Markdown(
                            "Ingresa valores ambientales para obtener una estimación "
                            "con el **modelo en memoria** (última sesión entrenada)."
                        )
                        sesion_info = gr.Markdown(
                            "_⚠️ Entrena primero un modelo para habilitar esta sección._"
                        )
                        with gr.Group():
                            gr.Markdown("##### Variables de entrada")
                            slider_componentes = [
                                gr.Number(
                                    label=f"feature_{i}", value=0,
                                    visible=False, interactive=True
                                )
                                for i in range(30)
                            ]
                        json_vals      = gr.Textbox(visible=False, value="{}")
                        btn_predecir   = gr.Button("🔮 Predecir", variant="primary")
                        resultado_pred = gr.Markdown(value="_El resultado aparecerá aquí._")

        gr.HTML("""
        <div style="text-align:center;padding:18px 0 8px;color:#475569;font-size:.76rem;">
            Surrogate Model v10.3 · FIX: Feature Selection · Checkboxes · Upload-only · Auto-GPU
        </div>
        """)

        # ── Funciones auxiliares de la UI ─────────────────────────────────────

        def _refresh_pred_ui():
            feat_cols  = SESION.get("feat_cols", [])
            feat_stats = SESION.get("feat_stats", {})
            parroquia  = SESION.get("parroquia", "")
            target     = SESION.get("target", "")
            sesion_msg = (
                f"✅ **{target}** · *{parroquia}* · {len(feat_cols)} features."
                if feat_cols else "_⚠️ Entrena primero un modelo._"
            )
            updates = []
            for i in range(30):
                if i < len(feat_cols):
                    col = feat_cols[i]
                    st  = feat_stats.get(col, {"min": 0, "max": 100, "mean": 50})
                    updates.append(gr.Number(
                        label=col,
                        value=round(st["mean"], 3),
                        visible=True,
                        interactive=True,
                    ))
                else:
                    updates.append(gr.Number(visible=False))
            return [sesion_msg] + updates

        def _construir_json(*vals):
            feat_cols = SESION.get("feat_cols", [])
            d = {
                feat_cols[i]: float(vals[i])
                for i in range(min(len(feat_cols), len(vals)))
                if vals[i] is not None
            }
            return json.dumps(d)

        # ── Eventos ───────────────────────────────────────────────────────────

        # [FIX-4] Todos los parámetros de la UI están conectados correctamente
        btn_train.click(
            fn=entrenar,
            inputs=[
                csv_upload,                 # [FIX-3] único origen de datos
                target_input,
                nombre_modelo_input,
                algoritmo_radio,
                plot_days_slider,
                train_ratio_slider,
                excluir_pandemia_chk,       # [FIX-2] checkbox funcional
                k_splits_slider,
                max_depth_slider,
                learning_rate_slider,
                subsample_slider,
                colsample_bytree_slider,
                reg_lambda_slider,
                reg_alpha_slider,
                gamma_slider,
                min_child_weight_slider,
                usar_fe_ck,                 # [FIX-2] checkbox funcional
            ],
            outputs=[
                metricas_output,
                fig_pred_output,
                fig_fi_output,
                fig_hm_output,
                fig_lc_output,
                estado_output,
            ],
        ).then(
            fn=_refresh_pred_ui,
            inputs=[],
            outputs=[sesion_info] + slider_componentes,
        )

        btn_predecir.click(
            fn=_construir_json,
            inputs=slider_componentes,
            outputs=json_vals,
        ).then(
            fn=predecir_desde_sesion,
            inputs=[json_vals],
            outputs=[resultado_pred],
        )

    return app


# ──────────────────────────────────────────────────────────────────────────────
# 13.  PUNTO DE ENTRADA
# ──────────────────────────────────────────────────────────────────────────────

def _imprimir_ip_fallback() -> None:
    import socket
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        print(f"  🖥️  IP local   : http://{s.getsockname()[0]}:<PUERTO>")
        s.close()
    except Exception:
        pass
    try:
        ips = subprocess.check_output(["hostname", "-I"], text=True, timeout=4).strip()
        print(f"  🌐  IPs servidor: {ips}")
    except Exception:
        pass
    try:
        ip_pub = subprocess.check_output(
            ["curl", "-s", "--max-time", "5", "https://ipecho.net/plain"],
            text=True, timeout=7,
        ).strip()
        if ip_pub:
            print(f"  🌍  IP pública  : http://{ip_pub}:<PUERTO>")
    except Exception:
        pass


if __name__ == "__main__":
    app = construir_app()

    print("\n" + "═" * 66)
    print("  🌬️  SURROGATE MODEL — Calidad del Aire  v10.3")
    print("  🔧  FIX: Feature Selection · Checkboxes · Upload-only")
    print("═" * 66)
    print(f"  Hardware  : {GPU_MSG}")
    print(f"  GPU activa: {GPU_DISPONIBLE}")
    print(f"  FE top-N  : {MAX_FEATURES_SEL} features (RandomForest selector)")
    print("  Puerto    : automático (server_port=None)")
    print("═" * 66)

    try:
        app.launch(
            server_name="0.0.0.0",
            server_port=None,
            share=True,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )
    except OSError as port_err:
        print(f"\n⚠️  Error de puerto: {port_err}\n🔄  Reintentando en puerto 7861…\n")
        app.launch(
            server_name="0.0.0.0",
            server_port=7861,
            share=True,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )
    except Exception as tunnel_err:
        print(f"\n⚠️  Túnel público falló: {tunnel_err}\n🔄  Relanzando sin túnel:\n")
        _imprimir_ip_fallback()
        app.launch(
            server_name="0.0.0.0",
            server_port=None,
            share=False,
            max_threads=40,
            debug=True,
            show_error=True,
            prevent_thread_lock=False,
            quiet=False,
        )

Version con nuevas funciones en la parte de prediccion del codigo anterior version claude

In [2]:
"""
================================================================================
  SURROGATE MODEL — CALIDAD DEL AIRE  |  Interfaz Web Gradio
  Versión : 10.4
  ─── Novedades v10.4 ────────────────────────────────────────────────────────
  [NEW-1]  Dropdown dinámico de target
           · csv_upload.change → actualizar_targets(archivo)
           · Solo muestra contaminantes presentes en el CSV subido.
  [NEW-2]  Tabla de lags climatológicos (lag_lookup)
           · _crear_lag_lookup(X_train, feat_cols) → media por (hora, mes).
           · Guardada en SESION["lag_lookup"] y SESION["ultimo_timestamp"].
  [NEW-3]  Pestaña de predicción simplificada + IQCA REMMAQ
           · Solo pide: fecha/hora, Temperatura, Humedad, Viento, Precipitación.
           · Lags y rolling stats se imputan desde lag_lookup o feat_stats.
           · calcular_iqca() e categoria_iqca() con breakpoints oficiales REMMAQ.
           · Advertencia si la fecha pedida está > 48 h del último dato.
  [NEW-4]  Conexiones de eventos actualizadas.
  ─── Correcciones heredadas de v10.3 ────────────────────────────────────────
  [FIX-1]  SelectFromModel con RandomForestRegressor (feature_importances_).
  [FIX-2]  Checkboxes funcionales (appearance:checkbox nativo).
  [FIX-3]  Solo gr.File — sin explorador de archivos locales.
  [FIX-4]  Todos los parámetros de la UI conectados a entrenar().
================================================================================
"""

# ──────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTACIONES
# ──────────────────────────────────────────────────────────────────────────────
import os
import warnings
import logging
import traceback
import subprocess
import pickle
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

import gradio as gr
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# ──────────────────────────────────────────────────────────────────────────────
# 1.  CONSTANTES
# ──────────────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("resultados_surrogate")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SESION: dict = {
    "modelo":           None,
    "scaler":           None,
    "feat_cols":        [],
    "target":           "",
    "parroquia":        "",
    "feat_stats":       {},
    "lag_lookup":       {},    # [NEW-2] {col: {(hora, mes): valor_medio}}
    "ultimo_timestamp": None,  # [NEW-2] pd.Timestamp del último dato de train
}

ANOS_PANDEMIA   = [2020, 2021]
CONTAMINANTES_Y = ["PM25", "PM10", "O3", "CO", "NO2", "SO2"]

FEATURES_X_BASE = [
    "Temperatura", "Humedad", "Viento_Velocidad", "Viento_Direccion", "Precipitacion",
    "hora_sin", "hora_cos", "mes_sin", "mes_cos",
    "PM25_lag_1h",  "PM25_lag_24h",
    "PM10_lag_1h",  "PM10_lag_24h",
    "O3_lag_1h",    "O3_lag_24h",
    "CO_lag_1h",    "CO_lag_24h",
    "NO2_lag_1h",   "NO2_lag_24h",
    "SO2_lag_1h",   "SO2_lag_24h",
]

# Nombres exactos de columnas meteorológicas base
METEO_COLS = ["Temperatura", "Humedad", "Viento_Velocidad", "Viento_Direccion", "Precipitacion"]

COLOR_REAL = "#3B82F6"
COLOR_PRED = "#F97316"
COLOR_POS  = "#22C55E"
COLOR_NEG  = "#EF4444"
BG_PLOT    = "#0F172A"
TEXT_PLOT  = "#E2E8F0"
GRID_PLOT  = "#1E293B"

MAX_FEATURES_SEL = 40


# ──────────────────────────────────────────────────────────────────────────────
# 2.  DETECCIÓN GPU
# ──────────────────────────────────────────────────────────────────────────────

def detectar_gpu() -> tuple[bool, str]:
    try:
        r = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=8,
        )
        if r.returncode == 0 and r.stdout.strip():
            return True, f"GPU detectada (nvidia-smi): {r.stdout.strip().split(chr(10))[0]}"
    except Exception:
        pass
    try:
        import torch
        if torch.cuda.is_available():
            return True, f"GPU detectada (torch.cuda): {torch.cuda.get_device_name(0)}"
    except ImportError:
        pass
    try:
        import cupy  # noqa: F401
        return True, "GPU detectada (cupy disponible)"
    except ImportError:
        pass
    return False, "No se detectó GPU — se usará CPU"


GPU_DISPONIBLE, GPU_MSG = detectar_gpu()
log.info(GPU_MSG)


def _xgb_tree_method() -> dict:
    return {"tree_method": "hist", "device": "cuda"} if GPU_DISPONIBLE else {"tree_method": "hist", "device": "cpu"}

def _lgbm_device() -> str:
    return "gpu" if GPU_DISPONIBLE else "cpu"


# ──────────────────────────────────────────────────────────────────────────────
# 3.  CARGA Y DETECCIÓN DE TIMESTAMP
# ──────────────────────────────────────────────────────────────────────────────

def _cargar_csv(ruta: str) -> pd.DataFrame:
    try:
        return pd.read_csv(ruta, comment="#", low_memory=False, on_bad_lines="warn")
    except Exception as e:
        raise RuntimeError(f"Error al leer '{ruta}': {e}") from e


def _detectar_timestamp(df: pd.DataFrame) -> str | None:
    keywords = ("time", "fecha", "date", "hora", "datetime", "timestamp")
    candidatos = [c for c in df.columns if any(k in c.lower() for k in keywords)]
    if candidatos:
        return candidatos[0]
    for c in df.columns:
        try:
            pd.to_datetime(df[c].dropna().astype(str).iloc[:10], infer_datetime_format=True)
            return c
        except Exception:
            continue
    return None


# ──────────────────────────────────────────────────────────────────────────────
# 4.  [NEW-1]  DROPDOWN DINÁMICO DE TARGET
# ──────────────────────────────────────────────────────────────────────────────

def actualizar_targets(archivo) -> gr.Dropdown:
    """
    Lee las primeras filas del CSV subido y devuelve un Dropdown cuyas
    opciones son solo los contaminantes de CONTAMINANTES_Y presentes en el archivo.
    Fallback: ["PM25"] si no se detecta ninguno.
    """
    if archivo is None:
        return gr.Dropdown(choices=CONTAMINANTES_Y, value="PM25")
    try:
        ruta = archivo if isinstance(archivo, str) else archivo.name
        df_head = pd.read_csv(ruta, comment="#", nrows=3, low_memory=False)
        cols_csv = set(df_head.columns.tolist())
        disponibles = [c for c in CONTAMINANTES_Y if c in cols_csv]
        if not disponibles:
            disponibles = ["PM25"]
        return gr.Dropdown(choices=disponibles, value=disponibles[0])
    except Exception:
        return gr.Dropdown(choices=CONTAMINANTES_Y, value="PM25")


# ──────────────────────────────────────────────────────────────────────────────
# 5.  PREPROCESAMIENTO
# ──────────────────────────────────────────────────────────────────────────────

def _preprocesar(df: pd.DataFrame, target_col: str, timestamp_col: str,
                 excluir_pandemia: bool = True) -> pd.DataFrame:
    df = df.copy()
    df[timestamp_col] = pd.to_datetime(
        df[timestamp_col], infer_datetime_format=True, errors="coerce"
    )
    df = df.dropna(subset=[timestamp_col]).set_index(timestamp_col).sort_index()
    if excluir_pandemia:
        df = df[~df.index.year.isin(ANOS_PANDEMIA)]
    obj_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
    if obj_cols:
        df[obj_cols] = df[obj_cols].apply(pd.to_numeric, errors="coerce")
    df = df.dropna(subset=[target_col])
    return df


def _dividir_cronologico(X, y, ratio: float = 0.80):
    corte = int(len(X) * ratio)
    return X.iloc[:corte], X.iloc[corte:], y.iloc[:corte], y.iloc[corte:]


def _resolver_features_x(df: pd.DataFrame, target_col: str) -> list[str]:
    cols = set(df.select_dtypes(include=[np.number]).columns)
    excluir = set(CONTAMINANTES_Y)
    return [c for c in cols if c not in excluir and c != target_col]


# ──────────────────────────────────────────────────────────────────────────────
# 6.  FEATURE ENGINEERING
# ──────────────────────────────────────────────────────────────────────────────

def _agregar_features_avanzadas(df: pd.DataFrame, target_col: str) -> pd.DataFrame:
    df = df.copy()
    contaminantes = [c for c in CONTAMINANTES_Y if c in df.columns]

    for col in contaminantes:
        for window in [3, 6, 12, 24]:
            df[f"{col}_roll_mean_{window}h"] = df[col].rolling(window, min_periods=1).mean()
            if window >= 6:
                df[f"{col}_roll_std_{window}h"] = df[col].rolling(window, min_periods=2).std()
        for lag in [1, 3, 6, 12, 24]:
            df[f"{col}_diff_{lag}h"] = df[col].diff(lag)

    if "Temperatura" in df.columns and "Humedad" in df.columns:
        df["temp_hum"] = df["Temperatura"] * df["Humedad"]
    if "Temperatura" in df.columns and "Viento_Velocidad" in df.columns:
        df["temp_wind"] = df["Temperatura"] * df["Viento_Velocidad"]
    if "Viento_Direccion" in df.columns and "Viento_Velocidad" in df.columns:
        dir_rad = np.radians(df["Viento_Direccion"])
        df["wind_u"] = df["Viento_Velocidad"] * np.cos(dir_rad)
        df["wind_v"] = df["Viento_Velocidad"] * np.sin(dir_rad)

    if hasattr(df.index, "weekday"):
        df["dia_semana"]     = df.index.weekday
        df["dia_semana_sin"] = np.sin(2 * np.pi * df["dia_semana"] / 7)
        df["dia_semana_cos"] = np.cos(2 * np.pi * df["dia_semana"] / 7)
        df["es_finde"]       = (df["dia_semana"] >= 5).astype(int)

    return df.dropna()


def _seleccionar_top_features(
    X_tr: pd.DataFrame, y_tr: pd.Series,
    n_features: int = MAX_FEATURES_SEL, log_fn=None,
) -> list[str]:
    if log_fn is None:
        log_fn = log.info
    all_cols = X_tr.columns.tolist()
    if len(all_cols) <= n_features:
        log_fn(f"  Todas las {len(all_cols)} features disponibles (≤ {n_features}).")
        return all_cols
    try:
        mask_ok = X_tr.notna().all(axis=1) & y_tr.notna()
        X_ok, y_ok = X_tr[mask_ok].values, y_tr[mask_ok].values
        if len(X_ok) < 50:
            log_fn("  ⚠️  Datos insuficientes para selección — usando todas.")
            return all_cols
        rf_sel = RandomForestRegressor(n_estimators=80, max_depth=6,
                                       min_samples_leaf=20, n_jobs=-1, random_state=42)
        rf_sel.fit(X_ok, y_ok)
        selector = SelectFromModel(rf_sel, threshold=-np.inf,
                                   max_features=n_features, prefit=True)
        selected = [c for c, k in zip(all_cols, selector.get_support()) if k]
        log_fn(f"  SelectFromModel: {len(all_cols)} → {len(selected)} features.")
        return selected if selected else all_cols
    except Exception as exc:
        log_fn(f"  ⚠️  Selección falló ({exc}) — usando todas las features.")
        return all_cols


# ──────────────────────────────────────────────────────────────────────────────
# 7.  [NEW-2]  TABLA DE LAGS CLIMATOLÓGICOS
# ──────────────────────────────────────────────────────────────────────────────

def _crear_lag_lookup(X_train: pd.DataFrame, feat_cols: list[str]) -> dict:
    """
    Crea un diccionario de valores medios por (hora, mes) para cada feature.

    Estructura devuelta:
        {
            "PM25_lag_1h": { (hora, mes): valor_medio, … },
            "PM10_roll_mean_6h": { … },
            …
        }

    Requiere que X_train tenga índice DatetimeIndex con hora y mes.
    Si el índice no es temporal, devuelve dict vacío.
    """
    lookup: dict = {}
    if not hasattr(X_train.index, "hour"):
        return lookup

    df_tmp = X_train[feat_cols].copy()
    df_tmp["_hora"] = X_train.index.hour
    df_tmp["_mes"]  = X_train.index.month

    for col in feat_cols:
        if col not in df_tmp.columns:
            continue
        grupo = (
            df_tmp.groupby(["_hora", "_mes"])[col]
            .mean()
            .dropna()
        )
        lookup[col] = {(int(h), int(m)): float(v) for (h, m), v in grupo.items()}

    return lookup


# ──────────────────────────────────────────────────────────────────────────────
# 8.  [NEW-3]  IQCA REMMAQ — Breakpoints y cálculo
# ──────────────────────────────────────────────────────────────────────────────

# Breakpoints oficiales REMMAQ (concentración → IQCA)
# Formato: [(C_low, C_high, IQCA_low, IQCA_high), …]
IQCA_BREAKPOINTS: dict[str, list[tuple]] = {
    "PM25": [
        (0.0,  12.0,   0,  50),
        (12.1, 37.4,  51, 100),
        (37.5, 55.4, 101, 150),
        (55.5, 150.4,151, 200),
        (150.5,250.4,201, 300),
        (250.5,500.4,301, 500),
    ],
    "PM10": [
        (0,   54,   0,  50),
        (55,  154,  51, 100),
        (155, 254, 101, 150),
        (255, 354, 151, 200),
        (355, 424, 201, 300),
        (425, 604, 301, 500),
    ],
    "O3": [
        (0,    54,   0,  50),
        (55,  124,  51, 100),
        (125, 164, 101, 150),
        (165, 204, 151, 200),
        (205, 404, 201, 300),
        (405, 604, 301, 500),
    ],
    "CO": [
        (0.0,  4.4,   0,  50),
        (4.5,  9.4,  51, 100),
        (9.5,  12.4, 101, 150),
        (12.5, 15.4, 151, 200),
        (15.5, 30.4, 201, 300),
        (30.5, 50.4, 301, 500),
    ],
    "NO2": [
        (0,    53,   0,  50),
        (54,  100,  51, 100),
        (101, 360, 101, 150),
        (361, 649, 151, 200),
        (650,1249, 201, 300),
        (1250,2049,301, 500),
    ],
    "SO2": [
        (0,    35,   0,  50),
        (36,   75,  51, 100),
        (76,  185, 101, 150),
        (186, 304, 151, 200),
        (305, 604, 201, 300),
        (605,1004, 301, 500),
    ],
}

# Categorías IQCA REMMAQ
CATEGORIAS_IQCA = [
    (0,   50,  "🟢 Deseable",    "#22C55E"),
    (51,  100, "🟡 Aceptable",   "#EAB308"),
    (101, 150, "🟠 Precaución",  "#F97316"),
    (151, 200, "🔴 Alerta",      "#EF4444"),
    (201, 300, "🟣 Alarma",      "#A855F7"),
    (301, 500, "⚫ Emergencia",   "#1E293B"),
]


def calcular_iqca(contaminante: str, concentracion: float) -> float | None:
    """
    Calcula el IQCA mediante interpolación lineal por tramos según breakpoints
    oficiales REMMAQ. Devuelve None si el contaminante no está en la tabla.
    """
    bp_list = IQCA_BREAKPOINTS.get(contaminante)
    if bp_list is None:
        return None
    for (c_lo, c_hi, i_lo, i_hi) in bp_list:
        if c_lo <= concentracion <= c_hi:
            # Interpolación lineal
            return i_lo + (concentracion - c_lo) * (i_hi - i_lo) / (c_hi - c_lo)
    # Fuera del rango superior → máximo
    if concentracion > bp_list[-1][1]:
        return 500.0
    return 0.0


def categoria_iqca(iqca: float) -> tuple[str, str]:
    """
    Devuelve (etiqueta, color_hex) para un valor de IQCA dado.
    """
    for (lo, hi, label, color) in CATEGORIAS_IQCA:
        if lo <= iqca <= hi:
            return label, color
    return "⚫ Emergencia", "#1E293B"


# ──────────────────────────────────────────────────────────────────────────────
# 9.  [NEW-3]  PREDICCIÓN SIMPLIFICADA
# ──────────────────────────────────────────────────────────────────────────────

def predecir_simple(
    fecha_hora,          # str o datetime (del gr.Textbox / gr.DateTime)
    temperatura: float,
    humedad: float,
    viento_vel: float,
    viento_dir: float,
    precipitacion: float,
) -> str:
    """
    Predicción con solo variables meteorológicas básicas.

    Los lags y rolling stats se imputan automáticamente desde lag_lookup
    (media climatológica por hora y mes). Si no hay modelo entrenado, avisa.
    """
    if SESION["modelo"] is None:
        return "### ⚠️ No hay modelo entrenado.\nEntrena primero un modelo."

    try:
        # ── Parsear fecha/hora ────────────────────────────────────────────────
        if fecha_hora is None or str(fecha_hora).strip() == "":
            return "### ❌ Ingresa una fecha y hora válida."

        try:
            ts = pd.Timestamp(str(fecha_hora))
        except Exception:
            return f"### ❌ Formato de fecha inválido: `{fecha_hora}`\nUsa: `YYYY-MM-DD HH:MM`"

        hora = ts.hour
        mes  = ts.month

        # ── Advertencia extrapolación > 48h ───────────────────────────────────
        ultimo = SESION.get("ultimo_timestamp")
        advertencia = ""
        if ultimo is not None:
            delta_h = (ts - ultimo).total_seconds() / 3600
            if delta_h > 48:
                advertencia = (
                    f"\n\n> ⚠️ **Extrapolación**: la fecha pedida está "
                    f"**{delta_h:.0f} horas** ({delta_h/24:.1f} días) después del "
                    f"último dato de entrenamiento ({ultimo.strftime('%Y-%m-%d %H:%M')}). "
                    f"Los lags se imputarán desde promedios climatológicos; "
                    f"la predicción puede ser menos fiable."
                )

        # ── Variables cíclicas ────────────────────────────────────────────────
        ciclicas = {
            "hora_sin": float(np.sin(2 * np.pi * hora / 24)),
            "hora_cos": float(np.cos(2 * np.pi * hora / 24)),
            "mes_sin":  float(np.sin(2 * np.pi * mes  / 12)),
            "mes_cos":  float(np.cos(2 * np.pi * mes  / 12)),
        }

        # ── Construir vector de entrada ───────────────────────────────────────
        feat_cols  = SESION["feat_cols"]
        feat_stats = SESION["feat_stats"]
        lag_lookup = SESION.get("lag_lookup", {})

        # Valores meteo base provistos por el usuario
        meteo_vals = {
            "Temperatura":      float(temperatura),
            "Humedad":          float(humedad),
            "Viento_Velocidad": float(viento_vel),
            "Viento_Direccion": float(viento_dir),
            "Precipitacion":    float(precipitacion),
        }

        # Interacciones derivadas de los valores del usuario (si están en feat_cols)
        dir_rad = np.radians(float(viento_dir))
        derivadas = {
            "temp_hum":  float(temperatura) * float(humedad),
            "temp_wind": float(temperatura) * float(viento_vel),
            "wind_u":    float(viento_vel)  * np.cos(dir_rad),
            "wind_v":    float(viento_vel)  * np.sin(dir_rad),
            "dia_semana":     float(ts.weekday()),
            "dia_semana_sin": float(np.sin(2 * np.pi * ts.weekday() / 7)),
            "dia_semana_cos": float(np.cos(2 * np.pi * ts.weekday() / 7)),
            "es_finde":       float(1 if ts.weekday() >= 5 else 0),
        }

        # Fuentes de valor en orden de prioridad: usuario > cíclicas > derivadas > lag_lookup > feat_stats mean
        row: dict = {}
        for col in feat_cols:
            if col in meteo_vals:
                row[col] = meteo_vals[col]
            elif col in ciclicas:
                row[col] = ciclicas[col]
            elif col in derivadas:
                row[col] = derivadas[col]
            elif col in lag_lookup and (hora, mes) in lag_lookup[col]:
                row[col] = lag_lookup[col][(hora, mes)]
            else:
                row[col] = feat_stats.get(col, {}).get("mean", 0.0)

        X_input = pd.DataFrame([row])[feat_cols]
        scaler  = SESION["scaler"]
        X_sc    = scaler.transform(X_input) if scaler else X_input.values
        pred    = float(SESION["modelo"].predict(X_sc)[0])

        # ── IQCA ─────────────────────────────────────────────────────────────
        target   = SESION["target"]
        parroquia = SESION["parroquia"]
        iqca_val = calcular_iqca(target, pred)

        if iqca_val is not None:
            cat_label, cat_color = categoria_iqca(iqca_val)
            iqca_row = (
                f"| **IQCA** | `{iqca_val:.1f}` |\n"
                f"| **Categoría REMMAQ** | {cat_label} |"
            )
        else:
            iqca_row = f"| **IQCA** | _No disponible para {target}_ |"

        # Fuente de los lags (info para el usuario)
        n_lookup = sum(
            1 for col in feat_cols
            if col in lag_lookup and (hora, mes) in lag_lookup.get(col, {})
        )
        n_mean   = len(feat_cols) - len(meteo_vals) - len([c for c in feat_cols if c in ciclicas]) - n_lookup
        n_mean   = max(n_mean, 0)

        result = (
            f"## 🔮 Predicción IQCA — `{target}` · *{parroquia}*\n\n"
            f"| Campo | Valor |\n|-------|-------|\n"
            f"| **Fecha / Hora** | `{ts.strftime('%Y-%m-%d %H:%M')}` |\n"
            f"| **{target} estimado** | `{pred:.3f} µg/m³` |\n"
            f"{iqca_row}\n\n"
            f"---\n\n"
            f"**Imputación de lags** → "
            f"`{n_lookup}` desde lag_lookup climatológico · "
            f"`{n_mean}` desde media general de entrenamiento"
            f"{advertencia}"
        )
        return result

    except Exception:
        return f"### ❌ Error en predicción\n```\n{traceback.format_exc()}\n```"


# ──────────────────────────────────────────────────────────────────────────────
# 10.  CONSTRUCCIÓN DEL MODELO
# ──────────────────────────────────────────────────────────────────────────────

def _construir_modelo(algoritmo: str, params: dict | None = None):
    if params is None:
        params = {}

    if "XGBoost" in algoritmo:
        import xgboost as xgb
        return xgb.XGBRegressor(
            n_estimators=2000,
            max_depth=int(params.get("max_depth", 5)),
            learning_rate=float(params.get("learning_rate", 0.05)),
            subsample=float(params.get("subsample", 0.8)),
            colsample_bytree=float(params.get("colsample_bytree", 0.8)),
            reg_lambda=float(params.get("reg_lambda", 2.0)),
            reg_alpha=float(params.get("reg_alpha", 0.5)),
            gamma=float(params.get("gamma", 0.0)),
            min_child_weight=float(params.get("min_child_weight", 1)),
            early_stopping_rounds=50,
            eval_metric="rmse",
            random_state=42, verbosity=0,
            **_xgb_tree_method(),
        )
    elif "LightGBM" in algoritmo:
        import lightgbm as lgb
        return lgb.LGBMRegressor(
            n_estimators=2000,
            max_depth=int(params.get("max_depth", 5)),
            learning_rate=float(params.get("learning_rate", 0.05)),
            subsample=float(params.get("subsample", 0.8)),
            colsample_bytree=float(params.get("colsample_bytree", 0.8)),
            reg_lambda=float(params.get("reg_lambda", 2.0)),
            reg_alpha=float(params.get("reg_alpha", 0.5)),
            min_child_samples=int(params.get("min_child_weight", 30)),
            metric="rmse", device=_lgbm_device(),
            random_state=42, verbose=-1,
        )
    else:
        return HistGradientBoostingRegressor(
            max_iter=1000, early_stopping=True, n_iter_no_change=30,
            validation_fraction=0.1,
            max_depth=int(params.get("max_depth", 5)),
            min_samples_leaf=int(params.get("min_child_weight", 30)),
            learning_rate=float(params.get("learning_rate", 0.05)),
            l2_regularization=float(params.get("reg_lambda", 1.0)),
            random_state=42,
        )


# ──────────────────────────────────────────────────────────────────────────────
# 11.  PIPELINE TimeSeriesSplit
# ──────────────────────────────────────────────────────────────────────────────

def _extraer_curvas_fold(modelo, algoritmo: str) -> tuple[list, list, str]:
    train_hist, val_hist, metric_name = [], [], "Score"
    try:
        if "XGBoost" in algoritmo:
            evals = modelo.evals_result()
            ks = list(evals.keys())
            met = list(evals[ks[0]].keys())[0]
            metric_name = met.upper()
            train_hist  = list(evals[ks[0]][met])
            val_hist    = list(evals[ks[1]][met]) if len(ks) > 1 else []
        elif "LightGBM" in algoritmo:
            evals = modelo.evals_result_
            ks = list(evals.keys())
            met = list(evals[ks[0]].keys())[0]
            metric_name = met.upper()
            train_hist  = list(evals[ks[0]][met])
            val_hist    = list(evals[ks[1]][met]) if len(ks) > 1 else []
        else:
            if hasattr(modelo, "train_score_") and modelo.train_score_ is not None:
                train_hist  = list(modelo.train_score_)
                metric_name = "R²"
            if hasattr(modelo, "validation_score_") and modelo.validation_score_ is not None:
                val_hist = list(modelo.validation_score_)
    except Exception as exc:
        log.warning(f"_extraer_curvas_fold: {exc}")
    return train_hist, val_hist, metric_name


def _entrenar_kfold(df, algoritmo, n_splits=5, log_fn=None, xgb_params=None):
    if log_fn is None:
        log_fn = log.info
    tscv = TimeSeriesSplit(n_splits=n_splits)
    filas, all_curves = [], {}
    cols_df = set(df.columns)

    for contaminante in CONTAMINANTES_Y:
        if contaminante not in cols_df:
            log_fn(f"  ⏭️  {contaminante} no disponible — omitido.")
            continue
        feat_cols = _resolver_features_x(df, contaminante)
        y_vals = df[contaminante].dropna()
        X_vals = df.loc[y_vals.index, feat_cols]
        mask_ok = X_vals.notna().all(axis=1) & y_vals.notna()
        X_vals, y_vals = X_vals[mask_ok], y_vals[mask_ok]
        if len(X_vals) < n_splits * 20:
            log_fn(f"  ⚠️  {contaminante}: {len(X_vals)} filas — omitido.")
            continue
        X_arr, y_arr = X_vals.values, y_vals.values
        fold_metrics, fold_curves_tr, fold_curves_va = [], [], []
        metric_name_cv = "Score"

        for fold_idx, (tr_idx, va_idx) in enumerate(tscv.split(X_arr), 1):
            X_tr, X_va = X_arr[tr_idx], X_arr[va_idx]
            y_tr, y_va = y_arr[tr_idx], y_arr[va_idx]
            scaler = StandardScaler()
            X_tr_sc = scaler.fit_transform(X_tr)
            X_va_sc = scaler.transform(X_va)
            modelo = _construir_modelo(algoritmo, xgb_params)
            if "XGBoost" in algoritmo:
                modelo.fit(X_tr_sc, y_tr, eval_set=[(X_tr_sc, y_tr), (X_va_sc, y_va)], verbose=False)
            elif "LightGBM" in algoritmo:
                import lightgbm as lgb
                modelo.fit(X_tr_sc, y_tr, eval_set=[(X_tr_sc, y_tr), (X_va_sc, y_va)],
                           callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)])
            else:
                modelo.fit(X_tr_sc, y_tr)
            tr_h, va_h, met = _extraer_curvas_fold(modelo, algoritmo)
            if tr_h: fold_curves_tr.append(tr_h)
            if va_h: fold_curves_va.append(va_h)
            metric_name_cv = met
            y_pred = modelo.predict(X_va_sc)
            fold_metrics.append({
                "MAE":  mean_absolute_error(y_va, y_pred),
                "RMSE": float(np.sqrt(mean_squared_error(y_va, y_pred))),
                "R2":   r2_score(y_va, y_pred),
            })
            log_fn(f"  {contaminante} | Fold {fold_idx}/{n_splits} → MAE={fold_metrics[-1]['MAE']:.3f} RMSE={fold_metrics[-1]['RMSE']:.3f} R²={fold_metrics[-1]['R2']:.3f}")

        all_curves[contaminante] = {"train": fold_curves_tr, "val": fold_curves_va, "metric": metric_name_cv}
        mf = pd.DataFrame(fold_metrics)
        filas.append({
            "Contaminante": contaminante, "Features_X": len(feat_cols),
            "MAE_mean": mf["MAE"].mean(), "MAE_std": mf["MAE"].std(),
            "RMSE_mean": mf["RMSE"].mean(), "RMSE_std": mf["RMSE"].std(),
            "R2_mean": mf["R2"].mean(), "R2_std": mf["R2"].std(),
        })

    return (pd.DataFrame(filas) if filas else pd.DataFrame()), all_curves


# ──────────────────────────────────────────────────────────────────────────────
# 12.  FIGURAS
# ──────────────────────────────────────────────────────────────────────────────

def _aplicar_estilo_ax(ax, titulo, xlabel, ylabel):
    ax.set_title(titulo, fontsize=11, fontweight="bold", color=TEXT_PLOT, pad=12)
    if xlabel: ax.set_xlabel(xlabel, color=TEXT_PLOT, fontsize=10)
    if ylabel: ax.set_ylabel(ylabel, color=TEXT_PLOT, fontsize=10)
    ax.tick_params(colors=TEXT_PLOT, labelsize=9)
    ax.grid(True, linestyle="--", alpha=0.18, color=TEXT_PLOT)
    for sp in ax.spines.values(): sp.set_edgecolor(GRID_PLOT)


def _fig_prediccion(y_test, y_pred, target_col, days, parroquia=""):
    df_p = pd.DataFrame({"Real": y_test.values, "Predicho": y_pred}, index=y_test.index)
    df_p = df_p[df_p.index >= df_p.index.max() - pd.Timedelta(days=days)]
    titulo = f"Surrogate Model — {target_col}  ·  Últimos {days} días (Test)"
    if parroquia: titulo = f"[{parroquia}]  {titulo}"
    fig, ax = plt.subplots(figsize=(13, 4.5), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    ax.plot(df_p.index, df_p["Real"],     label="Real",     color=COLOR_REAL, lw=1.8, alpha=0.95)
    ax.plot(df_p.index, df_p["Predicho"], label="Predicho", color=COLOR_PRED, lw=1.5, linestyle="--", alpha=0.90)
    ax.fill_between(df_p.index, df_p["Real"], df_p["Predicho"], alpha=0.07, color=COLOR_PRED)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    ax.xaxis.set_major_locator(mdates.DayLocator())
    plt.xticks(rotation=28, ha="right", color=TEXT_PLOT, fontsize=9)
    plt.yticks(color=TEXT_PLOT, fontsize=9)
    for sp in ax.spines.values(): sp.set_edgecolor(GRID_PLOT)
    ax.set_title(titulo, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=12)
    ax.set_ylabel(target_col, color=TEXT_PLOT)
    ax.set_xlabel("Fecha", color=TEXT_PLOT)
    ax.grid(True, linestyle="--", alpha=0.18, color=TEXT_PLOT)
    ax.legend(framealpha=0.15, labelcolor=TEXT_PLOT, facecolor=BG_PLOT, edgecolor=GRID_PLOT, fontsize=10)
    plt.tight_layout()
    return fig


def _fig_feature_importance(fi_df, target_col, parroquia=""):
    n = len(fi_df)
    fig, ax = plt.subplots(figsize=(9, max(4, n * 0.52)), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    colores = [COLOR_POS if v >= 0 else COLOR_NEG for v in fi_df["Importance"]]
    ax.barh(fi_df["Feature"][::-1], fi_df["Importance"][::-1],
            xerr=fi_df["Std"][::-1], color=colores[::-1],
            align="center", alpha=0.85, ecolor="#94A3B8", capsize=3, height=0.65)
    ax.axvline(0, color="#475569", linewidth=0.9, linestyle="--")
    for sp in ax.spines.values(): sp.set_edgecolor(GRID_PLOT)
    titulo_fi = f"Feature Importance  ·  {target_col}  (Permutation Δ R²)"
    if parroquia: titulo_fi = f"[{parroquia}]  {titulo_fi}"
    plt.yticks(color=TEXT_PLOT, fontsize=9)
    plt.xticks(color=TEXT_PLOT, fontsize=9)
    ax.set_title(titulo_fi, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=12)
    ax.set_xlabel("Importancia media (Δ R²)", color=TEXT_PLOT)
    ax.grid(True, axis="x", linestyle="--", alpha=0.14, color=TEXT_PLOT)
    plt.tight_layout()
    return fig


def _fig_heatmap(df_full, target_col, parroquia=""):
    num_df = df_full.select_dtypes(include=[np.number])
    if num_df.shape[1] < 2: return None
    corr = num_df.corr(method="pearson")
    n = len(corr)
    fig, ax = plt.subplots(figsize=(max(8, n * 0.60), max(7, n * 0.55)), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    mask = np.zeros_like(corr, dtype=bool)
    mask[np.triu_indices_from(mask, k=1)] = True
    cmap = sns.diverging_palette(230, 20, as_cmap=True)
    sns.heatmap(corr, mask=mask, cmap=cmap, vmin=-1, vmax=1, center=0,
                annot=True, fmt=".2f", annot_kws={"size": 7.5, "color": TEXT_PLOT},
                linewidths=0.4, linecolor=GRID_PLOT, square=True, ax=ax, cbar_kws={"shrink": 0.7})
    if target_col in corr.columns:
        idx = list(corr.columns).index(target_col)
        ax.add_patch(plt.Rectangle((idx, 0), 1, n, fill=False, edgecolor="#F97316", lw=2.5, clip_on=False))
        ax.add_patch(plt.Rectangle((0, idx), n, 1, fill=False, edgecolor="#F97316", lw=2.5, clip_on=False))
    titulo_hm = f"Correlación de Pearson — {target_col}"
    if parroquia: titulo_hm = f"[{parroquia}]  {titulo_hm}"
    ax.set_title(titulo_hm, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=14)
    ax.tick_params(colors=TEXT_PLOT, labelsize=8)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    plt.setp(ax.get_yticklabels(), rotation=0)
    ax.collections[0].colorbar.ax.tick_params(colors=TEXT_PLOT, labelsize=8)
    plt.tight_layout()
    return fig


def _fig_curva_aprendizaje(curvas, target_col, algoritmo, parroquia="", k_splits=5):
    fig, ax = plt.subplots(figsize=(13, 5), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    if not curvas or not curvas.get("train"):
        ax.text(0.5, 0.5, "No hay datos de curva.", ha="center", va="center",
                color=TEXT_PLOT, fontsize=11, transform=ax.transAxes)
        _aplicar_estilo_ax(ax, f"Curva de Aprendizaje — {target_col}", "", "")
        plt.tight_layout(); return fig

    train_curves = curvas["train"]
    val_curves   = curvas.get("val", [])
    metric_name  = curvas.get("metric", "Score")
    is_r2        = metric_name == "R²"

    all_raw = train_curves + (val_curves if val_curves else [])
    lengths = [len(c) for c in all_raw if c]
    if not lengths or min(lengths) < 2:
        ax.text(0.5, 0.5, "Historial demasiado corto.", ha="center", va="center",
                color=TEXT_PLOT, transform=ax.transAxes)
        _aplicar_estilo_ax(ax, f"Curva de Aprendizaje — {target_col}", "", "")
        plt.tight_layout(); return fig

    min_len = min(lengths)
    x = np.arange(min_len)
    tr_arr  = np.array([c[:min_len] for c in train_curves])
    tr_mean = tr_arr.mean(axis=0); tr_std = tr_arr.std(axis=0)
    for c in tr_arr: ax.plot(x, c, color=COLOR_REAL, alpha=0.10, lw=0.75, zorder=2)
    ax.plot(x, tr_mean, color=COLOR_REAL, lw=2.2, label=f"Train — {metric_name}  (μ)", zorder=5)
    ax.fill_between(x, tr_mean-tr_std, tr_mean+tr_std, alpha=0.14, color=COLOR_REAL, zorder=3)

    if val_curves:
        va_arr  = np.array([c[:min_len] for c in val_curves])
        va_mean = va_arr.mean(axis=0); va_std = va_arr.std(axis=0)
        for c in va_arr: ax.plot(x, c, color=COLOR_PRED, alpha=0.10, lw=0.75, zorder=2)
        ax.plot(x, va_mean, color=COLOR_PRED, lw=2.2, linestyle="--",
                label=f"Validación — {metric_name}  (μ)", zorder=5)
        ax.fill_between(x, va_mean-va_std, va_mean+va_std, alpha=0.14, color=COLOR_PRED, zorder=3)
        best_iter = int(np.argmax(va_mean) if is_r2 else np.argmin(va_mean))
        best_val  = va_mean[best_iter]
        ax.axvline(best_iter, color="#F59E0B", lw=1.6, linestyle=":", alpha=0.88,
                   label=f"Mejor iter: {best_iter}  ({best_val:.4f})", zorder=6)

    ax.axhline(tr_mean[-1], color=COLOR_REAL, lw=0.8, linestyle=":", alpha=0.40, zorder=1)
    algo_label = algoritmo.split("(")[0].strip()
    titulo = f"Curva de Aprendizaje — {target_col}  ·  {algo_label}  ·  TSS (K={k_splits})"
    if parroquia: titulo = f"[{parroquia}]  {titulo}"
    ylabel = "R²  (↑ mejor)" if is_r2 else f"{metric_name}  (↓ mejor)"
    _aplicar_estilo_ax(ax, titulo, "Iteración / Boosting Round", ylabel)
    ax.legend(framealpha=0.20, labelcolor=TEXT_PLOT, facecolor=BG_PLOT,
              edgecolor=GRID_PLOT, fontsize=9, loc="best")
    plt.tight_layout()
    return fig


# ──────────────────────────────────────────────────────────────────────────────
# 13.  PIPELINE PRINCIPAL
# ──────────────────────────────────────────────────────────────────────────────

def entrenar(
    csv_upload, target_col, nombre_modelo, algoritmo,
    plot_days, train_ratio, excluir_pandemia, k_splits,
    max_depth, learning_rate, subsample, colsample_bytree,
    reg_lambda, reg_alpha, gamma, min_child_weight,
    usar_feature_engineering,
):
    logs: list[str] = []
    def info(m): log.info(m);    logs.append(f"✅ {m}")
    def warn(m): log.warning(m); logs.append(f"⚠️  {m}")
    def err(m):  log.error(m);   logs.append(f"❌ {m}")
    def _estado(): return "**Registro:**  " + "  ·  ".join(logs[-30:])
    VACIO = (None, None, None, None)

    try:
        if csv_upload is None:
            err("No se subió ningún archivo CSV.")
            return "### ❌ Sube un archivo CSV antes de entrenar.", *VACIO, _estado()

        ruta_csv      = csv_upload if isinstance(csv_upload, str) else csv_upload.name
        target_col    = target_col.strip()
        nombre_modelo = nombre_modelo.strip() or "surrogate_model"
        parroquia     = Path(ruta_csv).stem.replace("_", " ").title()
        info(f"Archivo: {Path(ruta_csv).name}")

        df = _cargar_csv(ruta_csv)
        info(f"CSV cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")

        ts_col = _detectar_timestamp(df)
        if ts_col is None:
            return "### ❌ No se encontró columna Timestamp/Date/Fecha.", *VACIO, _estado()
        info(f"Timestamp: '{ts_col}'")

        if target_col not in df.columns:
            return (f"### ❌ Target **`{target_col}`** no encontrado.\n\n"
                    f"**Columnas:** `{', '.join(df.columns)}`", *VACIO, _estado())

        df_prep = _preprocesar(df, target_col, ts_col, excluir_pandemia)
        if excluir_pandemia: info(f"Pandemia excluida: {ANOS_PANDEMIA}.")

        y = df_prep[target_col]
        X = df_prep.drop(columns=[target_col])

        if usar_feature_engineering:
            info("Aplicando Feature Engineering avanzado…")
            df_full = _agregar_features_avanzadas(pd.concat([X, y], axis=1), target_col)
            y = df_full[target_col]; X = df_full.drop(columns=[target_col])
            X_tr_sel, _, y_tr_sel, _ = _dividir_cronologico(X, y, train_ratio)
            info(f"Seleccionando top-{MAX_FEATURES_SEL} features entre {X_tr_sel.shape[1]}…")
            selected  = _seleccionar_top_features(X_tr_sel, y_tr_sel, MAX_FEATURES_SEL, log_fn=info)
            X         = X[selected]
            feat_cols = list(X.columns)
            info(f"Features seleccionadas: {len(feat_cols)}")
        else:
            num_cols  = X.select_dtypes(include=[np.number]).columns.tolist()
            feat_cols = [c for c in num_cols if c != target_col]
            X         = X[feat_cols]
            info(f"Usando {len(feat_cols)} features originales.")

        n_nulos = int(X.isna().sum().sum())
        if n_nulos: warn(f"{n_nulos:,} NaN en features — eliminando filas.")
        mask = X.notna().all(axis=1); X = X.loc[mask]; y = y.loc[mask]
        df_clean = pd.concat([X, y], axis=1)
        info(f"Dataset limpio: {df_clean.shape[0]:,} filas")

        xgb_params = {
            "max_depth": int(max_depth), "learning_rate": float(learning_rate),
            "subsample": float(subsample), "colsample_bytree": float(colsample_bytree),
            "reg_lambda": float(reg_lambda), "reg_alpha": float(reg_alpha),
            "gamma": float(gamma), "min_child_weight": float(min_child_weight),
        }

        info(f"TimeSeriesSplit (K={k_splits}) · Algoritmo: {algoritmo}")
        kfold_logs = []
        def kf_log(m): log.info(m); kfold_logs.append(m)
        kf_resumen, kf_curvas = _entrenar_kfold(
            df_clean, algoritmo, n_splits=k_splits, log_fn=kf_log, xgb_params=xgb_params
        )
        for l in kfold_logs: logs.append(l)
        if kf_resumen.empty: warn("TSS K-Fold no produjo resultados.")

        fig_lc = _fig_curva_aprendizaje(
            kf_curvas.get(target_col, {}), target_col, algoritmo, parroquia, k_splits
        )

        X_tr, X_te, y_tr, y_te = _dividir_cronologico(X, y, train_ratio)
        sub_corte   = int(len(X_tr) * 0.80)
        X_sub_tr    = X_tr.iloc[:sub_corte]; X_val_sub = X_tr.iloc[sub_corte:]
        y_sub_tr    = y_tr.iloc[:sub_corte]; y_val_sub = y_tr.iloc[sub_corte:]

        scaler_final = StandardScaler()
        X_sub_tr_sc  = scaler_final.fit_transform(X_sub_tr)
        X_val_sub_sc = scaler_final.transform(X_val_sub)
        X_te_sc      = scaler_final.transform(X_te)
        X_tr_sc      = np.vstack([X_sub_tr_sc, X_val_sub_sc])
        y_tr_np      = y_tr.values

        modelo_final = _construir_modelo(algoritmo, xgb_params)
        if "XGBoost" in algoritmo:
            modelo_final.fit(X_sub_tr_sc, y_sub_tr.values,
                             eval_set=[(X_val_sub_sc, y_val_sub.values)], verbose=False)
        elif "LightGBM" in algoritmo:
            import lightgbm as lgb
            modelo_final.fit(X_sub_tr_sc, y_sub_tr.values,
                             eval_set=[(X_val_sub_sc, y_val_sub.values)],
                             callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)])
        else:
            modelo_final.fit(X_sub_tr_sc, y_sub_tr.values)

        y_pred_tr = modelo_final.predict(X_tr_sc)
        y_pred_te = modelo_final.predict(X_te_sc)

        def _m(yt, yp):
            return dict(MAE=mean_absolute_error(yt, yp),
                        RMSE=float(np.sqrt(mean_squared_error(yt, yp))),
                        R2=r2_score(yt, yp))
        m_tr = _m(y_tr_np, y_pred_tr); m_te = _m(y_te.values, y_pred_te)
        gap = m_tr["R2"] - m_te["R2"]
        if gap > 0.15: warn(f"Posible overfitting: ΔR² = {gap:.3f}")

        # ── [NEW-2] Lag lookup + ultimo_timestamp ─────────────────────────────
        info("Generando tabla de lags climatológicos (lag_lookup)…")
        lag_lookup       = _crear_lag_lookup(X_tr, feat_cols)
        ultimo_timestamp = X_tr.index.max() if hasattr(X_tr.index, "max") else None
        info(f"lag_lookup: {len(lag_lookup)} columnas · último dato: {ultimo_timestamp}")

        hw_label       = f"{'🟢 GPU' if GPU_DISPONIBLE else '🔵 CPU'} — {GPU_MSG}"
        pandemia_label = "🚫 2020-2021 excluidos" if excluir_pandemia else "⚠️ Pandemia incluida"

        if not kf_resumen.empty:
            filas_kf = []
            for _, row in kf_resumen.iterrows():
                ico = "🟢" if row["R2_mean"] >= 0.7 else ("🟡" if row["R2_mean"] >= 0.5 else "🔴")
                filas_kf.append(
                    f"| **{row['Contaminante']}** | `{int(row['Features_X'])}` "
                    f"| `{row['MAE_mean']:.3f} ± {row['MAE_std']:.3f}` "
                    f"| `{row['RMSE_mean']:.3f} ± {row['RMSE_std']:.3f}` "
                    f"| {ico} `{row['R2_mean']:.3f} ± {row['R2_std']:.3f}` |"
                )
            tabla_kf = "\n".join(filas_kf)
        else:
            tabla_kf = "| — | — | — | — | — |"

        metricas_md = f"""
## 📊 Surrogate Model v10.4 — *{parroquia}*

### TimeSeriesSplit (K={k_splits}) — 6 Contaminantes

| Contaminante | Features X | MAE (μ ± σ) | RMSE (μ ± σ) | R² (μ ± σ) |
|:------------:|:----------:|:-----------:|:------------:|:----------:|
{tabla_kf}

---

### Modelo Final — `{target_col}`  (Split {int(train_ratio*100)}/{int((1-train_ratio)*100)})

| Métrica | 🟦 Train | 🟧 Test ciego |
|---------|:--------:|:-------------:|
| **MAE** | `{m_tr['MAE']:.4f}` | `{m_te['MAE']:.4f}` |
| **RMSE** | `{m_tr['RMSE']:.4f}` | `{m_te['RMSE']:.4f}` |
| **R²** | `{m_tr['R2']:.4f}` | `{m_te['R2']:.4f}` |
| **Precisión (R² %)** | `{m_tr['R2']*100:.2f}%` | `{m_te['R2']*100:.2f}%` |

{"⚠️ **Posible overfitting** — ΔR² = `" + f"{gap:.3f}`" if gap > 0.15 else "✅ Sin señales de overfitting."}

| Parámetro | Valor |
|-----------|-------|
| Parroquia | `{parroquia}` | Algoritmo | `{algoritmo}` |
| Hardware | {hw_label} | Features | `{len(feat_cols)}` |
| Lag lookup | `{len(lag_lookup)}` columnas | Último dato | `{ultimo_timestamp}` |
| Pandemia | {pandemia_label} | FE | {'✅ top-' + str(MAX_FEATURES_SEL) if usar_feature_engineering else '❌'} |
"""

        perm  = permutation_importance(modelo_final, X_te_sc, y_te.values, n_repeats=8, random_state=42, scoring="r2")
        fi_df = pd.DataFrame({"Feature": feat_cols, "Importance": perm.importances_mean, "Std": perm.importances_std}
                              ).sort_values("Importance", ascending=False).reset_index(drop=True)

        tag = f"{nombre_modelo}_{parroquia.replace(' ', '_')}"
        pkl_path = OUTPUT_DIR / f"{tag}.pkl"
        with open(pkl_path, "wb") as fh:
            pickle.dump({
                "modelo": modelo_final, "scaler": scaler_final,
                "features": feat_cols, "target": target_col,
                "parroquia": parroquia, "kfold_resumen": kf_resumen,
                "kf_curvas": kf_curvas, "lag_lookup": lag_lookup,
                "ultimo_timestamp": ultimo_timestamp,
            }, fh)
        info(f"Modelo guardado: {pkl_path}")
        fi_df.to_csv(OUTPUT_DIR / f"{tag}_feature_importance.csv", index=False)
        if not kf_resumen.empty:
            kf_resumen.to_csv(OUTPUT_DIR / f"{tag}_kfold_resumen.csv", index=False)

        # ── [NEW-2] Actualizar SESION con lag_lookup ──────────────────────────
        SESION.update({
            "modelo":           modelo_final,
            "scaler":           scaler_final,
            "feat_cols":        feat_cols,
            "target":           target_col,
            "parroquia":        parroquia,
            "feat_stats":       {col: {"min": float(X[col].min()), "max": float(X[col].max()), "mean": float(X[col].mean())} for col in feat_cols},
            "lag_lookup":       lag_lookup,
            "ultimo_timestamp": ultimo_timestamp,
        })
        info("Sesión actualizada → pestaña Predicción IQCA lista.")

        y_te_series = pd.Series(y_te.values, index=X_te.index, name=target_col)
        fig_pred = _fig_prediccion(y_te_series, y_pred_te, target_col, plot_days, parroquia)
        fig_fi   = _fig_feature_importance(fi_df, target_col, parroquia)
        fig_hm   = _fig_heatmap(pd.concat([X, y], axis=1), target_col, parroquia)
        info("Figuras generadas.")

        return metricas_md, fig_pred, fig_fi, fig_hm, fig_lc, _estado()

    except ImportError as e:
        err(str(e))
        pkg = str(e).split("'")[-2] if "'" in str(e) else str(e).split()[-1]
        return (f"### ❌ Librería faltante\n```\n{e}\n```\n`pip install {pkg}`", *VACIO, _estado())
    except Exception as e:
        err(str(e))
        return (f"### ❌ Error\n```\n{traceback.format_exc()}\n```", *VACIO, _estado())


# ──────────────────────────────────────────────────────────────────────────────
# 14.  CSS
# ──────────────────────────────────────────────────────────────────────────────

CSS = """
:root {
    --bg:#0F172A; --card:#1E293B; --input:#0D1525; --border:#334155;
    --accent:#6366F1; --accentH:#818CF8; --text:#F1F5F9; --muted:#94A3B8;
    --ok:#22C55E; --warn:#F59E0B; --err:#EF4444; --r:10px;
}
body,.gradio-container{background:var(--bg)!important;color:var(--text)!important;
    font-family:'Inter','Segoe UI',sans-serif!important;}
.gr-group,.gr-box{background:var(--card)!important;border:1px solid var(--border)!important;
    border-radius:var(--r)!important;padding:16px!important;}
input[type="text"],input[type="number"],input[type="email"],textarea,select{
    background:var(--input)!important;color:var(--text)!important;
    border:1px solid var(--border)!important;border-radius:6px!important;}
label,.gr-label{color:var(--muted)!important;font-size:.8rem!important;font-weight:700!important;
    text-transform:uppercase;letter-spacing:.05em!important;}
input[type="checkbox"]{
    -webkit-appearance:checkbox!important;appearance:checkbox!important;
    width:18px!important;height:18px!important;min-width:18px!important;
    margin:0 8px 0 0!important;cursor:pointer!important;
    vertical-align:middle!important;accent-color:var(--accent);}
.gr-checkbox>label,.gr-checkbox label,[data-testid="checkbox"] label{
    display:flex!important;align-items:center!important;color:var(--text)!important;
    font-size:.9rem!important;font-weight:500!important;text-transform:none!important;
    cursor:pointer!important;gap:4px;}
button.primary{background:linear-gradient(135deg,var(--accent),#7C3AED)!important;
    color:#fff!important;border:none!important;border-radius:8px!important;
    font-weight:800!important;font-size:1rem!important;padding:12px 28px!important;
    box-shadow:0 4px 20px rgba(99,102,241,.45);transition:all .15s;}
button.primary:hover{transform:translateY(-2px);box-shadow:0 6px 28px rgba(99,102,241,.6);}
button.secondary{background:var(--card)!important;color:var(--text)!important;
    border:1px solid var(--border)!important;border-radius:8px!important;}
.gr-markdown{color:var(--text)!important;}
.gr-markdown table{border-collapse:collapse;width:100%;}
.gr-markdown th{background:#1E293B;color:var(--accentH);padding:8px 14px;border:1px solid var(--border);}
.gr-markdown td{color:var(--text);padding:7px 14px;border:1px solid var(--border);}
.gr-markdown tr:nth-child(even) td{background:#19253a;}
.gr-file{border:2px dashed var(--accent)!important;border-radius:var(--r)!important;
    background:rgba(99,102,241,.04)!important;}
.hero{text-align:center;padding:24px 0 6px;}
.hero h1{font-size:2rem;font-weight:900;
    background:linear-gradient(90deg,#6366F1,#38BDF8);
    -webkit-background-clip:text;-webkit-text-fill-color:transparent;}
.hero p{color:var(--muted);font-size:.88rem;}
.gpu-badge{display:inline-block;padding:4px 12px;border-radius:20px;
    font-size:.75rem;font-weight:700;margin-top:4px;}
.gpu-on{background:rgba(34,197,94,.18);color:#4ADE80;border:1px solid #22C55E;}
.gpu-off{background:rgba(99,102,241,.15);color:#A5B4FC;border:1px solid #6366F1;}
.logs-box{background:#0B1527!important;border:1px solid #1D3557!important;
    border-radius:8px;padding:10px 14px;font-family:'JetBrains Mono',monospace;
    font-size:.76rem;color:#7DD3FC;max-height:140px;overflow-y:auto;line-height:1.6;}
.fe-badge{background:rgba(99,102,241,.10);border:1px solid #4338CA;
    border-radius:8px;padding:8px 14px;font-size:.78rem;color:#C7D2FE;margin-top:4px;}
.iqca-box{background:rgba(34,197,94,.07);border:1px solid #22C55E;
    border-radius:10px;padding:12px 16px;margin-top:8px;font-size:.82rem;color:#86EFAC;}
"""


# ──────────────────────────────────────────────────────────────────────────────
# 15.  UI
# ──────────────────────────────────────────────────────────────────────────────

MODELOS = [
    "HistGradientBoosting (CPU — sin GPU requerida)",
    "XGBoost  (auto GPU/CPU)",
    "LightGBM (auto GPU/CPU)",
]

gpu_badge_html = (
    f'<span class="gpu-badge gpu-on">🟢 GPU: {GPU_MSG.split(": ")[-1]}</span>'
    if GPU_DISPONIBLE else
    f'<span class="gpu-badge gpu-off">🔵 CPU mode — {GPU_MSG}</span>'
)


def construir_app() -> gr.Blocks:
    with gr.Blocks(
        theme=gr.themes.Base(
            primary_hue="indigo", secondary_hue="sky", neutral_hue="slate",
            font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif"],
        ),
        css=CSS,
        title="Surrogate Model — Calidad del Aire v10.4",
    ) as app:

        gr.HTML(f"""
        <div class="hero">
            <h1>🌬️ Surrogate Model — Calidad del Aire</h1>
            <p>IQCA REMMAQ · Lag Lookup · Dropdown Dinámico · HistGB / XGBoost / LightGBM · v10.4</p>
            {gpu_badge_html}
        </div>
        """)

        with gr.Row(equal_height=False):

            # ── PANEL IZQUIERDO ───────────────────────────────────────────────
            with gr.Column(scale=1, min_width=360):

                with gr.Group():
                    gr.Markdown("### 📂 Dataset")
                    csv_upload = gr.File(
                        label="Arrastra o sube tu archivo CSV",
                        file_types=[".csv"], type="filepath",
                    )
                    gr.Markdown("_El nombre del archivo se usará como identificador de parroquia._")

                gr.HTML("<div style='height:8px'/>")

                with gr.Group():
                    gr.Markdown("### ⚙️ Configuración del Modelo")

                    with gr.Row():
                        # [NEW-1] Dropdown dinámico — se actualiza al subir el CSV
                        target_input = gr.Dropdown(
                            label="Variable objetivo (Target)",
                            choices=CONTAMINANTES_Y,
                            value="PM25",
                            allow_custom_value=True,
                            scale=3,
                        )
                        nombre_modelo_input = gr.Textbox(
                            label="Nombre del modelo (.pkl)",
                            value="surrogate_calidad_aire", scale=3,
                        )

                    algoritmo_radio = gr.Radio(
                        label="Algoritmo", choices=MODELOS, value=MODELOS[0],
                    )

                    with gr.Row():
                        excluir_pandemia_chk = gr.Checkbox(
                            label="🚫 Excluir pandemia (2020–2021)", value=True, scale=1,
                        )
                        usar_fe_ck = gr.Checkbox(
                            label="🧪 Feature Engineering", value=False, scale=1,
                        )

                    gr.HTML(
                        '<div class="fe-badge">'
                        '<b>FE</b>: rolling stats, diffs, interacciones meteo, '
                        'día semana → top-40 con RandomForest.'
                        '</div>'
                    )

                    with gr.Row():
                        train_ratio_slider = gr.Slider(
                            label="Train / Test split", minimum=0.6, maximum=0.95, step=0.05, value=0.80, scale=3,
                        )
                        k_splits_slider = gr.Slider(
                            label="K — pliegues TSCV", minimum=2, maximum=15, step=1, value=5, scale=2,
                        )
                        plot_days_slider = gr.Slider(
                            label="Días a graficar", minimum=3, maximum=30, step=1, value=7, scale=2,
                        )

                gr.HTML("<div style='height:8px'/>")

                with gr.Accordion("🧪 Hiperparámetros XGBoost / LightGBM / HistGB", open=False):
                    gr.Markdown("_`learning_rate`, `max_depth`, `reg_lambda` aplican también a HistGB._")
                    with gr.Row():
                        max_depth_slider        = gr.Slider(label="max_depth",        minimum=2,    maximum=10,   step=1,    value=5)
                        learning_rate_slider    = gr.Slider(label="learning_rate",    minimum=0.01, maximum=0.3,  step=0.01, value=0.05)
                    with gr.Row():
                        subsample_slider        = gr.Slider(label="subsample",        minimum=0.5,  maximum=1.0,  step=0.05, value=0.8)
                        colsample_bytree_slider = gr.Slider(label="colsample_bytree", minimum=0.5,  maximum=1.0,  step=0.05, value=0.8)
                    with gr.Row():
                        reg_lambda_slider       = gr.Slider(label="reg_lambda (L2)",  minimum=0.0,  maximum=10.0, step=0.5,  value=2.0)
                        reg_alpha_slider        = gr.Slider(label="reg_alpha (L1)",   minimum=0.0,  maximum=5.0,  step=0.1,  value=0.5)
                    with gr.Row():
                        gamma_slider            = gr.Slider(label="gamma",            minimum=0.0,  maximum=5.0,  step=0.1,  value=0.0)
                        min_child_weight_slider = gr.Slider(label="min_child_weight", minimum=1,    maximum=20,   step=1,    value=1)

                gr.HTML("<div style='height:10px'/>")
                btn_train = gr.Button("🚀  Iniciar Entrenamiento", variant="primary", size="lg")
                gr.Markdown("##### 🖥️ Registro de ejecución")
                estado_output = gr.Markdown(value="_Esperando…_", elem_classes=["logs-box"])

            # ── PANEL DERECHO ─────────────────────────────────────────────────
            with gr.Column(scale=2, min_width=580):
                with gr.Tabs():

                    with gr.Tab("📊 Métricas"):
                        metricas_output = gr.Markdown(value="*Entrena para ver métricas.*")

                    with gr.Tab("📈 Real vs Predicho"):
                        fig_pred_output = gr.Plot()

                    with gr.Tab("📉 Curva de Aprendizaje"):
                        gr.Markdown("Línea continua = Train · Discontinua = Validación · Banda = ±1σ")
                        fig_lc_output = gr.Plot()

                    with gr.Tab("🔍 Feature Importance"):
                        gr.Markdown("Importancia por permutación en test ciego.")
                        fig_fi_output = gr.Plot()

                    with gr.Tab("📊 Correlaciones"):
                        gr.Markdown("Correlación de Pearson. Columna target resaltada en naranja.")
                        fig_hm_output = gr.Plot()

                    # ── [NEW-3] Pestaña predicción simplificada + IQCA ────────
                    with gr.Tab("🌿 Predicción IQCA"):
                        gr.Markdown(
                            "### 🌿 Predicción simplificada con IQCA REMMAQ\n\n"
                            "Ingresa solo las variables meteorológicas básicas. "
                            "Los **lags y rolling stats** se imputan automáticamente "
                            "desde la tabla climatológica por hora y mes del dataset de entrenamiento.\n\n"
                            "> _Entrena primero un modelo para activar esta sección._"
                        )

                        gr.HTML('<div class="iqca-box">📌 <b>IQCA REMMAQ</b>: '
                                '0–50 Deseable · 51–100 Aceptable · 101–150 Precaución · '
                                '151–200 Alerta · 201–300 Alarma · 301–500 Emergencia</div>')

                        with gr.Group():
                            # Fecha / hora como Textbox (compatible con todas las versiones de Gradio)
                            fecha_hora_input = gr.Textbox(
                                label="📅 Fecha y Hora (YYYY-MM-DD HH:MM)",
                                placeholder="2024-03-15 14:00",
                                value="",
                            )
                            with gr.Row():
                                temp_input   = gr.Slider(label="🌡️ Temperatura (°C)",         minimum=-10, maximum=40,  step=0.1, value=18.0, scale=1)
                                hum_input    = gr.Slider(label="💧 Humedad (%)",               minimum=0,   maximum=100, step=1,   value=70,   scale=1)
                            with gr.Row():
                                vvel_input   = gr.Slider(label="💨 Velocidad del viento (m/s)", minimum=0,   maximum=30,  step=0.1, value=2.0,  scale=1)
                                vdir_input   = gr.Slider(label="🧭 Dirección del viento (°)",  minimum=0,   maximum=360, step=1,   value=180,  scale=1)
                            precip_input = gr.Slider(label="🌧️ Precipitación (mm)", minimum=0, maximum=200, step=0.1, value=0.0)

                        btn_predecir   = gr.Button("🔮  Estimar Calidad del Aire + IQCA", variant="primary")
                        resultado_pred = gr.Markdown(value="_El resultado aparecerá aquí tras entrenar un modelo._")

        gr.HTML("""
        <div style="text-align:center;padding:18px 0 8px;color:#475569;font-size:.76rem;">
            Surrogate Model v10.4 · IQCA REMMAQ · Lag Lookup · Dropdown Dinámico · Auto-GPU
        </div>
        """)

        # ── Eventos [NEW-4] ───────────────────────────────────────────────────

        # [NEW-1] Actualizar dropdown de target cuando se sube el CSV
        csv_upload.change(
            fn=actualizar_targets,
            inputs=[csv_upload],
            outputs=[target_input],
        )

        # Entrenamiento
        btn_train.click(
            fn=entrenar,
            inputs=[
                csv_upload, target_input, nombre_modelo_input,
                algoritmo_radio, plot_days_slider, train_ratio_slider,
                excluir_pandemia_chk, k_splits_slider,
                max_depth_slider, learning_rate_slider, subsample_slider,
                colsample_bytree_slider, reg_lambda_slider, reg_alpha_slider,
                gamma_slider, min_child_weight_slider, usar_fe_ck,
            ],
            outputs=[
                metricas_output, fig_pred_output, fig_fi_output,
                fig_hm_output, fig_lc_output, estado_output,
            ],
        )

        # [NEW-3] Predicción simplificada → IQCA
        btn_predecir.click(
            fn=predecir_simple,
            inputs=[
                fecha_hora_input,
                temp_input, hum_input,
                vvel_input, vdir_input,
                precip_input,
            ],
            outputs=[resultado_pred],
        )

    return app


# ──────────────────────────────────────────────────────────────────────────────
# 16.  PUNTO DE ENTRADA
# ──────────────────────────────────────────────────────────────────────────────

def _imprimir_ip_fallback() -> None:
    import socket
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8", 80)); print(f"  🖥️  IP local: http://{s.getsockname()[0]}:<PUERTO>"); s.close()
    except Exception: pass
    try:
        ips = subprocess.check_output(["hostname", "-I"], text=True, timeout=4).strip()
        print(f"  🌐  IPs servidor: {ips}")
    except Exception: pass


if __name__ == "__main__":
    app = construir_app()
    print("\n" + "═" * 68)
    print("  🌬️  SURROGATE MODEL — Calidad del Aire  v10.4")
    print("  ✨  IQCA REMMAQ · Lag Lookup · Dropdown Dinámico")
    print("═" * 68)
    print(f"  Hardware  : {GPU_MSG}")
    print(f"  GPU activa: {GPU_DISPONIBLE}")
    print(f"  FE top-N  : {MAX_FEATURES_SEL} features (RandomForest selector)")
    print("═" * 68)

    try:
        app.launch(server_name="0.0.0.0", server_port=None, share=True,
                   max_threads=40, debug=True, show_error=True,
                   prevent_thread_lock=False, quiet=False)
    except OSError:
        app.launch(server_name="0.0.0.0", server_port=7861, share=True, max_threads=40)
    except Exception:
        _imprimir_ip_fallback()
        app.launch(server_name="0.0.0.0", server_port=None, share=False, max_threads=40)

19:10:28 [INFO] GPU detectada (nvidia-smi): Tesla V100-PCIE-16GB, 16384 MiB



════════════════════════════════════════════════════════════════════
  🌬️  SURROGATE MODEL — Calidad del Aire  v10.4
  ✨  IQCA REMMAQ · Lag Lookup · Dropdown Dinámico
════════════════════════════════════════════════════════════════════
  Hardware  : GPU detectada (nvidia-smi): Tesla V100-PCIE-16GB, 16384 MiB
  GPU activa: True
  FE top-N  : 40 features (RandomForest selector)
════════════════════════════════════════════════════════════════════


Version con nueva interfaz y graficas echa por claude

In [ ]:
"""
================================================================================
  SURROGATE MODEL — CALIDAD DEL AIRE  |  Interfaz Web Gradio
  Versión : 10.5  (Material Design 3 Dark · Visual Overhaul)
  ─── Novedades v10.5 ────────────────────────────────────────────────────────
  [VIS-1]  CSS completamente reescrito con paleta Material Design 3 oscuro:
           #141218 bg · #CFBCFF primario · #CDC0E9 secundario · #E7C365 ámbar
           Inter para texto, JetBrains Mono para código/logs.
  [VIS-2]  Layout sidebar-like: columna izquierda (controles) + columna
           derecha (8 tabs con resultados). Cards con gr.HTML decorativos.
  [VIS-3]  Resumen automático de CSV al subir: rows, cols, timestamp,
           contaminantes disponibles con badges de color.
  [VIS-4]  Badge de estado: ⚪ Esperando / ✅ Completado / ❌ Error.
  [VIS-5]  Tres nuevas gráficas integradas:
           · _fig_mae_comparativo() → tab "📊 Métricas" (barras MAE/RMSE)
           · _fig_r2_comparativo()  → tab "📊 Rendimiento" (barras R² semánticas)
           · _fig_pie_importancia() → tab "🥧 Importancia" (donut chart)
  ─── Novedades heredadas de v10.4 ───────────────────────────────────────────
  [NEW-1]  Dropdown dinámico de target según contaminantes del CSV.
  [NEW-2]  Tabla de lags climatológicos (lag_lookup por hora/mes).
  [NEW-3]  Predicción simplificada + IQCA REMMAQ oficial.
  [NEW-4]  Feature Engineering avanzado (rolling, diffs, interacciones).
================================================================================
"""

# ──────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTACIONES
# ──────────────────────────────────────────────────────────────────────────────
import os
import warnings
import logging
import traceback
import subprocess
import pickle
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

import gradio as gr
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# ──────────────────────────────────────────────────────────────────────────────
# 1.  CONSTANTES
# ──────────────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("resultados_surrogate")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SESION: dict = {
    "modelo":           None,
    "scaler":           None,
    "feat_cols":        [],
    "target":           "",
    "parroquia":        "",
    "feat_stats":       {},
    "lag_lookup":       {},
    "ultimo_timestamp": None,
}

ANOS_PANDEMIA   = [2020, 2021]
CONTAMINANTES_Y = ["PM25", "PM10", "O3", "CO", "NO2", "SO2"]

FEATURES_X_BASE = [
    "Temperatura", "Humedad", "Viento_Velocidad", "Viento_Direccion", "Precipitacion",
    "hora_sin", "hora_cos", "mes_sin", "mes_cos",
    "PM25_lag_1h",  "PM25_lag_24h",
    "PM10_lag_1h",  "PM10_lag_24h",
    "O3_lag_1h",    "O3_lag_24h",
    "CO_lag_1h",    "CO_lag_24h",
    "NO2_lag_1h",   "NO2_lag_24h",
    "SO2_lag_1h",   "SO2_lag_24h",
]

METEO_COLS      = ["Temperatura", "Humedad", "Viento_Velocidad", "Viento_Direccion", "Precipitacion"]
MAX_FEATURES_SEL = 40

# ── Tema oscuro MD3 ───────────────────────────────────────────────────────────
COLOR_REAL   = "#CFBCFF"   # primario MD3
COLOR_PRED   = "#E7C365"   # ámbar MD3
COLOR_POS    = "#10B981"   # éxito
COLOR_NEG    = "#F87171"   # error suave
COLOR_SEC    = "#CDC0E9"   # secundario MD3
BG_PLOT      = "#141218"
TEXT_PLOT    = "#E6E0E9"
GRID_PLOT    = "#36343A"
SURF_PLOT    = "#211F24"


# ──────────────────────────────────────────────────────────────────────────────
# 2.  DETECCIÓN GPU
# ──────────────────────────────────────────────────────────────────────────────

def detectar_gpu() -> tuple[bool, str]:
    try:
        r = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=8,
        )
        if r.returncode == 0 and r.stdout.strip():
            return True, f"GPU detectada (nvidia-smi): {r.stdout.strip().split(chr(10))[0]}"
    except Exception:
        pass
    try:
        import torch
        if torch.cuda.is_available():
            return True, f"GPU detectada (torch.cuda): {torch.cuda.get_device_name(0)}"
    except ImportError:
        pass
    try:
        import cupy  # noqa: F401
        return True, "GPU detectada (cupy disponible)"
    except ImportError:
        pass
    return False, "No se detectó GPU — se usará CPU"


GPU_DISPONIBLE, GPU_MSG = detectar_gpu()
log.info(GPU_MSG)


def _xgb_tree_method() -> dict:
    return {"tree_method": "hist", "device": "cuda"} if GPU_DISPONIBLE \
           else {"tree_method": "hist", "device": "cpu"}

def _lgbm_device() -> str:
    return "gpu" if GPU_DISPONIBLE else "cpu"


# ──────────────────────────────────────────────────────────────────────────────
# 3.  CARGA Y DETECCIÓN DE TIMESTAMP
# ──────────────────────────────────────────────────────────────────────────────

def _cargar_csv(ruta: str) -> pd.DataFrame:
    try:
        return pd.read_csv(ruta, comment="#", low_memory=False, on_bad_lines="warn")
    except Exception as e:
        raise RuntimeError(f"Error al leer '{ruta}': {e}") from e


def _detectar_timestamp(df: pd.DataFrame) -> str | None:
    keywords = ("time", "fecha", "date", "hora", "datetime", "timestamp")
    candidatos = [c for c in df.columns if any(k in c.lower() for k in keywords)]
    if candidatos:
        return candidatos[0]
    for c in df.columns:
        try:
            pd.to_datetime(df[c].dropna().astype(str).iloc[:10], infer_datetime_format=True)
            return c
        except Exception:
            continue
    return None


# ──────────────────────────────────────────────────────────────────────────────
# 4.  DROPDOWN DINÁMICO DE TARGET
# ──────────────────────────────────────────────────────────────────────────────

def actualizar_targets(archivo) -> gr.Dropdown:
    if archivo is None:
        return gr.Dropdown(choices=CONTAMINANTES_Y, value="PM25")
    try:
        ruta = archivo if isinstance(archivo, str) else archivo.name
        df_head = pd.read_csv(ruta, comment="#", nrows=3, low_memory=False)
        cols_csv = set(df_head.columns.tolist())
        disponibles = [c for c in CONTAMINANTES_Y if c in cols_csv]
        if not disponibles:
            disponibles = ["PM25"]
        return gr.Dropdown(choices=disponibles, value=disponibles[0])
    except Exception:
        return gr.Dropdown(choices=CONTAMINANTES_Y, value="PM25")


# ──────────────────────────────────────────────────────────────────────────────
# 5.  PREPROCESAMIENTO
# ──────────────────────────────────────────────────────────────────────────────

def _preprocesar(df: pd.DataFrame, target_col: str, timestamp_col: str,
                 excluir_pandemia: bool = True) -> pd.DataFrame:
    df = df.copy()
    df[timestamp_col] = pd.to_datetime(
        df[timestamp_col], infer_datetime_format=True, errors="coerce"
    )
    df = df.dropna(subset=[timestamp_col]).set_index(timestamp_col).sort_index()
    if excluir_pandemia:
        df = df[~df.index.year.isin(ANOS_PANDEMIA)]
    obj_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
    if obj_cols:
        df[obj_cols] = df[obj_cols].apply(pd.to_numeric, errors="coerce")
    df = df.dropna(subset=[target_col])
    return df


def _dividir_cronologico(X, y, ratio: float = 0.80):
    corte = int(len(X) * ratio)
    return X.iloc[:corte], X.iloc[corte:], y.iloc[:corte], y.iloc[corte:]


def _resolver_features_x(df: pd.DataFrame, target_col: str) -> list[str]:
    cols = set(df.select_dtypes(include=[np.number]).columns)
    excluir = set(CONTAMINANTES_Y)
    return [c for c in cols if c not in excluir and c != target_col]


# ──────────────────────────────────────────────────────────────────────────────
# 6.  FEATURE ENGINEERING
# ──────────────────────────────────────────────────────────────────────────────

def _agregar_features_avanzadas(df: pd.DataFrame, target_col: str) -> pd.DataFrame:
    df = df.copy()
    contaminantes = [c for c in CONTAMINANTES_Y if c in df.columns]
    for col in contaminantes:
        for window in [3, 6, 12, 24]:
            df[f"{col}_roll_mean_{window}h"] = df[col].rolling(window, min_periods=1).mean()
            if window >= 6:
                df[f"{col}_roll_std_{window}h"] = df[col].rolling(window, min_periods=2).std()
        for lag in [1, 3, 6, 12, 24]:
            df[f"{col}_diff_{lag}h"] = df[col].diff(lag)
    if "Temperatura" in df.columns and "Humedad" in df.columns:
        df["temp_hum"] = df["Temperatura"] * df["Humedad"]
    if "Temperatura" in df.columns and "Viento_Velocidad" in df.columns:
        df["temp_wind"] = df["Temperatura"] * df["Viento_Velocidad"]
    if "Viento_Direccion" in df.columns and "Viento_Velocidad" in df.columns:
        dir_rad = np.radians(df["Viento_Direccion"])
        df["wind_u"] = df["Viento_Velocidad"] * np.cos(dir_rad)
        df["wind_v"] = df["Viento_Velocidad"] * np.sin(dir_rad)
    if hasattr(df.index, "weekday"):
        df["dia_semana"]     = df.index.weekday
        df["dia_semana_sin"] = np.sin(2 * np.pi * df["dia_semana"] / 7)
        df["dia_semana_cos"] = np.cos(2 * np.pi * df["dia_semana"] / 7)
        df["es_finde"]       = (df["dia_semana"] >= 5).astype(int)
    return df.dropna()


def _seleccionar_top_features(
    X_tr: pd.DataFrame, y_tr: pd.Series,
    n_features: int = MAX_FEATURES_SEL, log_fn=None,
) -> list[str]:
    if log_fn is None:
        log_fn = log.info
    all_cols = X_tr.columns.tolist()
    if len(all_cols) <= n_features:
        log_fn(f"  Todas las {len(all_cols)} features disponibles (≤ {n_features}).")
        return all_cols
    try:
        mask_ok = X_tr.notna().all(axis=1) & y_tr.notna()
        X_ok, y_ok = X_tr[mask_ok].values, y_tr[mask_ok].values
        if len(X_ok) < 50:
            log_fn("  ⚠️  Datos insuficientes para selección — usando todas.")
            return all_cols
        rf_sel = RandomForestRegressor(n_estimators=80, max_depth=6,
                                       min_samples_leaf=20, n_jobs=-1, random_state=42)
        rf_sel.fit(X_ok, y_ok)
        selector = SelectFromModel(rf_sel, threshold=-np.inf,
                                   max_features=n_features, prefit=True)
        selected = [c for c, k in zip(all_cols, selector.get_support()) if k]
        log_fn(f"  SelectFromModel: {len(all_cols)} → {len(selected)} features.")
        return selected if selected else all_cols
    except Exception as exc:
        log_fn(f"  ⚠️  Selección falló ({exc}) — usando todas las features.")
        return all_cols


# ──────────────────────────────────────────────────────────────────────────────
# 7.  TABLA DE LAGS CLIMATOLÓGICOS
# ──────────────────────────────────────────────────────────────────────────────

def _crear_lag_lookup(X_train: pd.DataFrame, feat_cols: list[str]) -> dict:
    lookup: dict = {}
    if not hasattr(X_train.index, "hour"):
        return lookup
    df_tmp = X_train[feat_cols].copy()
    df_tmp["_hora"] = X_train.index.hour
    df_tmp["_mes"]  = X_train.index.month
    for col in feat_cols:
        if col not in df_tmp.columns:
            continue
        grupo = df_tmp.groupby(["_hora", "_mes"])[col].mean().dropna()
        lookup[col] = {(int(h), int(m)): float(v) for (h, m), v in grupo.items()}
    return lookup


# ──────────────────────────────────────────────────────────────────────────────
# 8.  IQCA REMMAQ
# ──────────────────────────────────────────────────────────────────────────────

IQCA_BREAKPOINTS: dict[str, list[tuple]] = {
    "PM25": [(0.0,12.0,0,50),(12.1,37.4,51,100),(37.5,55.4,101,150),
             (55.5,150.4,151,200),(150.5,250.4,201,300),(250.5,500.4,301,500)],
    "PM10": [(0,54,0,50),(55,154,51,100),(155,254,101,150),
             (255,354,151,200),(355,424,201,300),(425,604,301,500)],
    "O3":   [(0,54,0,50),(55,124,51,100),(125,164,101,150),
             (165,204,151,200),(205,404,201,300),(405,604,301,500)],
    "CO":   [(0.0,4.4,0,50),(4.5,9.4,51,100),(9.5,12.4,101,150),
             (12.5,15.4,151,200),(15.5,30.4,201,300),(30.5,50.4,301,500)],
    "NO2":  [(0,53,0,50),(54,100,51,100),(101,360,101,150),
             (361,649,151,200),(650,1249,201,300),(1250,2049,301,500)],
    "SO2":  [(0,35,0,50),(36,75,51,100),(76,185,101,150),
             (186,304,151,200),(305,604,201,300),(605,1004,301,500)],
}
CATEGORIAS_IQCA = [
    (0,   50,  "🟢 Deseable",   "#10B981"),
    (51,  100, "🟡 Aceptable",  "#EAB308"),
    (101, 150, "🟠 Precaución", "#F97316"),
    (151, 200, "🔴 Alerta",     "#EF4444"),
    (201, 300, "🟣 Alarma",     "#A855F7"),
    (301, 500, "⚫ Emergencia",  "#36343A"),
]


def calcular_iqca(contaminante: str, concentracion: float) -> float | None:
    bp_list = IQCA_BREAKPOINTS.get(contaminante)
    if bp_list is None:
        return None
    for (c_lo, c_hi, i_lo, i_hi) in bp_list:
        if c_lo <= concentracion <= c_hi:
            return i_lo + (concentracion - c_lo) * (i_hi - i_lo) / (c_hi - c_lo)
    if concentracion > bp_list[-1][1]:
        return 500.0
    return 0.0


def categoria_iqca(iqca: float) -> tuple[str, str]:
    for (lo, hi, label, color) in CATEGORIAS_IQCA:
        if lo <= iqca <= hi:
            return label, color
    return "⚫ Emergencia", "#36343A"


# ──────────────────────────────────────────────────────────────────────────────
# 9.  PREDICCIÓN SIMPLIFICADA
# ──────────────────────────────────────────────────────────────────────────────

def predecir_simple(fecha_hora, temperatura, humedad, viento_vel, viento_dir, precipitacion) -> str:
    if SESION["modelo"] is None:
        return "### ⚠️ No hay modelo entrenado.\nEntrena primero un modelo."
    try:
        if fecha_hora is None or str(fecha_hora).strip() == "":
            return "### ❌ Ingresa una fecha y hora válida."
        try:
            ts = pd.Timestamp(str(fecha_hora))
        except Exception:
            return f"### ❌ Formato inválido: `{fecha_hora}`\nUsa: `YYYY-MM-DD HH:MM`"
        hora = ts.hour
        mes  = ts.month
        ultimo = SESION.get("ultimo_timestamp")
        advertencia = ""
        if ultimo is not None:
            delta_h = (ts - ultimo).total_seconds() / 3600
            if delta_h > 48:
                advertencia = (
                    f"\n\n> ⚠️ **Extrapolación**: la fecha pedida está **{delta_h:.0f}h** "
                    f"después del último dato ({ultimo.strftime('%Y-%m-%d %H:%M')}). "
                    f"Los lags se imputarán desde promedios climatológicos."
                )
        ciclicas = {
            "hora_sin": float(np.sin(2 * np.pi * hora / 24)),
            "hora_cos": float(np.cos(2 * np.pi * hora / 24)),
            "mes_sin":  float(np.sin(2 * np.pi * mes  / 12)),
            "mes_cos":  float(np.cos(2 * np.pi * mes  / 12)),
        }
        feat_cols  = SESION["feat_cols"]
        feat_stats = SESION["feat_stats"]
        lag_lookup = SESION.get("lag_lookup", {})
        meteo_vals = {
            "Temperatura":      float(temperatura),
            "Humedad":          float(humedad),
            "Viento_Velocidad": float(viento_vel),
            "Viento_Direccion": float(viento_dir),
            "Precipitacion":    float(precipitacion),
        }
        dir_rad = np.radians(float(viento_dir))
        derivadas = {
            "temp_hum":  float(temperatura) * float(humedad),
            "temp_wind": float(temperatura) * float(viento_vel),
            "wind_u":    float(viento_vel)  * np.cos(dir_rad),
            "wind_v":    float(viento_vel)  * np.sin(dir_rad),
            "dia_semana":     float(ts.weekday()),
            "dia_semana_sin": float(np.sin(2 * np.pi * ts.weekday() / 7)),
            "dia_semana_cos": float(np.cos(2 * np.pi * ts.weekday() / 7)),
            "es_finde":       float(1 if ts.weekday() >= 5 else 0),
        }
        row: dict = {}
        for col in feat_cols:
            if col in meteo_vals:
                row[col] = meteo_vals[col]
            elif col in ciclicas:
                row[col] = ciclicas[col]
            elif col in derivadas:
                row[col] = derivadas[col]
            elif col in lag_lookup and (hora, mes) in lag_lookup[col]:
                row[col] = lag_lookup[col][(hora, mes)]
            else:
                row[col] = feat_stats.get(col, {}).get("mean", 0.0)
        X_input = pd.DataFrame([row])[feat_cols]
        scaler  = SESION["scaler"]
        X_sc    = scaler.transform(X_input) if scaler else X_input.values
        pred    = float(SESION["modelo"].predict(X_sc)[0])
        target    = SESION["target"]
        parroquia = SESION["parroquia"]
        iqca_val  = calcular_iqca(target, pred)
        if iqca_val is not None:
            cat_label, _ = categoria_iqca(iqca_val)
            iqca_row = (f"| **IQCA** | `{iqca_val:.1f}` |\n"
                        f"| **Categoría REMMAQ** | {cat_label} |")
        else:
            iqca_row = f"| **IQCA** | _N/D para {target}_ |"
        n_lookup = sum(1 for col in feat_cols if col in lag_lookup
                       and (hora, mes) in lag_lookup.get(col, {}))
        return (
            f"## 🔮 Predicción IQCA — `{target}` · *{parroquia}*\n\n"
            f"| Campo | Valor |\n|-------|-------|\n"
            f"| **Fecha / Hora** | `{ts.strftime('%Y-%m-%d %H:%M')}` |\n"
            f"| **{target} estimado** | `{pred:.3f} µg/m³` |\n"
            f"{iqca_row}\n\n---\n\n"
            f"**Imputación** → `{n_lookup}` lags desde tabla climatológica"
            f"{advertencia}"
        )
    except Exception:
        return f"### ❌ Error\n```\n{traceback.format_exc()}\n```"


# ──────────────────────────────────────────────────────────────────────────────
# 10. CONSTRUCCIÓN DEL MODELO
# ──────────────────────────────────────────────────────────────────────────────

def _construir_modelo(algoritmo: str, params: dict | None = None):
    if params is None:
        params = {}
    if "XGBoost" in algoritmo:
        import xgboost as xgb
        return xgb.XGBRegressor(
            n_estimators=2000,
            max_depth=int(params.get("max_depth", 5)),
            learning_rate=float(params.get("learning_rate", 0.05)),
            subsample=float(params.get("subsample", 0.8)),
            colsample_bytree=float(params.get("colsample_bytree", 0.8)),
            reg_lambda=float(params.get("reg_lambda", 2.0)),
            reg_alpha=float(params.get("reg_alpha", 0.5)),
            gamma=float(params.get("gamma", 0.0)),
            min_child_weight=float(params.get("min_child_weight", 1)),
            early_stopping_rounds=50, eval_metric="rmse",
            random_state=42, verbosity=0, **_xgb_tree_method(),
        )
    elif "LightGBM" in algoritmo:
        import lightgbm as lgb
        return lgb.LGBMRegressor(
            n_estimators=2000,
            max_depth=int(params.get("max_depth", 5)),
            learning_rate=float(params.get("learning_rate", 0.05)),
            subsample=float(params.get("subsample", 0.8)),
            colsample_bytree=float(params.get("colsample_bytree", 0.8)),
            reg_lambda=float(params.get("reg_lambda", 2.0)),
            reg_alpha=float(params.get("reg_alpha", 0.5)),
            min_child_samples=int(params.get("min_child_weight", 30)),
            metric="rmse", device=_lgbm_device(), random_state=42, verbose=-1,
        )
    else:
        return HistGradientBoostingRegressor(
            max_iter=1000, early_stopping=True, n_iter_no_change=30,
            validation_fraction=0.1,
            max_depth=int(params.get("max_depth", 5)),
            min_samples_leaf=int(params.get("min_child_weight", 30)),
            learning_rate=float(params.get("learning_rate", 0.05)),
            l2_regularization=float(params.get("reg_lambda", 1.0)),
            random_state=42,
        )


# ──────────────────────────────────────────────────────────────────────────────
# 11. PIPELINE TimeSeriesSplit
# ──────────────────────────────────────────────────────────────────────────────

def _extraer_curvas_fold(modelo, algoritmo: str) -> tuple[list, list, str]:
    train_hist, val_hist, metric_name = [], [], "Score"
    try:
        if "XGBoost" in algoritmo:
            evals = modelo.evals_result()
            ks = list(evals.keys()); met = list(evals[ks[0]].keys())[0]
            metric_name = met.upper()
            train_hist = list(evals[ks[0]][met])
            val_hist   = list(evals[ks[1]][met]) if len(ks) > 1 else []
        elif "LightGBM" in algoritmo:
            evals = modelo.evals_result_
            ks = list(evals.keys()); met = list(evals[ks[0]].keys())[0]
            metric_name = met.upper()
            train_hist = list(evals[ks[0]][met])
            val_hist   = list(evals[ks[1]][met]) if len(ks) > 1 else []
        else:
            if hasattr(modelo, "train_score_") and modelo.train_score_ is not None:
                train_hist = list(modelo.train_score_); metric_name = "R²"
            if hasattr(modelo, "validation_score_") and modelo.validation_score_ is not None:
                val_hist = list(modelo.validation_score_)
    except Exception as exc:
        log.warning(f"_extraer_curvas_fold: {exc}")
    return train_hist, val_hist, metric_name


def _entrenar_kfold(df, algoritmo, n_splits=5, log_fn=None, xgb_params=None):
    if log_fn is None:
        log_fn = log.info
    tscv = TimeSeriesSplit(n_splits=n_splits)
    filas, all_curves = [], {}
    cols_df = set(df.columns)
    for contaminante in CONTAMINANTES_Y:
        if contaminante not in cols_df:
            log_fn(f"  ⏭️  {contaminante} no disponible — omitido."); continue
        feat_cols = _resolver_features_x(df, contaminante)
        y_vals = df[contaminante].dropna()
        X_vals = df.loc[y_vals.index, feat_cols]
        mask_ok = X_vals.notna().all(axis=1) & y_vals.notna()
        X_vals, y_vals = X_vals[mask_ok], y_vals[mask_ok]
        if len(X_vals) < n_splits * 20:
            log_fn(f"  ⚠️  {contaminante}: {len(X_vals)} filas — omitido."); continue
        X_arr, y_arr = X_vals.values, y_vals.values
        fold_metrics, fold_curves_tr, fold_curves_va = [], [], []
        metric_name_cv = "Score"
        for fold_idx, (tr_idx, va_idx) in enumerate(tscv.split(X_arr), 1):
            X_tr, X_va = X_arr[tr_idx], X_arr[va_idx]
            y_tr, y_va = y_arr[tr_idx], y_arr[va_idx]
            scaler = StandardScaler()
            X_tr_sc = scaler.fit_transform(X_tr); X_va_sc = scaler.transform(X_va)
            modelo = _construir_modelo(algoritmo, xgb_params)
            if "XGBoost" in algoritmo:
                modelo.fit(X_tr_sc, y_tr, eval_set=[(X_tr_sc, y_tr), (X_va_sc, y_va)], verbose=False)
            elif "LightGBM" in algoritmo:
                import lightgbm as lgb
                modelo.fit(X_tr_sc, y_tr, eval_set=[(X_tr_sc, y_tr), (X_va_sc, y_va)],
                           callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)])
            else:
                modelo.fit(X_tr_sc, y_tr)
            tr_h, va_h, met = _extraer_curvas_fold(modelo, algoritmo)
            if tr_h: fold_curves_tr.append(tr_h)
            if va_h: fold_curves_va.append(va_h)
            metric_name_cv = met
            y_pred = modelo.predict(X_va_sc)
            fold_metrics.append({
                "MAE":  mean_absolute_error(y_va, y_pred),
                "RMSE": float(np.sqrt(mean_squared_error(y_va, y_pred))),
                "R2":   r2_score(y_va, y_pred),
            })
            log_fn(f"  {contaminante} | Fold {fold_idx}/{n_splits} → "
                   f"MAE={fold_metrics[-1]['MAE']:.3f} RMSE={fold_metrics[-1]['RMSE']:.3f} "
                   f"R²={fold_metrics[-1]['R2']:.3f}")
        all_curves[contaminante] = {"train": fold_curves_tr, "val": fold_curves_va, "metric": metric_name_cv}
        mf = pd.DataFrame(fold_metrics)
        filas.append({
            "Contaminante": contaminante, "Features_X": len(feat_cols),
            "MAE_mean": mf["MAE"].mean(), "MAE_std": mf["MAE"].std(),
            "RMSE_mean": mf["RMSE"].mean(), "RMSE_std": mf["RMSE"].std(),
            "R2_mean": mf["R2"].mean(), "R2_std": mf["R2"].std(),
        })
    return (pd.DataFrame(filas) if filas else pd.DataFrame()), all_curves


# ──────────────────────────────────────────────────────────────────────────────
# 12. FIGURAS
# ──────────────────────────────────────────────────────────────────────────────

def _aplicar_estilo_ax(ax, titulo, xlabel, ylabel):
    ax.set_title(titulo, fontsize=11, fontweight="bold", color=TEXT_PLOT, pad=12)
    if xlabel: ax.set_xlabel(xlabel, color=TEXT_PLOT, fontsize=10)
    if ylabel: ax.set_ylabel(ylabel, color=TEXT_PLOT, fontsize=10)
    ax.tick_params(colors=TEXT_PLOT, labelsize=9)
    ax.grid(True, linestyle="--", alpha=0.15, color=GRID_PLOT)
    for sp in ax.spines.values(): sp.set_edgecolor(GRID_PLOT)


def _fig_prediccion(y_test, y_pred, target_col, days, parroquia=""):
    df_p = pd.DataFrame({"Real": y_test.values, "Predicho": y_pred}, index=y_test.index)
    df_p = df_p[df_p.index >= df_p.index.max() - pd.Timedelta(days=days)]
    titulo = f"Real vs Predicho — {target_col}  ·  Últimos {days} días (Test)"
    if parroquia: titulo = f"[{parroquia}]  {titulo}"
    fig, ax = plt.subplots(figsize=(13, 4.5), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    ax.plot(df_p.index, df_p["Real"],     label="Real",     color=COLOR_REAL, lw=1.8, alpha=0.95)
    ax.plot(df_p.index, df_p["Predicho"], label="Predicho", color=COLOR_PRED, lw=1.5,
            linestyle="--", alpha=0.90)
    ax.fill_between(df_p.index, df_p["Real"], df_p["Predicho"], alpha=0.07, color=COLOR_PRED)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    ax.xaxis.set_major_locator(mdates.DayLocator())
    plt.xticks(rotation=28, ha="right", color=TEXT_PLOT, fontsize=9)
    plt.yticks(color=TEXT_PLOT, fontsize=9)
    for sp in ax.spines.values(): sp.set_edgecolor(GRID_PLOT)
    ax.set_title(titulo, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=12)
    ax.set_ylabel(target_col, color=TEXT_PLOT); ax.set_xlabel("Fecha", color=TEXT_PLOT)
    ax.grid(True, linestyle="--", alpha=0.15, color=GRID_PLOT)
    ax.legend(framealpha=0.15, labelcolor=TEXT_PLOT, facecolor=SURF_PLOT, edgecolor=GRID_PLOT, fontsize=10)
    plt.tight_layout(); return fig


def _fig_feature_importance(fi_df, target_col, parroquia=""):
    n = len(fi_df)
    fig, ax = plt.subplots(figsize=(9, max(4, n * 0.52)), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    colores = [COLOR_POS if v >= 0 else COLOR_NEG for v in fi_df["Importance"]]
    ax.barh(fi_df["Feature"][::-1], fi_df["Importance"][::-1],
            xerr=fi_df["Std"][::-1], color=colores[::-1],
            align="center", alpha=0.85, ecolor="#948E9C", capsize=3, height=0.65)
    ax.axvline(0, color="#494551", linewidth=0.9, linestyle="--")
    for sp in ax.spines.values(): sp.set_edgecolor(GRID_PLOT)
    titulo_fi = f"Feature Importance  ·  {target_col}  (Permutation Δ R²)"
    if parroquia: titulo_fi = f"[{parroquia}]  {titulo_fi}"
    plt.yticks(color=TEXT_PLOT, fontsize=9); plt.xticks(color=TEXT_PLOT, fontsize=9)
    ax.set_title(titulo_fi, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=12)
    ax.set_xlabel("Importancia media (Δ R²)", color=TEXT_PLOT)
    ax.grid(True, axis="x", linestyle="--", alpha=0.14, color=GRID_PLOT)
    plt.tight_layout(); return fig


def _fig_heatmap(df_full, target_col, parroquia=""):
    num_df = df_full.select_dtypes(include=[np.number])
    if num_df.shape[1] < 2: return None
    corr = num_df.corr(method="pearson"); n = len(corr)
    fig, ax = plt.subplots(figsize=(max(8, n * 0.60), max(7, n * 0.55)), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    mask = np.zeros_like(corr, dtype=bool)
    mask[np.triu_indices_from(mask, k=1)] = True
    cmap = sns.diverging_palette(260, 40, as_cmap=True)
    sns.heatmap(corr, mask=mask, cmap=cmap, vmin=-1, vmax=1, center=0,
                annot=True, fmt=".2f", annot_kws={"size": 7.5, "color": TEXT_PLOT},
                linewidths=0.4, linecolor=GRID_PLOT, square=True, ax=ax, cbar_kws={"shrink": 0.7})
    if target_col in corr.columns:
        idx = list(corr.columns).index(target_col)
        ax.add_patch(plt.Rectangle((idx, 0), 1, n, fill=False, edgecolor=COLOR_PRED, lw=2.5, clip_on=False))
        ax.add_patch(plt.Rectangle((0, idx), n, 1, fill=False, edgecolor=COLOR_PRED, lw=2.5, clip_on=False))
    titulo_hm = f"Correlación de Pearson — {target_col}"
    if parroquia: titulo_hm = f"[{parroquia}]  {titulo_hm}"
    ax.set_title(titulo_hm, fontsize=12, fontweight="bold", color=TEXT_PLOT, pad=14)
    ax.tick_params(colors=TEXT_PLOT, labelsize=8)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    plt.setp(ax.get_yticklabels(), rotation=0)
    ax.collections[0].colorbar.ax.tick_params(colors=TEXT_PLOT, labelsize=8)
    plt.tight_layout(); return fig


def _fig_curva_aprendizaje(curvas, target_col, algoritmo, parroquia="", k_splits=5):
    fig, ax = plt.subplots(figsize=(13, 5), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    if not curvas or not curvas.get("train"):
        ax.text(0.5, 0.5, "No hay datos de curva.", ha="center", va="center",
                color=TEXT_PLOT, fontsize=11, transform=ax.transAxes)
        _aplicar_estilo_ax(ax, f"Curva de Aprendizaje — {target_col}", "", "")
        plt.tight_layout(); return fig
    train_curves = curvas["train"]; val_curves = curvas.get("val", [])
    metric_name  = curvas.get("metric", "Score"); is_r2 = metric_name == "R²"
    all_raw = train_curves + (val_curves if val_curves else [])
    lengths = [len(c) for c in all_raw if c]
    if not lengths or min(lengths) < 2:
        ax.text(0.5, 0.5, "Historial demasiado corto.", ha="center", va="center",
                color=TEXT_PLOT, transform=ax.transAxes)
        _aplicar_estilo_ax(ax, f"Curva de Aprendizaje — {target_col}", "", "")
        plt.tight_layout(); return fig
    min_len = min(lengths); x = np.arange(min_len)
    tr_arr = np.array([c[:min_len] for c in train_curves])
    tr_mean = tr_arr.mean(axis=0); tr_std = tr_arr.std(axis=0)
    for c in tr_arr: ax.plot(x, c, color=COLOR_REAL, alpha=0.09, lw=0.75, zorder=2)
    ax.plot(x, tr_mean, color=COLOR_REAL, lw=2.2, label=f"Train — {metric_name}  (μ TSS)", zorder=5)
    ax.fill_between(x, tr_mean-tr_std, tr_mean+tr_std, alpha=0.13, color=COLOR_REAL, zorder=3)
    if val_curves:
        va_arr = np.array([c[:min_len] for c in val_curves])
        va_mean = va_arr.mean(axis=0); va_std = va_arr.std(axis=0)
        for c in va_arr: ax.plot(x, c, color=COLOR_PRED, alpha=0.09, lw=0.75, zorder=2)
        ax.plot(x, va_mean, color=COLOR_PRED, lw=2.2, linestyle="--",
                label=f"Validación — {metric_name}  (μ TSS)", zorder=5)
        ax.fill_between(x, va_mean-va_std, va_mean+va_std, alpha=0.13, color=COLOR_PRED, zorder=3)
        best_iter = int(np.argmax(va_mean) if is_r2 else np.argmin(va_mean))
        best_val  = va_mean[best_iter]
        ax.axvline(best_iter, color=COLOR_SEC, lw=1.6, linestyle=":", alpha=0.9,
                   label=f"Mejor iter: {best_iter}  ({best_val:.4f})", zorder=6)
        ax.scatter([best_iter], [best_val], color=COLOR_SEC, s=65, zorder=8)
    ax.axhline(tr_mean[-1], color=COLOR_REAL, lw=0.7, linestyle=":", alpha=0.35, zorder=1)
    algo_label = algoritmo.split("(")[0].strip()
    titulo = f"Curva de Aprendizaje — {target_col}  ·  {algo_label}  ·  TSS (K={k_splits})"
    if parroquia: titulo = f"[{parroquia}]  {titulo}"
    ylabel = "R²  (↑ mejor)" if is_r2 else f"{metric_name}  (↓ mejor)"
    _aplicar_estilo_ax(ax, titulo, "Iteración / Boosting Round", ylabel)
    ax.legend(framealpha=0.20, labelcolor=TEXT_PLOT, facecolor=SURF_PLOT,
              edgecolor=GRID_PLOT, fontsize=9, loc="best")
    plt.tight_layout(); return fig


# ── [VIS-5]  NUEVAS FIGURAS ───────────────────────────────────────────────────

def _fig_mae_comparativo(kf_resumen: pd.DataFrame, target_col: str,
                          parroquia: str = "") -> plt.Figure | None:
    """
    [VIS-5a] Barras agrupadas MAE / RMSE por contaminante (resultados TSS).
    El target seleccionado se resalta con borde ámbar.
    """
    if kf_resumen.empty:
        return None

    contams = kf_resumen["Contaminante"].tolist()
    mae_m   = kf_resumen["MAE_mean"].values
    mae_s   = kf_resumen["MAE_std"].values
    rmse_m  = kf_resumen["RMSE_mean"].values
    rmse_s  = kf_resumen["RMSE_std"].values

    x = np.arange(len(contams))
    w = 0.38

    fig, ax = plt.subplots(figsize=(11, 5), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)

    b1 = ax.bar(x - w/2, mae_m, w, yerr=mae_s, label="MAE",
                color=COLOR_REAL, alpha=0.85, ecolor="#948E9C", capsize=4)
    b2 = ax.bar(x + w/2, rmse_m, w, yerr=rmse_s, label="RMSE",
                color=COLOR_PRED, alpha=0.85, ecolor="#948E9C", capsize=4)

    if target_col in contams:
        idx = contams.index(target_col)
        for bar in (b1[idx], b2[idx]):
            bar.set_edgecolor(COLOR_SEC)
            bar.set_linewidth(2.5)

    ax.set_xticks(x)
    ax.set_xticklabels(contams, color=TEXT_PLOT, fontsize=10)
    ax.tick_params(colors=TEXT_PLOT, labelsize=9)
    for sp in ax.spines.values(): sp.set_edgecolor(GRID_PLOT)

    titulo = "Comparativo MAE / RMSE por Contaminante  ·  TimeSeriesSplit (K=5)"
    if parroquia: titulo = f"[{parroquia}]  {titulo}"
    _aplicar_estilo_ax(ax, titulo, "Contaminante", "Error promedio (µg/m³ / ppm)")
    ax.legend(framealpha=0.18, labelcolor=TEXT_PLOT, facecolor=SURF_PLOT,
              edgecolor=GRID_PLOT, fontsize=10)
    plt.tight_layout()
    return fig


def _fig_r2_comparativo(kf_resumen: pd.DataFrame, target_col: str,
                         parroquia: str = "") -> plt.Figure | None:
    """
    [VIS-5b] Barras horizontales de R² con colores semánticos MD3:
    verde ≥0.7, ámbar ≥0.5, rojo <0.5. Líneas de referencia y anotaciones.
    """
    if kf_resumen.empty:
        return None

    df  = kf_resumen.sort_values("R2_mean", ascending=True).reset_index(drop=True)
    contams = df["Contaminante"].tolist()
    r2_m    = df["R2_mean"].values
    r2_s    = df["R2_std"].values

    colores = [
        "#10B981" if v >= 0.70 else ("#F59E0B" if v >= 0.50 else "#F87171")
        for v in r2_m
    ]

    fig, ax = plt.subplots(figsize=(10, max(3.5, len(contams) * 0.70)), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)

    bars = ax.barh(contams, r2_m, xerr=r2_s, color=colores,
                   alpha=0.88, ecolor="#948E9C", capsize=4, height=0.58)

    if target_col in contams:
        idx = contams.index(target_col)
        bars[idx].set_edgecolor(COLOR_SEC)
        bars[idx].set_linewidth(2.5)

    for xv, lbl, col, ls in [
        (0.5, "Mínimo (0.5)",   "#F59E0B", ":"),
        (0.7, "Bueno (0.7)",    "#10B981", "--"),
        (0.9, "Excelente (0.9)", COLOR_REAL, "-.")
    ]:
        ax.axvline(xv, color=col, lw=1.1, linestyle=ls, alpha=0.60, label=lbl)

    for i, (v, s) in enumerate(zip(r2_m, r2_s)):
        ax.text(min(v + 0.01, 1.0), i, f"{v:.3f}±{s:.3f}",
                va="center", color=TEXT_PLOT, fontsize=8.5)

    ax.set_xlim(0, 1.05)
    ax.tick_params(colors=TEXT_PLOT, labelsize=9)
    for sp in ax.spines.values(): sp.set_edgecolor(GRID_PLOT)

    titulo = "R² por Contaminante  ·  TimeSeriesSplit (K=5)  ·  Colores semánticos"
    if parroquia: titulo = f"[{parroquia}]  {titulo}"
    _aplicar_estilo_ax(ax, titulo, "R²   (0 = sin predicción   |   1 = perfecto)", "")
    ax.legend(framealpha=0.18, labelcolor=TEXT_PLOT, facecolor=SURF_PLOT,
              edgecolor=GRID_PLOT, fontsize=8.5, loc="lower right")
    plt.tight_layout()
    return fig


def _fig_pie_importancia(fi_df: pd.DataFrame, target_col: str,
                          parroquia: str = "") -> plt.Figure | None:
    """
    [VIS-5c] Donut chart — distribución de las top-10 features con
    paleta Material Design 3 oscuro. El centro muestra el target.
    """
    if fi_df.empty:
        return None

    TOP_N   = 10
    fi_pos  = fi_df[fi_df["Importance"] > 0].reset_index(drop=True)
    if fi_pos.empty:
        return None

    top_n  = fi_pos.head(TOP_N)
    otros  = fi_pos.iloc[TOP_N:]
    labels = top_n["Feature"].tolist()
    values = top_n["Importance"].tolist()

    if len(otros) > 0 and otros["Importance"].sum() > 0:
        labels.append(f"Otros ({len(otros)})")
        values.append(float(otros["Importance"].sum()))

    # Paleta MD3
    palette = [
        "#CFBCFF", "#CDC0E9", "#E7C365", "#10B981", "#F97316",
        "#3B82F6", "#A78BFA", "#F472B6", "#34D399", "#FB923C", "#818CF8",
    ]
    colors_pie = [palette[i % len(palette)] for i in range(len(labels))]

    fig, ax = plt.subplots(figsize=(9, 7), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)

    wedges, _, autotexts = ax.pie(
        values, labels=None, colors=colors_pie, autopct="%1.1f%%",
        startangle=140, pctdistance=0.80,
        wedgeprops={"edgecolor": BG_PLOT, "linewidth": 2.5, "antialiased": True},
    )
    for at in autotexts:
        at.set_fontsize(8); at.set_color(BG_PLOT); at.set_fontweight("bold")

    # Donut hole
    ax.add_patch(plt.Circle((0, 0), 0.55, fc=BG_PLOT))

    # Centro
    ax.text(0, 0.08, target_col, ha="center", va="center",
            color=TEXT_PLOT, fontsize=14, fontweight="bold")
    ax.text(0, -0.12, "top features", ha="center", va="center",
            color=TEXT_PLOT, fontsize=8.5, alpha=0.65)

    ax.legend(wedges, labels, loc="lower center", bbox_to_anchor=(0.5, -0.16),
              ncol=3, framealpha=0.12, labelcolor=TEXT_PLOT, facecolor=SURF_PLOT,
              edgecolor=GRID_PLOT, fontsize=8.5)

    titulo = f"Distribución de Importancia — {target_col}  (top {TOP_N})"
    if parroquia: titulo = f"[{parroquia}]  {titulo}"
    ax.set_title(titulo, fontsize=11, fontweight="bold", color=TEXT_PLOT, pad=14)
    plt.tight_layout(rect=[0, 0.10, 1, 1])
    return fig


# ──────────────────────────────────────────────────────────────────────────────
# 13. PIPELINE PRINCIPAL  — 9 outputs
# ──────────────────────────────────────────────────────────────────────────────

def entrenar(
    csv_upload, target_col, nombre_modelo, algoritmo,
    plot_days, train_ratio, excluir_pandemia, k_splits,
    max_depth, learning_rate, subsample, colsample_bytree,
    reg_lambda, reg_alpha, gamma, min_child_weight,
    usar_feature_engineering,
):
    logs: list[str] = []
    def info(m): log.info(m);    logs.append(f"✅ {m}")
    def warn(m): log.warning(m); logs.append(f"⚠️  {m}")
    def err(m):  log.error(m);   logs.append(f"❌ {m}")
    def _estado(): return "**Registro:**  " + "  ·  ".join(logs[-30:])

    # 9 outputs: metricas_md, fig_mae, fig_pred, fig_lc, fig_r2, fig_fi, fig_pie, fig_hm, estado
    VACIO = (None, None, None, None, None, None, None)   # 7 Nones para figuras 2-8

    try:
        if csv_upload is None:
            err("No se subió ningún archivo CSV.")
            return "### ❌ Sube un archivo CSV antes de entrenar.", *VACIO, _estado()

        ruta_csv      = csv_upload if isinstance(csv_upload, str) else csv_upload.name
        target_col    = target_col.strip()
        nombre_modelo = nombre_modelo.strip() or "surrogate_model"
        parroquia     = Path(ruta_csv).stem.replace("_", " ").title()
        info(f"Archivo: {Path(ruta_csv).name}")

        df = _cargar_csv(ruta_csv)
        info(f"CSV cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")

        ts_col = _detectar_timestamp(df)
        if ts_col is None:
            return "### ❌ No se encontró columna Timestamp/Date/Fecha.", *VACIO, _estado()
        info(f"Timestamp: '{ts_col}'")

        if target_col not in df.columns:
            return (f"### ❌ Target **`{target_col}`** no encontrado.\n\n"
                    f"**Columnas:** `{', '.join(df.columns)}`", *VACIO, _estado())

        df_prep = _preprocesar(df, target_col, ts_col, excluir_pandemia)
        if excluir_pandemia: info(f"Pandemia excluida: {ANOS_PANDEMIA}.")

        y = df_prep[target_col]
        X = df_prep.drop(columns=[target_col])

        if usar_feature_engineering:
            info("Aplicando Feature Engineering avanzado…")
            df_full = _agregar_features_avanzadas(pd.concat([X, y], axis=1), target_col)
            y = df_full[target_col]; X = df_full.drop(columns=[target_col])
            X_tr_sel, _, y_tr_sel, _ = _dividir_cronologico(X, y, train_ratio)
            info(f"Seleccionando top-{MAX_FEATURES_SEL} features entre {X_tr_sel.shape[1]}…")
            selected  = _seleccionar_top_features(X_tr_sel, y_tr_sel, MAX_FEATURES_SEL, log_fn=info)
            X         = X[selected]; feat_cols = list(X.columns)
            info(f"Features seleccionadas: {len(feat_cols)}")
        else:
            num_cols  = X.select_dtypes(include=[np.number]).columns.tolist()
            feat_cols = [c for c in num_cols if c != target_col]
            X         = X[feat_cols]; info(f"Usando {len(feat_cols)} features originales.")

        n_nulos = int(X.isna().sum().sum())
        if n_nulos: warn(f"{n_nulos:,} NaN en features — eliminando filas.")
        mask = X.notna().all(axis=1); X = X.loc[mask]; y = y.loc[mask]
        df_clean = pd.concat([X, y], axis=1); info(f"Dataset limpio: {df_clean.shape[0]:,} filas")

        xgb_params = {
            "max_depth": int(max_depth), "learning_rate": float(learning_rate),
            "subsample": float(subsample), "colsample_bytree": float(colsample_bytree),
            "reg_lambda": float(reg_lambda), "reg_alpha": float(reg_alpha),
            "gamma": float(gamma), "min_child_weight": float(min_child_weight),
        }

        info(f"TimeSeriesSplit (K={k_splits}) · Algoritmo: {algoritmo}")
        kfold_logs = []
        def kf_log(m): log.info(m); kfold_logs.append(m)
        kf_resumen, kf_curvas = _entrenar_kfold(
            df_clean, algoritmo, n_splits=k_splits, log_fn=kf_log, xgb_params=xgb_params
        )
        for l in kfold_logs: logs.append(l)
        if kf_resumen.empty: warn("TSS K-Fold no produjo resultados.")

        fig_lc = _fig_curva_aprendizaje(
            kf_curvas.get(target_col, {}), target_col, algoritmo, parroquia, k_splits
        )

        X_tr, X_te, y_tr, y_te = _dividir_cronologico(X, y, train_ratio)
        sub_corte   = int(len(X_tr) * 0.80)
        X_sub_tr    = X_tr.iloc[:sub_corte]; X_val_sub = X_tr.iloc[sub_corte:]
        y_sub_tr    = y_tr.iloc[:sub_corte]; y_val_sub = y_tr.iloc[sub_corte:]

        scaler_final = StandardScaler()
        X_sub_tr_sc  = scaler_final.fit_transform(X_sub_tr)
        X_val_sub_sc = scaler_final.transform(X_val_sub)
        X_te_sc      = scaler_final.transform(X_te)
        X_tr_sc      = np.vstack([X_sub_tr_sc, X_val_sub_sc])
        y_tr_np      = y_tr.values

        modelo_final = _construir_modelo(algoritmo, xgb_params)
        if "XGBoost" in algoritmo:
            modelo_final.fit(X_sub_tr_sc, y_sub_tr.values,
                             eval_set=[(X_val_sub_sc, y_val_sub.values)], verbose=False)
        elif "LightGBM" in algoritmo:
            import lightgbm as lgb
            modelo_final.fit(X_sub_tr_sc, y_sub_tr.values,
                             eval_set=[(X_val_sub_sc, y_val_sub.values)],
                             callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)])
        else:
            modelo_final.fit(X_sub_tr_sc, y_sub_tr.values)

        y_pred_tr = modelo_final.predict(X_tr_sc)
        y_pred_te = modelo_final.predict(X_te_sc)

        def _m(yt, yp):
            return dict(MAE=mean_absolute_error(yt, yp),
                        RMSE=float(np.sqrt(mean_squared_error(yt, yp))),
                        R2=r2_score(yt, yp))
        m_tr = _m(y_tr_np, y_pred_tr); m_te = _m(y_te.values, y_pred_te)
        gap = m_tr["R2"] - m_te["R2"]
        if gap > 0.15: warn(f"Posible overfitting: ΔR² = {gap:.3f}")

        info("Generando tabla de lags climatológicos…")
        lag_lookup       = _crear_lag_lookup(X_tr, feat_cols)
        ultimo_timestamp = X_tr.index.max() if hasattr(X_tr.index, "max") else None
        info(f"lag_lookup: {len(lag_lookup)} columnas · último dato: {ultimo_timestamp}")

        hw_label       = f"{'🟢 GPU' if GPU_DISPONIBLE else '🔵 CPU'} — {GPU_MSG}"
        pandemia_label = "🚫 2020-2021 excluidos" if excluir_pandemia else "⚠️ Pandemia incluida"

        if not kf_resumen.empty:
            filas_kf = []
            for _, row in kf_resumen.iterrows():
                ico = "🟢" if row["R2_mean"] >= 0.7 else ("🟡" if row["R2_mean"] >= 0.5 else "🔴")
                filas_kf.append(
                    f"| **{row['Contaminante']}** | `{int(row['Features_X'])}` "
                    f"| `{row['MAE_mean']:.3f} ± {row['MAE_std']:.3f}` "
                    f"| `{row['RMSE_mean']:.3f} ± {row['RMSE_std']:.3f}` "
                    f"| {ico} `{row['R2_mean']:.3f} ± {row['R2_std']:.3f}` |"
                )
            tabla_kf = "\n".join(filas_kf)
        else:
            tabla_kf = "| — | — | — | — | — |"

        metricas_md = f"""
## 📊 Surrogate Model v10.5 — *{parroquia}*

### TimeSeriesSplit (K={k_splits}) — 6 Contaminantes

| Contaminante | Features X | MAE (μ ± σ) | RMSE (μ ± σ) | R² (μ ± σ) |
|:------------:|:----------:|:-----------:|:------------:|:----------:|
{tabla_kf}

---

### Modelo Final — `{target_col}`  (Split {int(train_ratio*100)}/{int((1-train_ratio)*100)})

| Métrica | 🟦 Train | 🟧 Test ciego |
|---------|:--------:|:-------------:|
| **MAE** | `{m_tr['MAE']:.4f}` | `{m_te['MAE']:.4f}` |
| **RMSE** | `{m_tr['RMSE']:.4f}` | `{m_te['RMSE']:.4f}` |
| **R²** | `{m_tr['R2']:.4f}` | `{m_te['R2']:.4f}` |
| **Precisión (R² %)** | `{m_tr['R2']*100:.2f}%` | `{m_te['R2']*100:.2f}%` |

{"⚠️ **Posible overfitting** — ΔR² = `" + f"{gap:.3f}`" if gap > 0.15 else "✅ Sin señales de overfitting."}

| Parámetro | Valor |
|-----------|-------|
| Hardware | {hw_label} |
| Features | `{len(feat_cols)}` cols |
| Lag lookup | `{len(lag_lookup)}` cols |
| Último dato | `{ultimo_timestamp}` |
| Pandemia | {pandemia_label} |
"""

        perm  = permutation_importance(modelo_final, X_te_sc, y_te.values,
                                       n_repeats=8, random_state=42, scoring="r2")
        fi_df = pd.DataFrame({
            "Feature": feat_cols,
            "Importance": perm.importances_mean,
            "Std": perm.importances_std
        }).sort_values("Importance", ascending=False).reset_index(drop=True)
        info("Permutation Importance lista.")

        # ── [VIS-5] Nuevas figuras ────────────────────────────────────────────
        fig_mae = _fig_mae_comparativo(kf_resumen, target_col, parroquia)
        fig_r2  = _fig_r2_comparativo(kf_resumen, target_col, parroquia)
        fig_pie = _fig_pie_importancia(fi_df, target_col, parroquia)

        tag = f"{nombre_modelo}_{parroquia.replace(' ', '_')}"
        pkl_path = OUTPUT_DIR / f"{tag}.pkl"
        with open(pkl_path, "wb") as fh:
            pickle.dump({
                "modelo": modelo_final, "scaler": scaler_final,
                "features": feat_cols, "target": target_col,
                "parroquia": parroquia, "kfold_resumen": kf_resumen,
                "kf_curvas": kf_curvas, "lag_lookup": lag_lookup,
                "ultimo_timestamp": ultimo_timestamp,
            }, fh)
        info(f"Modelo guardado: {pkl_path}")
        fi_df.to_csv(OUTPUT_DIR / f"{tag}_feature_importance.csv", index=False)
        if not kf_resumen.empty:
            kf_resumen.to_csv(OUTPUT_DIR / f"{tag}_kfold_resumen.csv", index=False)

        SESION.update({
            "modelo": modelo_final, "scaler": scaler_final,
            "feat_cols": feat_cols, "target": target_col,
            "parroquia": parroquia,
            "feat_stats": {col: {"min": float(X[col].min()), "max": float(X[col].max()),
                                  "mean": float(X[col].mean())} for col in feat_cols},
            "lag_lookup": lag_lookup, "ultimo_timestamp": ultimo_timestamp,
        })
        info("Sesión actualizada → pestaña Predicción IQCA lista.")

        y_te_series = pd.Series(y_te.values, index=X_te.index, name=target_col)
        fig_pred = _fig_prediccion(y_te_series, y_pred_te, target_col, plot_days, parroquia)
        fig_fi   = _fig_feature_importance(fi_df, target_col, parroquia)
        fig_hm   = _fig_heatmap(pd.concat([X, y], axis=1), target_col, parroquia)
        info("Todas las figuras generadas.")

        # 9 valores de retorno: metricas, fig_mae, fig_pred, fig_lc, fig_r2, fig_fi, fig_pie, fig_hm, estado
        return metricas_md, fig_mae, fig_pred, fig_lc, fig_r2, fig_fi, fig_pie, fig_hm, _estado()

    except ImportError as e:
        err(str(e))
        pkg = str(e).split("'")[-2] if "'" in str(e) else str(e).split()[-1]
        return (f"### ❌ Librería faltante\n```\n{e}\n```\n`pip install {pkg}`", *VACIO, _estado())
    except Exception as e:
        err(str(e))
        return (f"### ❌ Error\n```\n{traceback.format_exc()}\n```", *VACIO, _estado())


# ──────────────────────────────────────────────────────────────────────────────
# 14. [VIS-3]  RESUMEN DINÁMICO DE CSV
# ──────────────────────────────────────────────────────────────────────────────

def _resumen_csv_html(archivo) -> tuple[gr.Dropdown, str]:
    """
    Combina actualizar_targets() + HTML de resumen.
    Devuelve (dropdown_update, html_resumen).
    """
    dropdown_update = actualizar_targets(archivo)
    if archivo is None:
        return dropdown_update, ""
    try:
        ruta    = archivo if isinstance(archivo, str) else archivo.name
        df_full = pd.read_csv(ruta, comment="#", low_memory=False)
        ts      = _detectar_timestamp(df_full)
        contams = [c for c in CONTAMINANTES_Y if c in df_full.columns]

        badges = "".join([
            f'<span style="background:#4D4465;color:#CDC0E9;padding:2px 10px;'
            f'border-radius:20px;font-size:11px;font-weight:600;margin:2px;">{c}</span>'
            for c in contams
        ])
        ts_badge = (
            f'<code style="background:#2B292F;color:#CFBCFF;padding:1px 7px;'
            f'border-radius:4px;font-size:11px;font-family:\'JetBrains Mono\',monospace;">{ts}</code>'
            if ts else '<span style="color:#F59E0B;font-size:11px;">⚠️ No detectado</span>'
        )
        ok_color = "#10B981" if contams else "#F59E0B"
        ok_icon  = "✅" if contams else "⚠️"

        html = f"""
        <div style="background:#1D1B20;border:1px solid #494551;border-radius:10px;
                    padding:12px 14px;margin-top:8px;">
            <div style="display:flex;align-items:center;gap:8px;margin-bottom:10px;">
                <span style="color:{ok_color};font-size:16px;">{ok_icon}</span>
                <span style="color:#E6E0E9;font-weight:600;font-size:13px;
                             font-family:'Inter',sans-serif;">{Path(ruta).name}</span>
            </div>
            <div style="display:grid;grid-template-columns:1fr 1fr;gap:6px;margin-bottom:10px;">
                <div style="color:#CBC4D2;font-size:12px;">
                    📊 <b style="color:#E6E0E9;">{df_full.shape[0]:,}</b> filas ×
                    <b style="color:#E6E0E9;">{df_full.shape[1]}</b> cols
                </div>
                <div style="color:#CBC4D2;font-size:12px;">🕐 {ts_badge}</div>
            </div>
            <div style="color:#948E9C;font-size:10px;font-weight:600;
                        text-transform:uppercase;letter-spacing:0.07em;margin-bottom:6px;">
                Contaminantes REMMAQ detectados
            </div>
            <div style="display:flex;flex-wrap:wrap;gap:4px;">
                {badges if badges else
                 '<span style="color:#948E9C;font-size:11px;">Ninguno detectado — verifica nombres</span>'}
            </div>
        </div>
        """
        return dropdown_update, html
    except Exception as e:
        return dropdown_update, (
            f'<div style="color:#FFB4AB;font-size:12px;padding:8px;'
            f'background:#1D1B20;border-radius:8px;">❌ Error: {e}</div>'
        )


# ──────────────────────────────────────────────────────────────────────────────
# 15. [VIS-4]  BADGE DE ESTADO
# ──────────────────────────────────────────────────────────────────────────────

def _badge_estado(metricas_md: str) -> str:
    """Devuelve HTML del badge según el resultado de entrenar()."""
    if metricas_md and "❌" not in metricas_md and "Error" not in metricas_md:
        return """
        <div style="display:inline-flex;align-items:center;gap:6px;padding:5px 14px;
                    border-radius:20px;font-size:0.75rem;font-weight:600;
                    background:rgba(16,185,129,0.12);color:#34D399;
                    border:1px solid #10B981;margin:8px 0;">
            ✅ Entrenamiento completado exitosamente
        </div>"""
    if metricas_md and ("❌" in metricas_md or "Error" in metricas_md):
        return """
        <div style="display:inline-flex;align-items:center;gap:6px;padding:5px 14px;
                    border-radius:20px;font-size:0.75rem;font-weight:600;
                    background:rgba(255,180,171,0.12);color:#FFB4AB;
                    border:1px solid #93000A;margin:8px 0;">
            ❌ Error en entrenamiento — revisa los logs
        </div>"""
    return """
    <div style="display:inline-flex;align-items:center;gap:6px;padding:5px 14px;
                border-radius:20px;font-size:0.75rem;font-weight:600;
                background:rgba(148,142,156,0.10);color:#948E9C;
                border:1px solid #494551;margin:8px 0;">
        ⚪ Esperando ejecución...
    </div>"""


# ──────────────────────────────────────────────────────────────────────────────
# 16. [VIS-1]  CSS — MATERIAL DESIGN 3 DARK
# ──────────────────────────────────────────────────────────────────────────────

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500&display=swap');

/* ── Variables MD3 dark ──────────────────────────── */
:root {
    --bg:      #141218;
    --surface: #211F24;
    --surf-lo: #1D1B20;
    --surf-hi: #2B292F;
    --surf-ct: #36343A;
    --border:  #494551;
    --bdr-lo:  #36343A;
    --primary: #CFBCFF;
    --prim-c:  #4F378A;
    --prim-cx: #6750A4;
    --second:  #CDC0E9;
    --sec-c:   #4D4465;
    --amber:   #E7C365;
    --amber-c: #C9A74D;
    --text:    #E6E0E9;
    --txt-sec: #CBC4D2;
    --muted:   #948E9C;
    --ok:      #10B981;
    --warn:    #F59E0B;
    --err:     #FFB4AB;
    --err-c:   #93000A;
    --r:       12px;
    --r-sm:    8px;
    --r-xs:    4px;
}

/* ── Fuentes globales ───────────────────────────── */
body, .gradio-container, * {
    font-family: 'Inter', 'Segoe UI', system-ui, sans-serif !important;
}

/* ── Fondo principal ────────────────────────────── */
body, .gradio-container {
    background: var(--bg) !important;
    color: var(--text) !important;
    line-height: 1.5;
}

/* ── Grupos y cards ─────────────────────────────── */
.gr-group, .gr-box {
    background: var(--surface) !important;
    border: 1px solid var(--bdr-lo) !important;
    border-radius: var(--r) !important;
    padding: 16px !important;
}

/* ── Inputs ─────────────────────────────────────── */
input[type="text"],
input[type="number"],
input[type="search"],
textarea, select {
    background: var(--surf-lo) !important;
    color: var(--text) !important;
    border: 1px solid var(--bdr-lo) !important;
    border-radius: var(--r-sm) !important;
    font-family: 'Inter', sans-serif !important;
}
input[type="text"]:focus,
input[type="number"]:focus,
textarea:focus {
    border-color: var(--prim-cx) !important;
    box-shadow: 0 0 0 2px rgba(103, 80, 164, 0.25) !important;
    outline: none !important;
}

/* ── Labels ─────────────────────────────────────── */
label, .gr-label, .block > label > span {
    color: var(--txt-sec) !important;
    font-size: 0.73rem !important;
    font-weight: 600 !important;
    text-transform: uppercase !important;
    letter-spacing: 0.07em !important;
}

/* ── Botón primario ─────────────────────────────── */
button.primary, .gr-button-primary {
    background: linear-gradient(135deg, #4F46E5 0%, #7C3AED 100%) !important;
    color: #fff !important;
    border: none !important;
    border-radius: var(--r-sm) !important;
    font-weight: 700 !important;
    font-size: 0.95rem !important;
    padding: 13px 28px !important;
    box-shadow: 0 0 20px rgba(124,58,237,0.30),
                0 4px 14px rgba(79,70,229,0.22) !important;
    transition: all 0.18s ease !important;
    letter-spacing: 0.02em !important;
}
button.primary:hover, .gr-button-primary:hover {
    opacity: 0.88 !important;
    box-shadow: 0 0 32px rgba(124,58,237,0.50),
                0 6px 22px rgba(79,70,229,0.35) !important;
    transform: translateY(-1px) !important;
}
button.primary:active { transform: translateY(0) !important; }

/* ── Botón secundario ───────────────────────────── */
button.secondary {
    background: var(--surf-hi) !important;
    color: var(--text) !important;
    border: 1px solid var(--border) !important;
    border-radius: var(--r-sm) !important;
    font-weight: 500 !important;
}
button.secondary:hover {
    background: var(--surf-ct) !important;
    border-color: var(--primary) !important;
}

/* ── Markdown ────────────────────────────────────── */
.gr-markdown { color: var(--text) !important; }
.gr-markdown h1, .gr-markdown h2 {
    color: var(--primary) !important;
    font-weight: 700 !important;
    border-bottom: 1px solid var(--bdr-lo) !important;
    padding-bottom: 6px !important;
}
.gr-markdown h3 {
    color: var(--second) !important;
    font-weight: 600 !important;
    margin-bottom: 8px !important;
}

/* ── Tablas markdown ────────────────────────────── */
.gr-markdown table {
    border-collapse: collapse !important;
    width: 100% !important;
    font-size: 0.83rem !important;
    margin: 10px 0 !important;
    border-radius: var(--r-sm) !important;
    overflow: hidden !important;
}
.gr-markdown th {
    background: var(--surf-hi) !important;
    color: var(--primary) !important;
    padding: 9px 14px !important;
    border: 1px solid var(--bdr-lo) !important;
    font-size: 0.70rem !important;
    text-transform: uppercase !important;
    letter-spacing: 0.08em !important;
    font-weight: 700 !important;
}
.gr-markdown td {
    color: var(--text) !important;
    padding: 8px 14px !important;
    border: 1px solid var(--bdr-lo) !important;
    font-family: 'JetBrains Mono', monospace !important;
    font-size: 0.80rem !important;
    line-height: 1.4 !important;
}
.gr-markdown tr:nth-child(even) td {
    background: rgba(79, 55, 138, 0.08) !important;
}
.gr-markdown tr:hover td {
    background: rgba(207, 188, 255, 0.05) !important;
    transition: background 0.1s !important;
}
.gr-markdown code {
    background: var(--surf-hi) !important;
    color: var(--primary) !important;
    padding: 2px 6px !important;
    border-radius: 4px !important;
    font-family: 'JetBrains Mono', monospace !important;
    font-size: 0.80rem !important;
}
.gr-markdown blockquote {
    border-left: 3px solid var(--prim-c) !important;
    padding: 8px 14px !important;
    background: rgba(79, 55, 138, 0.10) !important;
    border-radius: 0 var(--r-xs) var(--r-xs) 0 !important;
    color: var(--txt-sec) !important;
    font-style: italic !important;
}

/* ── Checkboxes ─────────────────────────────────── */
input[type="checkbox"] {
    -webkit-appearance: checkbox !important;
    appearance: checkbox !important;
    accent-color: var(--prim-cx) !important;
    width: 15px !important;
    height: 15px !important;
    cursor: pointer !important;
}
.gr-checkbox > label, .gr-checkbox label,
[data-testid="checkbox"] label {
    display: flex !important;
    align-items: center !important;
    color: var(--text) !important;
    font-size: 0.87rem !important;
    font-weight: 500 !important;
    text-transform: none !important;
    cursor: pointer !important;
    gap: 6px !important;
}

/* ── Radio buttons ──────────────────────────────── */
.gr-radio label {
    color: var(--txt-sec) !important;
    font-size: 0.87rem !important;
    text-transform: none !important;
    font-weight: 500 !important;
}

/* ── File upload ─────────────────────────────────── */
.gr-file {
    background: rgba(79, 70, 229, 0.04) !important;
    border: 2px dashed var(--prim-c) !important;
    border-radius: var(--r) !important;
    transition: all 0.2s !important;
}
.gr-file:hover {
    background: rgba(103, 80, 164, 0.07) !important;
    border-color: var(--primary) !important;
}

/* ── Tabs ────────────────────────────────────────── */
.gr-tab-nav {
    border-bottom: 1px solid var(--bdr-lo) !important;
    background: var(--surf-lo) !important;
}
.gr-tab-nav button {
    color: var(--muted) !important;
    border-bottom: 2px solid transparent !important;
    font-size: 0.82rem !important;
    font-weight: 500 !important;
    padding: 10px 16px !important;
    transition: all 0.15s !important;
    background: transparent !important;
    border-radius: 0 !important;
}
.gr-tab-nav button.selected,
.gr-tab-nav button[aria-selected="true"] {
    color: var(--primary) !important;
    border-bottom: 2px solid var(--primary) !important;
    background: transparent !important;
    font-weight: 600 !important;
}
.gr-tab-nav button:hover {
    color: var(--text) !important;
    background: rgba(207, 188, 255, 0.05) !important;
}

/* ── Sliders ─────────────────────────────────────── */
input[type="range"] {
    accent-color: var(--prim-cx) !important;
    background: var(--bdr-lo) !important;
}

/* ── Accordion / Details ─────────────────────────── */
.gr-accordion summary,
details > summary {
    background: var(--surf-hi) !important;
    color: var(--text) !important;
    border: 1px solid var(--bdr-lo) !important;
    border-radius: var(--r-sm) !important;
    padding: 10px 14px !important;
    cursor: pointer !important;
    font-weight: 500 !important;
    font-size: 0.87rem !important;
}

/* ── Logs box ────────────────────────────────────── */
.logs-box {
    background: #0D0B11 !important;
    border: 1px solid var(--bdr-lo) !important;
    border-radius: var(--r-sm) !important;
    padding: 10px 14px !important;
    font-family: 'JetBrains Mono', monospace !important;
    font-size: 0.70rem !important;
    color: #A78BFA !important;
    max-height: 150px !important;
    overflow-y: auto !important;
    line-height: 1.75 !important;
}

/* ── GPU badge ───────────────────────────────────── */
.gpu-badge {
    display: inline-flex;
    align-items: center;
    gap: 5px;
    padding: 3px 10px;
    border-radius: 20px;
    font-size: 0.70rem;
    font-weight: 600;
    margin-top: 5px;
}
.gpu-on  { background: rgba(16,185,129,.12); color:#34D399; border:1px solid #10B981; }
.gpu-off { background: rgba(207,188,255,.10); color:var(--primary); border:1px solid var(--prim-c); }

/* ── FE badge ────────────────────────────────────── */
.fe-badge {
    background: rgba(207,188,255,0.06);
    border: 1px solid var(--prim-c);
    border-radius: var(--r-sm);
    padding: 8px 12px;
    font-size: 0.73rem;
    color: var(--second);
    margin: 5px 0;
}

/* ── IQCA box ────────────────────────────────────── */
.iqca-box {
    background: rgba(16,185,129,0.06);
    border: 1px solid #10B981;
    border-radius: 10px;
    padding: 10px 14px;
    font-size: 0.75rem;
    color: #6EE7B7;
    margin: 6px 0;
}

/* ── Card section headers ───────────────────────── */
.sect-header {
    display: flex;
    align-items: center;
    gap: 8px;
    padding-bottom: 10px;
    border-bottom: 1px solid var(--bdr-lo);
    margin-bottom: 12px;
}
.sect-header h3 {
    color: var(--text) !important;
    font-size: 0.95rem !important;
    font-weight: 600 !important;
    margin: 0 !important;
    text-transform: none !important;
    letter-spacing: normal !important;
}

/* ── Scrollbar ───────────────────────────────────── */
::-webkit-scrollbar { width: 5px; height: 5px; }
::-webkit-scrollbar-track { background: var(--bg); }
::-webkit-scrollbar-thumb { background: var(--border); border-radius: 3px; }
::-webkit-scrollbar-thumb:hover { background: var(--muted); }
"""


# ──────────────────────────────────────────────────────────────────────────────
# 17. [VIS-2]  CONSTRUCCIÓN DE LA UI
# ──────────────────────────────────────────────────────────────────────────────

MODELOS = [
    "HistGradientBoosting (CPU — sin GPU requerida)",
    "XGBoost  (auto GPU/CPU)",
    "LightGBM (auto GPU/CPU)",
]

gpu_badge_html = (
    f'<span class="gpu-badge gpu-on">🟢 GPU: {GPU_MSG.split(": ")[-1]}</span>'
    if GPU_DISPONIBLE else
    f'<span class="gpu-badge gpu-off">🔵 CPU — {GPU_MSG}</span>'
)


def construir_app() -> gr.Blocks:
    with gr.Blocks(
        theme=gr.themes.Base(
            primary_hue="violet",
            secondary_hue="purple",
            neutral_hue="zinc",
            font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif"],
        ),
        css=CSS,
        title="Quito Air ML — Surrogate Model v10.5",
    ) as app:

        # ── Hero ──────────────────────────────────────────────────────────────
        gr.HTML("""
        <div style="padding:20px 24px 14px;border-bottom:1px solid #36343A;
                    margin-bottom:0;display:flex;align-items:center;
                    justify-content:space-between;flex-wrap:wrap;gap:12px;">
            <div>
                <h1 style="font-size:1.65rem;font-weight:800;margin:0 0 4px;
                           background:linear-gradient(90deg,#CFBCFF,#E9DDFF 60%,#CDC0E9);
                           -webkit-background-clip:text;-webkit-text-fill-color:transparent;">
                    🌬️ Quito Air ML
                </h1>
                <p style="color:#948E9C;font-size:0.80rem;margin:0;">
                    Surrogate Model · Calidad del Aire · IQCA REMMAQ · v10.5
                </p>
            </div>
        </div>
        """)
        gr.HTML(f'<div style="padding:6px 24px 0;">{gpu_badge_html}</div>')
        gr.HTML("<div style='height:10px'/>")

        with gr.Row(equal_height=False, variant="compact"):

            # ══════════════════════════════════════════════════════════════════
            # COLUMNA IZQUIERDA — SIDEBAR
            # ══════════════════════════════════════════════════════════════════
            with gr.Column(scale=1, min_width=370):

                # ── Dataset Card ──────────────────────────────────────────────
                with gr.Group():
                    gr.HTML("""
                    <div class="sect-header">
                        <span style="color:#CFBCFF;font-size:1.1rem;">📂</span>
                        <h3>Dataset</h3>
                    </div>
                    """)
                    csv_upload = gr.File(
                        label="Arrastra o sube tu archivo CSV",
                        file_types=[".csv"], type="filepath",
                    )
                    csv_info = gr.HTML(value="", label="")

                gr.HTML("<div style='height:10px'/>")

                # ── Modelo Card ───────────────────────────────────────────────
                with gr.Group():
                    gr.HTML("""
                    <div class="sect-header">
                        <span style="color:#CFBCFF;font-size:1.1rem;">⚙️</span>
                        <h3>Configuración del Modelo</h3>
                    </div>
                    """)
                    with gr.Row():
                        target_input = gr.Dropdown(
                            label="Target (contaminante)",
                            choices=CONTAMINANTES_Y, value="PM25",
                            allow_custom_value=True, scale=3,
                        )
                        nombre_modelo_input = gr.Textbox(
                            label="Nombre modelo (.pkl)",
                            value="surrogate_calidad_aire", scale=3,
                        )
                    algoritmo_radio = gr.Radio(
                        label="Algoritmo",
                        choices=MODELOS, value=MODELOS[0],
                    )
                    with gr.Row():
                        excluir_pandemia_chk = gr.Checkbox(
                            label="🚫 Excluir pandemia 2020–2021",
                            value=True, scale=1,
                        )
                        usar_fe_ck = gr.Checkbox(
                            label="🧪 Feature Engineering",
                            value=False, scale=1,
                        )
                    gr.HTML(
                        '<div class="fe-badge">FE activa: rolling stats, diffs, '
                        'interacciones meteo, día semana → top-40 (RandomForest).</div>'
                    )
                    with gr.Row():
                        train_ratio_slider = gr.Slider(
                            label="Train / Test split",
                            minimum=0.6, maximum=0.95, step=0.05, value=0.80,
                            info="El (1-ratio)% final = Test Ciego intocable.", scale=3,
                        )
                        k_splits_slider = gr.Slider(
                            label="K — pliegues TSCV",
                            minimum=2, maximum=15, step=1, value=5, scale=2,
                        )
                        plot_days_slider = gr.Slider(
                            label="Días a graficar",
                            minimum=3, maximum=30, step=1, value=7, scale=2,
                        )

                gr.HTML("<div style='height:10px'/>")

                # ── Hiperparámetros Accordion ──────────────────────────────────
                with gr.Accordion("🧪 Hiperparámetros XGBoost / LightGBM / HistGB", open=False):
                    gr.HTML('<p style="color:#CBC4D2;font-size:0.75rem;margin:0 0 10px;">'
                            'learning_rate, max_depth y reg_lambda aplican también a HistGB.</p>')
                    with gr.Row():
                        max_depth_slider     = gr.Slider(label="max_depth",      minimum=2,    maximum=10,   step=1,    value=5)
                        learning_rate_slider = gr.Slider(label="learning_rate",  minimum=0.01, maximum=0.3,  step=0.01, value=0.05)
                    with gr.Row():
                        subsample_slider        = gr.Slider(label="subsample",        minimum=0.5, maximum=1.0, step=0.05, value=0.8)
                        colsample_bytree_slider = gr.Slider(label="colsample_bytree", minimum=0.5, maximum=1.0, step=0.05, value=0.8)
                    with gr.Row():
                        reg_lambda_slider       = gr.Slider(label="reg_lambda (L2)", minimum=0.0, maximum=10.0, step=0.5, value=2.0)
                        reg_alpha_slider        = gr.Slider(label="reg_alpha  (L1)", minimum=0.0, maximum=5.0,  step=0.1, value=0.5)
                    with gr.Row():
                        gamma_slider            = gr.Slider(label="gamma",            minimum=0.0, maximum=5.0,  step=0.1, value=0.0)
                        min_child_weight_slider = gr.Slider(label="min_child_weight", minimum=1,   maximum=20,   step=1,   value=1)

                gr.HTML("<div style='height:12px'/>")

                # ── Botón + Status + Logs ─────────────────────────────────────
                btn_train = gr.Button(
                    "🚀  Iniciar Entrenamiento", variant="primary", size="lg"
                )
                status_badge = gr.HTML(value=_badge_estado(""))
                gr.HTML(
                    '<p style="color:#948E9C;font-size:0.70rem;font-weight:600;'
                    'text-transform:uppercase;letter-spacing:0.07em;margin:10px 0 4px;">'
                    '🖥️ Registro de ejecución</p>'
                )
                estado_output = gr.Markdown(
                    value="_Esperando ejecución…_",
                    elem_classes=["logs-box"],
                )

            # ══════════════════════════════════════════════════════════════════
            # COLUMNA DERECHA — RESULTADOS (8 tabs, 9 outputs)
            # ══════════════════════════════════════════════════════════════════
            with gr.Column(scale=2, min_width=620):
                with gr.Tabs():

                    # Tab 1: Métricas + MAE comparativo
                    with gr.Tab("📊 Métricas"):
                        gr.HTML("""
                        <div style="display:grid;grid-template-columns:repeat(3,1fr);gap:10px;
                                    margin-bottom:14px;margin-top:4px;">
                            <div style="background:#1D1B20;border:1px solid #36343A;border-radius:10px;
                                        padding:12px 14px;">
                                <div style="font-size:0.65rem;text-transform:uppercase;letter-spacing:0.07em;
                                            color:#948E9C;font-weight:600;margin-bottom:6px;">Test R² Score</div>
                                <div style="font-size:1.5rem;font-weight:700;color:#CFBCFF;
                                            font-family:'JetBrains Mono',monospace;">—</div>
                            </div>
                            <div style="background:#1D1B20;border:1px solid #36343A;border-radius:10px;
                                        padding:12px 14px;">
                                <div style="font-size:0.65rem;text-transform:uppercase;letter-spacing:0.07em;
                                            color:#948E9C;font-weight:600;margin-bottom:6px;">Test MAE</div>
                                <div style="font-size:1.5rem;font-weight:700;color:#CDC0E9;
                                            font-family:'JetBrains Mono',monospace;">—</div>
                            </div>
                            <div style="background:#1D1B20;border:1px solid #36343A;border-radius:10px;
                                        padding:12px 14px;">
                                <div style="font-size:0.65rem;text-transform:uppercase;letter-spacing:0.07em;
                                            color:#948E9C;font-weight:600;margin-bottom:6px;">Estado</div>
                                <div style="font-size:0.85rem;font-weight:600;color:#10B981;">
                                    Listo para entrenar
                                </div>
                            </div>
                        </div>
                        """)
                        metricas_output = gr.Markdown(
                            value="*Las métricas aparecerán aquí tras entrenar.*"
                        )
                        gr.HTML('<hr style="border-color:#36343A;margin:12px 0;">')
                        gr.HTML('<p style="color:#CDC0E9;font-size:0.78rem;font-weight:600;margin:0 0 6px;">📊 MAE / RMSE por Contaminante</p>')
                        fig_mae_output = gr.Plot(label="Comparativo MAE / RMSE — TSS K=5")

                    # Tab 2: Real vs Predicho
                    with gr.Tab("📈 Real vs Predicho"):
                        fig_pred_output = gr.Plot(label="Serie temporal — Test Ciego (últimos N días)")

                    # Tab 3: Curva de Aprendizaje
                    with gr.Tab("📉 Curva Aprendizaje"):
                        gr.Markdown(
                            "Línea **continua** = Train · Línea **discontinua** = Validación TSS.\n"
                            "Banda = ±1σ entre los K pliegues. "
                            "La línea vertical marca la mejor iteración promedio."
                        )
                        fig_lc_output = gr.Plot(label="Loss / Error vs Iteración — TSS K=5")

                    # Tab 4: Rendimiento R² (NUEVA)
                    with gr.Tab("📊 Rendimiento"):
                        gr.Markdown(
                            "**R² por contaminante** con colores semánticos MD3.  \n"
                            "🟢 ≥0.70 Bueno  ·  🟡 ≥0.50 Aceptable  ·  🔴 <0.50 Mejorable  \n"
                            "La barra del target seleccionado aparece resaltada."
                        )
                        fig_r2_output = gr.Plot(label="Comparativo R² — TSS K=5")

                    # Tab 5: Feature Importance
                    with gr.Tab("🔍 Feature Importance"):
                        gr.Markdown(
                            "Importancia por permutación calculada sobre el **Test Ciego**.\n"
                            "Valores positivos = feature relevante. Negativos = ruido."
                        )
                        fig_fi_output = gr.Plot(label="Permutation Importance (Δ R²) — Test Ciego")

                    # Tab 6: Donut importancia (NUEVA)
                    with gr.Tab("🥧 Importancia"):
                        gr.Markdown(
                            "Distribución **relativa** de la importancia entre las top-10 features.  \n"
                            "Paleta Material Design 3 oscuro. El centro muestra el target activo."
                        )
                        fig_pie_output = gr.Plot(label="Distribución de Importancia — Donut Chart")

                    # Tab 7: Correlaciones
                    with gr.Tab("📊 Correlaciones"):
                        gr.Markdown(
                            "**Correlación de Pearson** entre todas las variables.  \n"
                            "La fila/columna del target aparece resaltada en ámbar."
                        )
                        fig_hm_output = gr.Plot(label="Mapa de calor — Pearson")

                    # Tab 8: Predicción IQCA
                    with gr.Tab("🌿 Predicción IQCA"):
                        gr.Markdown(
                            "### 🌿 Predicción con IQCA REMMAQ\n\n"
                            "Ingresa solo variables meteorológicas. Los **lags** se imputan "
                            "automáticamente desde la tabla climatológica (hora × mes).\n\n"
                            "> Entrena primero un modelo para activar esta sección."
                        )
                        gr.HTML(
                            '<div class="iqca-box">📌 <b>IQCA REMMAQ:</b> '
                            '0–50 Deseable · 51–100 Aceptable · 101–150 Precaución · '
                            '151–200 Alerta · 201–300 Alarma · 301–500 Emergencia</div>'
                        )
                        with gr.Group():
                            fecha_hora_input = gr.Textbox(
                                label="📅 Fecha y Hora  (YYYY-MM-DD HH:MM)",
                                placeholder="2024-03-15 14:00", value="",
                            )
                            with gr.Row():
                                temp_input  = gr.Slider(label="🌡️ Temperatura (°C)",       minimum=-10, maximum=40,  step=0.1, value=18.0, scale=1)
                                hum_input   = gr.Slider(label="💧 Humedad (%)",             minimum=0,   maximum=100, step=1,   value=70,   scale=1)
                            with gr.Row():
                                vvel_input  = gr.Slider(label="💨 Viento vel. (m/s)",       minimum=0,   maximum=30,  step=0.1, value=2.0,  scale=1)
                                vdir_input  = gr.Slider(label="🧭 Viento dirección (°)",    minimum=0,   maximum=360, step=1,   value=180,  scale=1)
                            precip_input = gr.Slider(
                                label="🌧️ Precipitación (mm)", minimum=0, maximum=200, step=0.1, value=0.0
                            )
                        btn_predecir   = gr.Button("🔮  Estimar Calidad del Aire + IQCA", variant="primary")
                        resultado_pred = gr.Markdown(value="_El resultado aparecerá aquí tras entrenar un modelo._")

        gr.HTML("""
        <div style="text-align:center;padding:16px 0 8px;
                    color:#494551;font-size:0.72rem;border-top:1px solid #36343A;margin-top:8px;">
            Quito Air ML · Surrogate Model v10.5 · Material Design 3 Dark ·
            TimeSeriesSplit · IQCA REMMAQ · Auto-GPU
        </div>
        """)

        # ── Eventos ───────────────────────────────────────────────────────────

        # [VIS-3] Resumen dinámico al subir CSV (actualiza también el dropdown)
        csv_upload.change(
            fn=_resumen_csv_html,
            inputs=[csv_upload],
            outputs=[target_input, csv_info],
        )

        # Entrenamiento → 9 outputs
        btn_train.click(
            fn=entrenar,
            inputs=[
                csv_upload, target_input, nombre_modelo_input,
                algoritmo_radio, plot_days_slider, train_ratio_slider,
                excluir_pandemia_chk, k_splits_slider,
                max_depth_slider, learning_rate_slider, subsample_slider,
                colsample_bytree_slider, reg_lambda_slider, reg_alpha_slider,
                gamma_slider, min_child_weight_slider, usar_fe_ck,
            ],
            outputs=[
                metricas_output,   # 1
                fig_mae_output,    # 2
                fig_pred_output,   # 3
                fig_lc_output,     # 4
                fig_r2_output,     # 5
                fig_fi_output,     # 6
                fig_pie_output,    # 7
                fig_hm_output,     # 8
                estado_output,     # 9
            ],
        ).then(
            fn=_badge_estado,
            inputs=[metricas_output],
            outputs=[status_badge],
        )

        # [VIS-4] Badge de estado también al iniciar
        btn_train.click(
            fn=lambda: '<div style="display:inline-flex;align-items:center;gap:6px;'
                       'padding:5px 14px;border-radius:20px;font-size:0.75rem;font-weight:600;'
                       'background:rgba(207,188,255,0.10);color:#CFBCFF;'
                       'border:1px solid #4F378A;margin:8px 0;'
                       'animation:pulse 1.5s ease-in-out infinite;">🔄 Entrenando...</div>',
            inputs=[], outputs=[status_badge], queue=False
        )

        # Predicción IQCA
        btn_predecir.click(
            fn=predecir_simple,
            inputs=[fecha_hora_input, temp_input, hum_input, vvel_input, vdir_input, precip_input],
            outputs=[resultado_pred],
        )

    return app


# ──────────────────────────────────────────────────────────────────────────────
# 18. PUNTO DE ENTRADA
# ──────────────────────────────────────────────────────────────────────────────

def _imprimir_ip_fallback() -> None:
    import socket
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8", 80)); print(f"  🖥️  IP local: http://{s.getsockname()[0]}:<PUERTO>"); s.close()
    except Exception: pass
    try:
        ips = subprocess.check_output(["hostname", "-I"], text=True, timeout=4).strip()
        print(f"  🌐  IPs servidor: {ips}")
    except Exception: pass


if __name__ == "__main__":
    app = construir_app()
    print("\n" + "═" * 68)
    print("  🌬️  Quito Air ML — Surrogate Model  v10.5")
    print("  🎨  Material Design 3 Dark · Visual Overhaul")
    print("═" * 68)
    print(f"  Hardware  : {GPU_MSG}")
    print(f"  GPU activa: {GPU_DISPONIBLE}")
    print(f"  FE top-N  : {MAX_FEATURES_SEL} features (RandomForest selector)")
    print("═" * 68)

    try:
        app.launch(server_name="0.0.0.0", server_port=None, share=True,
                   max_threads=40, debug=True, show_error=True,
                   prevent_thread_lock=False, quiet=False)
    except OSError:
        app.launch(server_name="0.0.0.0", server_port=7861, share=True, max_threads=40)
    except Exception:
        _imprimir_ip_fallback()
        app.launch(server_name="0.0.0.0", server_port=None, share=False, max_threads=40)

16:45:37 [INFO] Archivo: dataset_ml_carapungo.csv
16:45:38 [INFO] CSV cargado: 160,262 filas × 19 columnas
16:45:38 [INFO] Timestamp: 'Timestamp'
16:45:38 [INFO] Pandemia excluida: [2020, 2021].
16:45:38 [INFO] Aplicando Feature Engineering avanzado…
16:45:38 [INFO] Seleccionando top-40 features entre 97…
16:45:40 [INFO]   SelectFromModel: 97 → 40 features.
16:45:40 [INFO] Features seleccionadas: 40
16:45:40 [INFO] Dataset limpio: 30,951 filas
16:45:40 [INFO] TimeSeriesSplit (K=5) · Algoritmo: XGBoost  (auto GPU/CPU)
16:45:45 [INFO]   PM25 | Fold 1/5 → MAE=0.712 RMSE=2.231 R²=0.966
16:45:48 [INFO]   PM25 | Fold 2/5 → MAE=0.512 RMSE=1.939 R²=0.972
16:45:52 [INFO]   PM25 | Fold 3/5 → MAE=0.372 RMSE=1.177 R²=0.987
16:45:55 [INFO]   PM25 | Fold 4/5 → MAE=0.658 RMSE=4.611 R²=0.930
16:45:59 [INFO]   PM25 | Fold 5/5 → MAE=0.404 RMSE=1.092 R²=0.989
16:46:01 [INFO]   PM10 | Fold 1/5 → MAE=7.873 RMSE=12.675 R²=0.844
16:46:02 [INFO]   PM10 | Fold 2/5 → MAE=6.658 RMSE=10.595 R²=0.876
16:46:03 [INF

In [ ]:
"""
================================================================================
  SURROGATE MODEL — CALIDAD DEL AIRE  |  Interfaz Web Gradio
  Versión : 10.5-L2  (CSS Blanco Limpio — fix dark-override de Gradio)
  ─── Cambios vs v10.5-L ─────────────────────────────────────────────────────
  [CSS-FIX]  CSS completamente reescrito con selectores ultraespecíficos que
             sobreescriben los fondos oscuros que Gradio inyecta desde su
             tema Base. Todo el UI es blanco/lavanda suave; solo los botones,
             badges y acentos llevan color.
================================================================================
"""

import os, warnings, logging, traceback, subprocess, pickle, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

import gradio as gr
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(message)s",
                    datefmt="%H:%M:%S")
log = logging.getLogger(__name__)

# ──────────────────────────────────────────────────────────────────────────────
# 1.  CONSTANTES
# ──────────────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("resultados_surrogate")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SESION: dict = {
    "modelo": None, "scaler": None, "feat_cols": [],
    "target": "", "parroquia": "", "feat_stats": {},
    "lag_lookup": {}, "ultimo_timestamp": None,
}

ANOS_PANDEMIA    = [2020, 2021]
CONTAMINANTES_Y  = ["PM25", "PM10", "O3", "CO", "NO2", "SO2"]
FEATURES_X_BASE  = [
    "Temperatura", "Humedad", "Viento_Velocidad", "Viento_Direccion", "Precipitacion",
    "hora_sin", "hora_cos", "mes_sin", "mes_cos",
    "PM25_lag_1h",  "PM25_lag_24h", "PM10_lag_1h",  "PM10_lag_24h",
    "O3_lag_1h",    "O3_lag_24h",   "CO_lag_1h",    "CO_lag_24h",
    "NO2_lag_1h",   "NO2_lag_24h",  "SO2_lag_1h",   "SO2_lag_24h",
]
METEO_COLS       = ["Temperatura", "Humedad", "Viento_Velocidad", "Viento_Direccion", "Precipitacion"]
MAX_FEATURES_SEL = 40

# Colores para gráficos (fondo blanco)
COLOR_REAL = "#6750A4"
COLOR_PRED = "#C2670A"
COLOR_POS  = "#166534"
COLOR_NEG  = "#9B1C1C"
COLOR_SEC  = "#B45309"
BG_PLOT    = "#FFFFFF"
TEXT_PLOT  = "#1C1B1F"
GRID_PLOT  = "#EDE9F5"
SURF_PLOT  = "#F5F3FF"

CONTAM_COLORS = {
    "PM25": "#6750A4", "PM10": "#C2670A", "O3":  "#166534",
    "CO":   "#9B1C1C", "NO2": "#0369A1", "SO2": "#92400E",
}


# ──────────────────────────────────────────────────────────────────────────────
# 2.  GPU
# ──────────────────────────────────────────────────────────────────────────────
def detectar_gpu() -> tuple[bool, str]:
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                            "--format=csv,noheader"],
                           capture_output=True, text=True, timeout=8)
        if r.returncode == 0 and r.stdout.strip():
            return True, f"GPU detectada: {r.stdout.strip().split(chr(10))[0]}"
    except Exception:
        pass
    try:
        import torch
        if torch.cuda.is_available():
            return True, f"GPU (torch): {torch.cuda.get_device_name(0)}"
    except ImportError:
        pass
    return False, "No se detectó GPU — CPU"

GPU_DISPONIBLE, GPU_MSG = detectar_gpu()
log.info(GPU_MSG)

def _xgb_tree_method():
    return {"tree_method": "hist", "device": "cuda"} if GPU_DISPONIBLE \
           else {"tree_method": "hist", "device": "cpu"}
def _lgbm_device():
    return "gpu" if GPU_DISPONIBLE else "cpu"


# ──────────────────────────────────────────────────────────────────────────────
# 3.  CSV
# ──────────────────────────────────────────────────────────────────────────────
def _cargar_csv(ruta):
    try:
        return pd.read_csv(ruta, comment="#", low_memory=False, on_bad_lines="warn")
    except Exception as e:
        raise RuntimeError(f"Error al leer '{ruta}': {e}") from e

def _detectar_timestamp(df):
    kw = ("time","fecha","date","hora","datetime","timestamp")
    cands = [c for c in df.columns if any(k in c.lower() for k in kw)]
    if cands: return cands[0]
    for c in df.columns:
        try:
            pd.to_datetime(df[c].dropna().astype(str).iloc[:10], infer_datetime_format=True)
            return c
        except Exception:
            continue
    return None


# ──────────────────────────────────────────────────────────────────────────────
# 4.  DROPDOWN DINÁMICO + RESUMEN CSV
# ──────────────────────────────────────────────────────────────────────────────
def actualizar_targets(archivo):
    if archivo is None:
        return gr.Dropdown(choices=CONTAMINANTES_Y, value="PM25")
    try:
        ruta = archivo if isinstance(archivo, str) else archivo.name
        df   = pd.read_csv(ruta, comment="#", nrows=3, low_memory=False)
        disp = [c for c in CONTAMINANTES_Y if c in df.columns] or ["PM25"]
        return gr.Dropdown(choices=disp, value=disp[0])
    except Exception:
        return gr.Dropdown(choices=CONTAMINANTES_Y, value="PM25")

def _resumen_csv_html(archivo):
    dropdown = actualizar_targets(archivo)
    if archivo is None:
        return dropdown, ""
    try:
        ruta = archivo if isinstance(archivo, str) else archivo.name
        df   = pd.read_csv(ruta, comment="#", low_memory=False)
        ts   = _detectar_timestamp(df)
        contams = [c for c in CONTAMINANTES_Y if c in df.columns]
        badges = "".join([
            f'<span style="background:{CONTAM_COLORS.get(c,"#6750A4")};color:#fff;'
            f'padding:3px 10px;border-radius:20px;font-size:11px;font-weight:700;'
            f'margin:2px 2px;display:inline-block;">{c}</span>'
            for c in contams
        ])
        ts_txt = f'<code style="background:#EDE9F5;color:#4A0E8F;padding:2px 7px;border-radius:4px;font-size:11px;">{ts}</code>' \
                 if ts else '<span style="color:#92400E;font-size:11px;">⚠️ No detectado</span>'
        html = f"""
        <div style="background:#F5F3FF;border:1.5px solid #C8BCEC;border-radius:10px;
                    padding:12px 14px;margin-top:6px;">
            <div style="font-weight:700;color:#1C1B1F;font-size:12px;margin-bottom:8px;">
                ✅ {Path(ruta).name}
                &nbsp;·&nbsp; <span style="font-weight:400;color:#49454F;">
                {df.shape[0]:,} filas × {df.shape[1]} cols</span>
                &nbsp;·&nbsp; {ts_txt}
            </div>
            <div style="display:flex;flex-wrap:wrap;gap:3px;">{badges or
                '<span style="color:#92400E;font-size:11px;">⚠️ Sin contaminantes detectados</span>'
            }</div>
        </div>"""
        return dropdown, html
    except Exception as e:
        return dropdown, f'<div style="color:#9B1C1C;font-size:12px;padding:8px;background:#FEE2E2;border-radius:8px;">❌ {e}</div>'


# ──────────────────────────────────────────────────────────────────────────────
# 5.  PREPROCESAMIENTO
# ──────────────────────────────────────────────────────────────────────────────
def _preprocesar(df, target_col, timestamp_col, excluir_pandemia=True):
    df = df.copy()
    df[timestamp_col] = pd.to_datetime(df[timestamp_col], infer_datetime_format=True, errors="coerce")
    df = df.dropna(subset=[timestamp_col]).set_index(timestamp_col).sort_index()
    if excluir_pandemia:
        df = df[~df.index.year.isin(ANOS_PANDEMIA)]
    obj = df.select_dtypes(include=["object","string"]).columns.tolist()
    if obj: df[obj] = df[obj].apply(pd.to_numeric, errors="coerce")
    return df.dropna(subset=[target_col])

def _dividir_cronologico(X, y, ratio=0.80):
    c = int(len(X)*ratio)
    return X.iloc[:c], X.iloc[c:], y.iloc[:c], y.iloc[c:]

def _resolver_features_x(df, target_col):
    cols = set(df.select_dtypes(include=[np.number]).columns)
    return [c for c in cols if c not in set(CONTAMINANTES_Y) and c != target_col]


# ──────────────────────────────────────────────────────────────────────────────
# 6.  FEATURE ENGINEERING
# ──────────────────────────────────────────────────────────────────────────────
def _agregar_features_avanzadas(df, target_col):
    df = df.copy()
    for col in [c for c in CONTAMINANTES_Y if c in df.columns]:
        for w in [3,6,12,24]:
            df[f"{col}_roll_mean_{w}h"] = df[col].rolling(w, min_periods=1).mean()
            if w >= 6:
                df[f"{col}_roll_std_{w}h"] = df[col].rolling(w, min_periods=2).std()
        for lag in [1,3,6,12,24]:
            df[f"{col}_diff_{lag}h"] = df[col].diff(lag)
    if "Temperatura" in df.columns and "Humedad" in df.columns:
        df["temp_hum"] = df["Temperatura"] * df["Humedad"]
    if "Temperatura" in df.columns and "Viento_Velocidad" in df.columns:
        df["temp_wind"] = df["Temperatura"] * df["Viento_Velocidad"]
    if "Viento_Direccion" in df.columns and "Viento_Velocidad" in df.columns:
        r = np.radians(df["Viento_Direccion"])
        df["wind_u"] = df["Viento_Velocidad"] * np.cos(r)
        df["wind_v"] = df["Viento_Velocidad"] * np.sin(r)
    if hasattr(df.index, "weekday"):
        df["dia_semana"]     = df.index.weekday
        df["dia_semana_sin"] = np.sin(2*np.pi*df["dia_semana"]/7)
        df["dia_semana_cos"] = np.cos(2*np.pi*df["dia_semana"]/7)
        df["es_finde"]       = (df["dia_semana"] >= 5).astype(int)
    return df.dropna()

def _seleccionar_top_features(X_tr, y_tr, n=MAX_FEATURES_SEL, log_fn=None):
    if log_fn is None: log_fn = log.info
    all_cols = X_tr.columns.tolist()
    if len(all_cols) <= n:
        log_fn(f"  Todas las {len(all_cols)} features disponibles."); return all_cols
    try:
        mask  = X_tr.notna().all(axis=1) & y_tr.notna()
        Xo, yo = X_tr[mask].values, y_tr[mask].values
        if len(Xo) < 50: return all_cols
        rf = RandomForestRegressor(n_estimators=80, max_depth=6,
                                   min_samples_leaf=20, n_jobs=-1, random_state=42)
        rf.fit(Xo, yo)
        sel = SelectFromModel(rf, threshold=-np.inf, max_features=n, prefit=True)
        selected = [c for c,k in zip(all_cols, sel.get_support()) if k]
        log_fn(f"  SelectFromModel: {len(all_cols)} → {len(selected)} features.")
        return selected if selected else all_cols
    except Exception as e:
        log_fn(f"  ⚠️ Selección falló ({e}) — usando todas."); return all_cols


# ──────────────────────────────────────────────────────────────────────────────
# 7.  LAG LOOKUP
# ──────────────────────────────────────────────────────────────────────────────
def _crear_lag_lookup(X_train, feat_cols):
    if not hasattr(X_train.index, "hour"): return {}
    df = X_train[feat_cols].copy()
    df["_h"] = X_train.index.hour
    df["_m"] = X_train.index.month
    return {col: {(int(h),int(m)): float(v)
                  for (h,m),v in df.groupby(["_h","_m"])[col].mean().dropna().items()}
            for col in feat_cols if col in df.columns}


# ──────────────────────────────────────────────────────────────────────────────
# 8.  IQCA REMMAQ
# ──────────────────────────────────────────────────────────────────────────────
IQCA_BP = {
    "PM25": [(0.0,12.0,0,50),(12.1,37.4,51,100),(37.5,55.4,101,150),
             (55.5,150.4,151,200),(150.5,250.4,201,300),(250.5,500.4,301,500)],
    "PM10": [(0,54,0,50),(55,154,51,100),(155,254,101,150),
             (255,354,151,200),(355,424,201,300),(425,604,301,500)],
    "O3":   [(0,54,0,50),(55,124,51,100),(125,164,101,150),
             (165,204,151,200),(205,404,201,300),(405,604,301,500)],
    "CO":   [(0.0,4.4,0,50),(4.5,9.4,51,100),(9.5,12.4,101,150),
             (12.5,15.4,151,200),(15.5,30.4,201,300),(30.5,50.4,301,500)],
    "NO2":  [(0,53,0,50),(54,100,51,100),(101,360,101,150),
             (361,649,151,200),(650,1249,201,300),(1250,2049,301,500)],
    "SO2":  [(0,35,0,50),(36,75,51,100),(76,185,101,150),
             (186,304,151,200),(305,604,201,300),(605,1004,301,500)],
}
CATS_IQCA = [
    (0,50,"🟢 Deseable","#166534"),(51,100,"🟡 Aceptable","#92400E"),
    (101,150,"🟠 Precaución","#C2670A"),(151,200,"🔴 Alerta","#9B1C1C"),
    (201,300,"🟣 Alarma","#581C87"),(301,500,"⚫ Emergencia","#1C1B1F"),
]

def calcular_iqca(cont, conc):
    bp = IQCA_BP.get(cont)
    if not bp: return None
    for c_lo,c_hi,i_lo,i_hi in bp:
        if c_lo <= conc <= c_hi:
            return i_lo + (conc-c_lo)*(i_hi-i_lo)/(c_hi-c_lo)
    return 500.0 if conc > bp[-1][1] else 0.0

def categoria_iqca(v):
    for lo,hi,lbl,col in CATS_IQCA:
        if lo <= v <= hi: return lbl, col
    return "⚫ Emergencia","#1C1B1F"


# ──────────────────────────────────────────────────────────────────────────────
# 9.  PREDICCIÓN SIMPLIFICADA
# ──────────────────────────────────────────────────────────────────────────────
def predecir_simple(fecha_hora, temperatura, humedad, viento_vel, viento_dir, precipitacion):
    if SESION["modelo"] is None:
        return "### ⚠️ Entrena primero un modelo."
    try:
        if not str(fecha_hora).strip():
            return "### ❌ Ingresa una fecha y hora válida."
        try:
            ts = pd.Timestamp(str(fecha_hora))
        except Exception:
            return f"### ❌ Formato inválido: usa `YYYY-MM-DD HH:MM`"
        hora, mes = ts.hour, ts.month
        ultimo = SESION.get("ultimo_timestamp")
        adv = ""
        if ultimo:
            dh = (ts - ultimo).total_seconds()/3600
            if dh > 48:
                adv = f"\n\n> ⚠️ **Extrapolación**: {dh:.0f}h después del último dato ({ultimo:%Y-%m-%d %H:%M}). Los lags son promedios climatológicos."
        ciclicas = {
            "hora_sin": float(np.sin(2*np.pi*hora/24)), "hora_cos": float(np.cos(2*np.pi*hora/24)),
            "mes_sin":  float(np.sin(2*np.pi*mes/12)),  "mes_cos":  float(np.cos(2*np.pi*mes/12)),
        }
        meteo = {"Temperatura": float(temperatura), "Humedad": float(humedad),
                 "Viento_Velocidad": float(viento_vel), "Viento_Direccion": float(viento_dir),
                 "Precipitacion": float(precipitacion)}
        dr = np.radians(float(viento_dir))
        deriv = {
            "temp_hum": float(temperatura)*float(humedad),
            "temp_wind": float(temperatura)*float(viento_vel),
            "wind_u": float(viento_vel)*np.cos(dr), "wind_v": float(viento_vel)*np.sin(dr),
            "dia_semana": float(ts.weekday()),
            "dia_semana_sin": float(np.sin(2*np.pi*ts.weekday()/7)),
            "dia_semana_cos": float(np.cos(2*np.pi*ts.weekday()/7)),
            "es_finde": float(1 if ts.weekday()>=5 else 0),
        }
        feat_cols = SESION["feat_cols"]
        lag_lk    = SESION.get("lag_lookup", {})
        row = {}
        for col in feat_cols:
            if col in meteo: row[col] = meteo[col]
            elif col in ciclicas: row[col] = ciclicas[col]
            elif col in deriv: row[col] = deriv[col]
            elif col in lag_lk and (hora,mes) in lag_lk[col]: row[col] = lag_lk[col][(hora,mes)]
            else: row[col] = SESION["feat_stats"].get(col,{}).get("mean",0.0)
        X_in = pd.DataFrame([row])[feat_cols]
        sc   = SESION["scaler"]
        Xsc  = sc.transform(X_in) if sc else X_in.values
        pred = float(SESION["modelo"].predict(Xsc)[0])
        tgt  = SESION["target"]
        iqca_v = calcular_iqca(tgt, pred)
        iqca_r = (f"| **IQCA** | `{iqca_v:.1f}` |\n| **Categoría** | {categoria_iqca(iqca_v)[0]} |"
                  if iqca_v is not None else f"| **IQCA** | N/D para {tgt} |")
        n_lk = sum(1 for c in feat_cols if c in lag_lk and (hora,mes) in lag_lk.get(c,{}))
        return (f"## 🔮 Predicción IQCA — `{tgt}` · *{SESION['parroquia']}*\n\n"
                f"| Campo | Valor |\n|---|---|\n"
                f"| **Fecha/Hora** | `{ts:%Y-%m-%d %H:%M}` |\n"
                f"| **{tgt} estimado** | `{pred:.3f} µg/m³` |\n"
                f"{iqca_r}\n\n_Lags imputados desde lag_lookup: {n_lk}/{len(feat_cols)}_"
                f"{adv}")
    except Exception:
        return f"### ❌ Error\n```\n{traceback.format_exc()}\n```"


# ──────────────────────────────────────────────────────────────────────────────
# 10. MODELO
# ──────────────────────────────────────────────────────────────────────────────
def _construir_modelo(algoritmo, params=None):
    if not params: params = {}
    if "XGBoost" in algoritmo:
        import xgboost as xgb
        return xgb.XGBRegressor(
            n_estimators=2000, max_depth=int(params.get("max_depth",5)),
            learning_rate=float(params.get("learning_rate",0.05)),
            subsample=float(params.get("subsample",0.8)),
            colsample_bytree=float(params.get("colsample_bytree",0.8)),
            reg_lambda=float(params.get("reg_lambda",2.0)),
            reg_alpha=float(params.get("reg_alpha",0.5)),
            gamma=float(params.get("gamma",0.0)),
            min_child_weight=float(params.get("min_child_weight",1)),
            early_stopping_rounds=50, eval_metric="rmse",
            random_state=42, verbosity=0, **_xgb_tree_method())
    elif "LightGBM" in algoritmo:
        import lightgbm as lgb
        return lgb.LGBMRegressor(
            n_estimators=2000, max_depth=int(params.get("max_depth",5)),
            learning_rate=float(params.get("learning_rate",0.05)),
            subsample=float(params.get("subsample",0.8)),
            colsample_bytree=float(params.get("colsample_bytree",0.8)),
            reg_lambda=float(params.get("reg_lambda",2.0)),
            reg_alpha=float(params.get("reg_alpha",0.5)),
            min_child_samples=int(params.get("min_child_weight",30)),
            metric="rmse", device=_lgbm_device(), random_state=42, verbose=-1)
    else:
        return HistGradientBoostingRegressor(
            max_iter=1000, early_stopping=True, n_iter_no_change=30,
            validation_fraction=0.1, max_depth=int(params.get("max_depth",5)),
            min_samples_leaf=int(params.get("min_child_weight",30)),
            learning_rate=float(params.get("learning_rate",0.05)),
            l2_regularization=float(params.get("reg_lambda",1.0)), random_state=42)


# ──────────────────────────────────────────────────────────────────────────────
# 11. K-FOLD
# ──────────────────────────────────────────────────────────────────────────────
def _extraer_curvas(modelo, algoritmo):
    tr, va, met = [], [], "Score"
    try:
        if "XGBoost" in algoritmo:
            ev = modelo.evals_result(); ks = list(ev.keys()); m = list(ev[ks[0]].keys())[0]
            met = m.upper(); tr = list(ev[ks[0]][m]); va = list(ev[ks[1]][m]) if len(ks)>1 else []
        elif "LightGBM" in algoritmo:
            ev = modelo.evals_result_; ks = list(ev.keys()); m = list(ev[ks[0]].keys())[0]
            met = m.upper(); tr = list(ev[ks[0]][m]); va = list(ev[ks[1]][m]) if len(ks)>1 else []
        else:
            if hasattr(modelo,"train_score_") and modelo.train_score_ is not None:
                tr = list(modelo.train_score_); met = "R²"
            if hasattr(modelo,"validation_score_") and modelo.validation_score_ is not None:
                va = list(modelo.validation_score_)
    except Exception as e: log.warning(f"curvas: {e}")
    return tr, va, met

def _entrenar_kfold(df, algoritmo, n_splits=5, log_fn=None, xgb_params=None):
    if not log_fn: log_fn = log.info
    tscv = TimeSeriesSplit(n_splits=n_splits)
    filas, all_curves = [], {}
    cols = set(df.columns)
    for cont in CONTAMINANTES_Y:
        if cont not in cols: log_fn(f"  ⏭️ {cont} no disponible."); continue
        fc = _resolver_features_x(df, cont)
        y  = df[cont].dropna(); X = df.loc[y.index, fc]
        m  = X.notna().all(axis=1) & y.notna(); X, y = X[m], y[m]
        if len(X) < n_splits*20: log_fn(f"  ⚠️ {cont}: {len(X)} filas."); continue
        Xa, ya = X.values, y.values
        fm, ftr, fva, met_cv = [], [], [], "Score"
        for fi,(tr,va) in enumerate(tscv.split(Xa), 1):
            Xtr,Xva = Xa[tr],Xa[va]; ytr,yva = ya[tr],ya[va]
            sc = StandardScaler(); Xtr_s = sc.fit_transform(Xtr); Xva_s = sc.transform(Xva)
            mod = _construir_modelo(algoritmo, xgb_params)
            if "XGBoost" in algoritmo:
                mod.fit(Xtr_s, ytr, eval_set=[(Xtr_s,ytr),(Xva_s,yva)], verbose=False)
            elif "LightGBM" in algoritmo:
                import lightgbm as lgb
                mod.fit(Xtr_s, ytr, eval_set=[(Xtr_s,ytr),(Xva_s,yva)],
                        callbacks=[lgb.early_stopping(30,verbose=False),lgb.log_evaluation(-1)])
            else: mod.fit(Xtr_s, ytr)
            th,vh,met_cv2 = _extraer_curvas(mod, algoritmo)
            if th: ftr.append(th)
            if vh: fva.append(vh)
            met_cv = met_cv2
            yp = mod.predict(Xva_s)
            fm.append({"MAE":mean_absolute_error(yva,yp),
                       "RMSE":float(np.sqrt(mean_squared_error(yva,yp))),
                       "R2":r2_score(yva,yp)})
            log_fn(f"  {cont} | Fold {fi}/{n_splits} → MAE={fm[-1]['MAE']:.3f} R²={fm[-1]['R2']:.3f}")
        all_curves[cont] = {"train":ftr,"val":fva,"metric":met_cv}
        mf = pd.DataFrame(fm)
        filas.append({"Contaminante":cont,"Features_X":len(fc),
                      "MAE_mean":mf["MAE"].mean(),"MAE_std":mf["MAE"].std(),
                      "RMSE_mean":mf["RMSE"].mean(),"RMSE_std":mf["RMSE"].std(),
                      "R2_mean":mf["R2"].mean(),"R2_std":mf["R2"].std()})
    return (pd.DataFrame(filas) if filas else pd.DataFrame()), all_curves


# ──────────────────────────────────────────────────────────────────────────────
# 12. FIGURAS
# ──────────────────────────────────────────────────────────────────────────────
def _ax_style(ax, titulo, xlabel, ylabel):
    ax.set_facecolor(BG_PLOT)
    ax.set_title(titulo, fontsize=11, fontweight="bold", color=TEXT_PLOT, pad=12)
    if xlabel: ax.set_xlabel(xlabel, color=TEXT_PLOT, fontsize=10)
    if ylabel: ax.set_ylabel(ylabel, color=TEXT_PLOT, fontsize=10)
    ax.tick_params(colors=TEXT_PLOT, labelsize=9)
    ax.grid(True, linestyle="--", alpha=0.45, color=GRID_PLOT)
    for sp in ax.spines.values(): sp.set_edgecolor("#D1C9E8"); sp.set_linewidth(0.8)

def _fig_pred(y_test, y_pred, target_col, days, parroquia=""):
    dfp = pd.DataFrame({"Real":y_test.values,"Predicho":y_pred}, index=y_test.index)
    dfp = dfp[dfp.index >= dfp.index.max()-pd.Timedelta(days=days)]
    fig, ax = plt.subplots(figsize=(13,4.5), facecolor=BG_PLOT); ax.set_facecolor(BG_PLOT)
    ax.plot(dfp.index, dfp["Real"],     color=COLOR_REAL, lw=2.0, alpha=0.95, label="Real")
    ax.plot(dfp.index, dfp["Predicho"], color=COLOR_PRED, lw=1.6, linestyle="--", alpha=0.90, label="Predicho")
    ax.fill_between(dfp.index, dfp["Real"], dfp["Predicho"], alpha=0.08, color=COLOR_PRED)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b")); ax.xaxis.set_major_locator(mdates.DayLocator())
    plt.xticks(rotation=28, ha="right", color=TEXT_PLOT, fontsize=9); plt.yticks(color=TEXT_PLOT, fontsize=9)
    titulo = f"Real vs Predicho — {target_col}  ·  Test Ciego (últimos {days} días)"
    if parroquia: titulo = f"[{parroquia}]  {titulo}"
    _ax_style(ax, titulo, "Fecha", target_col)
    ax.legend(framealpha=0.92, labelcolor=TEXT_PLOT, facecolor=SURF_PLOT, edgecolor="#D1C9E8", fontsize=10)
    plt.tight_layout(); return fig

def _fig_fi(fi_df, target_col, parroquia=""):
    n = len(fi_df)
    fig, ax = plt.subplots(figsize=(9, max(4, n*0.52)), facecolor=BG_PLOT); ax.set_facecolor(BG_PLOT)
    colores = [COLOR_POS if v>=0 else COLOR_NEG for v in fi_df["Importance"]]
    ax.barh(fi_df["Feature"][::-1], fi_df["Importance"][::-1],
            xerr=fi_df["Std"][::-1], color=colores[::-1],
            align="center", alpha=0.80, ecolor="#9E97B0", capsize=3, height=0.65)
    ax.axvline(0, color=COLOR_REAL, lw=0.9, linestyle="--", alpha=0.5)
    titulo = f"Feature Importance — {target_col}  (Permutation Δ R²)"
    if parroquia: titulo = f"[{parroquia}]  {titulo}"
    _ax_style(ax, titulo, "Importancia media (Δ R²)", "")
    plt.tight_layout(); return fig

def _fig_heatmap(df_source, target_col, parroquia=""):
    contams = [c for c in CONTAMINANTES_Y if c in df_source.columns]
    if len(contams) < 2:
        fig, ax = plt.subplots(figsize=(5,3), facecolor=BG_PLOT); ax.set_facecolor(BG_PLOT)
        ax.text(0.5,0.5,f"Solo {len(contams)} contaminante disponible.",
                ha="center",va="center",color=TEXT_PLOT,transform=ax.transAxes); ax.axis("off")
        plt.tight_layout(); return fig
    df_c = df_source[contams].dropna()
    if df_c.shape[0] < 10: return None
    corr = df_c.corr(method="pearson")
    n    = len(corr)
    fig, ax = plt.subplots(figsize=(max(5.5, n*1.2), max(4.5, n*1.1)), facecolor=BG_PLOT)
    ax.set_facecolor(BG_PLOT)
    mask = np.zeros_like(corr, dtype=bool); mask[np.triu_indices_from(mask, k=1)] = True
    cmap = sns.diverging_palette(220,15,s=80,l=45,as_cmap=True)
    sns.heatmap(corr, mask=mask, cmap=cmap, vmin=-1, vmax=1, center=0,
                annot=True, fmt=".2f", annot_kws={"size":11,"color":"#1C1B1F","fontweight":"bold"},
                linewidths=1.2, linecolor="#E8E3F2", square=True, ax=ax,
                cbar_kws={"shrink":0.72,"pad":0.02})
    ax.collections[0].colorbar.ax.tick_params(colors=TEXT_PLOT, labelsize=9)
    cols_l = list(corr.columns)
    if target_col in cols_l:
        i = cols_l.index(target_col); c = CONTAM_COLORS.get(target_col,"#6750A4")
        ax.add_patch(plt.Rectangle((i,0),1,n,fill=False,edgecolor=c,lw=3,clip_on=False,zorder=5))
        ax.add_patch(plt.Rectangle((0,i),n,1,fill=False,edgecolor=c,lw=3,clip_on=False,zorder=5))
    titulo = f"Correlación de Pearson — Contaminantes ({', '.join(contams)})"
    if parroquia: titulo = f"[{parroquia}]  {titulo}"
    ax.set_title(titulo, fontsize=11, fontweight="bold", color=TEXT_PLOT, pad=14)
    ax.tick_params(colors=TEXT_PLOT, labelsize=10)
    plt.setp(ax.get_xticklabels(), rotation=0, ha="center", fontweight="600")
    plt.setp(ax.get_yticklabels(), rotation=0, fontweight="600")
    for lbl in ax.get_xticklabels(): lbl.set_color(CONTAM_COLORS.get(lbl.get_text(), TEXT_PLOT))
    for lbl in ax.get_yticklabels(): lbl.set_color(CONTAM_COLORS.get(lbl.get_text(), TEXT_PLOT))
    plt.tight_layout(rect=[0,0.03,1,1]); return fig

def _fig_curva(curvas, target_col, algoritmo, parroquia="", k=5):
    fig, ax = plt.subplots(figsize=(13,5), facecolor=BG_PLOT); ax.set_facecolor(BG_PLOT)
    if not curvas or not curvas.get("train"):
        ax.text(0.5,0.5,"No hay datos de curva.",ha="center",va="center",
                color=TEXT_PLOT,fontsize=11,transform=ax.transAxes)
        _ax_style(ax,f"Curva — {target_col}","",""); plt.tight_layout(); return fig
    tc  = curvas["train"]; vc = curvas.get("val",[]); met = curvas.get("metric","Score")
    is_r2 = met=="R²"
    lengths = [len(c) for c in (tc+(vc if vc else [])) if c]
    if not lengths or min(lengths) < 2:
        ax.text(0.5,0.5,"Historial corto.",ha="center",va="center",color=TEXT_PLOT,transform=ax.transAxes)
        _ax_style(ax,f"Curva — {target_col}","",""); plt.tight_layout(); return fig
    ml = min(lengths); x = np.arange(ml)
    ta = np.array([c[:ml] for c in tc]); tm,ts = ta.mean(0),ta.std(0)
    for c in ta: ax.plot(x,c,color=COLOR_REAL,alpha=0.10,lw=0.75,zorder=2)
    ax.plot(x,tm,color=COLOR_REAL,lw=2.2,label=f"Train — {met}",zorder=5)
    ax.fill_between(x,tm-ts,tm+ts,alpha=0.12,color=COLOR_REAL,zorder=3)
    if vc:
        va = np.array([c[:ml] for c in vc]); vm,vs = va.mean(0),va.std(0)
        for c in va: ax.plot(x,c,color=COLOR_PRED,alpha=0.10,lw=0.75,zorder=2)
        ax.plot(x,vm,color=COLOR_PRED,lw=2.2,linestyle="--",label=f"Val — {met}",zorder=5)
        ax.fill_between(x,vm-vs,vm+vs,alpha=0.12,color=COLOR_PRED,zorder=3)
        bi = int(np.argmax(vm) if is_r2 else np.argmin(vm)); bv = vm[bi]
        ax.axvline(bi,color=COLOR_SEC,lw=1.6,linestyle=":",alpha=0.9,
                   label=f"Mejor iter: {bi}  ({bv:.4f})",zorder=6)
        ax.scatter([bi],[bv],color=COLOR_SEC,s=65,zorder=8)
    al = algoritmo.split("(")[0].strip()
    titulo = f"Curva de Aprendizaje — {target_col}  ·  {al}  ·  TSS (K={k})"
    if parroquia: titulo = f"[{parroquia}]  {titulo}"
    _ax_style(ax,titulo,"Iteración","R²  (↑ mejor)" if is_r2 else f"{met}  (↓ mejor)")
    ax.legend(framealpha=0.92,labelcolor=TEXT_PLOT,facecolor=SURF_PLOT,
              edgecolor="#D1C9E8",fontsize=9,loc="best")
    plt.tight_layout(); return fig

def _fig_mae(kf, target_col, parroquia=""):
    if kf.empty: return None
    cont = kf["Contaminante"].tolist()
    mm,ms = kf["MAE_mean"].values,kf["MAE_std"].values
    rm,rs = kf["RMSE_mean"].values,kf["RMSE_std"].values
    x = np.arange(len(cont)); w=0.38
    fig,ax = plt.subplots(figsize=(11,5),facecolor=BG_PLOT); ax.set_facecolor(BG_PLOT)
    b1 = ax.bar(x-w/2,mm,w,yerr=ms,label="MAE",color=COLOR_REAL,alpha=0.75,ecolor="#9E97B0",capsize=4)
    b2 = ax.bar(x+w/2,rm,w,yerr=rs,label="RMSE",color=COLOR_PRED,alpha=0.75,ecolor="#9E97B0",capsize=4)
    if target_col in cont:
        i=cont.index(target_col)
        for b in (b1[i],b2[i]): b.set_edgecolor("#6750A4"); b.set_linewidth(2.5)
    ax.set_xticks(x); ax.set_xticklabels(cont,color=TEXT_PLOT,fontsize=10)
    titulo = "MAE / RMSE por Contaminante  ·  TimeSeriesSplit"
    if parroquia: titulo = f"[{parroquia}]  {titulo}"
    _ax_style(ax,titulo,"Contaminante","Error (µg/m³)")
    ax.legend(framealpha=0.92,labelcolor=TEXT_PLOT,facecolor=SURF_PLOT,edgecolor="#D1C9E8",fontsize=10)
    plt.tight_layout(); return fig

def _fig_r2(kf, target_col, parroquia=""):
    if kf.empty: return None
    df = kf.sort_values("R2_mean",ascending=True).reset_index(drop=True)
    cont = df["Contaminante"].tolist(); rm = df["R2_mean"].values; rs = df["R2_std"].values
    colors = ["#166534" if v>=0.70 else ("#92400E" if v>=0.50 else "#9B1C1C") for v in rm]
    fig,ax = plt.subplots(figsize=(10,max(3.5,len(cont)*0.70)),facecolor=BG_PLOT); ax.set_facecolor(BG_PLOT)
    bars = ax.barh(cont,rm,xerr=rs,color=colors,alpha=0.80,ecolor="#9E97B0",capsize=4,height=0.58)
    if target_col in cont:
        i=cont.index(target_col); bars[i].set_edgecolor("#6750A4"); bars[i].set_linewidth(2.5)
    for xv,lbl,col,ls in [(0.5,"Mínimo (0.5)","#92400E",":"),(0.7,"Bueno (0.7)","#166534","--"),(0.9,"Excelente (0.9)",COLOR_REAL,"-.")]:
        ax.axvline(xv,color=col,lw=1.1,linestyle=ls,alpha=0.65,label=lbl)
    for i,(v,s) in enumerate(zip(rm,rs)):
        ax.text(min(v+0.01,1.0),i,f"{v:.3f}±{s:.3f}",va="center",color=TEXT_PLOT,fontsize=8.5)
    ax.set_xlim(0,1.05)
    titulo = "R² por Contaminante  ·  TimeSeriesSplit"
    if parroquia: titulo = f"[{parroquia}]  {titulo}"
    _ax_style(ax,titulo,"R²","")
    ax.legend(framealpha=0.92,labelcolor=TEXT_PLOT,facecolor=SURF_PLOT,edgecolor="#D1C9E8",fontsize=8.5,loc="lower right")
    plt.tight_layout(); return fig

def _fig_pie(fi_df, target_col, parroquia=""):
    if fi_df.empty: return None
    pos = fi_df[fi_df["Importance"]>0].reset_index(drop=True)
    if pos.empty: return None
    top = pos.head(10); otros = pos.iloc[10:]
    labels = top["Feature"].tolist(); vals = top["Importance"].tolist()
    if len(otros)>0 and otros["Importance"].sum()>0:
        labels.append(f"Otros ({len(otros)})"); vals.append(float(otros["Importance"].sum()))
    pal = ["#6750A4","#C2670A","#166534","#9B1C1C","#0369A1","#92400E",
           "#7C3AED","#BE185D","#065F46","#B45309","#1D4ED8","#6B21A8"]
    cols = [pal[i%len(pal)] for i in range(len(labels))]
    fig,ax = plt.subplots(figsize=(9,7),facecolor=BG_PLOT); ax.set_facecolor(BG_PLOT)
    wedges,_,autotexts = ax.pie(vals,colors=cols,autopct="%1.1f%%",startangle=140,pctdistance=0.80,
                                wedgeprops={"edgecolor":"#FFFFFF","linewidth":2.5,"antialiased":True})
    for at in autotexts: at.set_fontsize(8.5); at.set_color("#FFFFFF"); at.set_fontweight("bold")
    ax.add_patch(plt.Circle((0,0),0.55,fc="#FFFFFF"))
    ax.text(0,0.08,target_col,ha="center",va="center",color=TEXT_PLOT,fontsize=14,fontweight="bold")
    ax.text(0,-0.10,"top features",ha="center",va="center",color="#49454F",fontsize=8.5)
    ax.legend(wedges,labels,loc="lower center",bbox_to_anchor=(0.5,-0.16),
              ncol=3,framealpha=0.92,labelcolor=TEXT_PLOT,facecolor=BG_PLOT,
              edgecolor="#D1C9E8",fontsize=8.5)
    titulo = f"Distribución de Importancia — {target_col}  (top 10)"
    if parroquia: titulo = f"[{parroquia}]  {titulo}"
    ax.set_title(titulo,fontsize=11,fontweight="bold",color=TEXT_PLOT,pad=14)
    plt.tight_layout(rect=[0,0.10,1,1]); return fig


# ──────────────────────────────────────────────────────────────────────────────
# 13. PIPELINE PRINCIPAL
# ──────────────────────────────────────────────────────────────────────────────
def entrenar(csv_upload, target_col, nombre_modelo, algoritmo,
             plot_days, train_ratio, excluir_pandemia, k_splits,
             max_depth, learning_rate, subsample, colsample_bytree,
             reg_lambda, reg_alpha, gamma, min_child_weight,
             usar_feature_engineering):
    logs = []
    def info(m): log.info(m);    logs.append(f"✅ {m}")
    def warn(m): log.warning(m); logs.append(f"⚠️  {m}")
    def err(m):  log.error(m);   logs.append(f"❌ {m}")
    def _est():  return "  ·  ".join(logs[-30:])
    V7 = (None,)*7

    try:
        if csv_upload is None: return "### ❌ Sube un archivo CSV.", *V7, _est()
        ruta = csv_upload if isinstance(csv_upload,str) else csv_upload.name
        target_col = target_col.strip(); nombre_modelo = nombre_modelo.strip() or "surrogate"
        parroquia  = Path(ruta).stem.replace("_"," ").title()
        info(f"Archivo: {Path(ruta).name}")

        df = _cargar_csv(ruta); info(f"CSV: {df.shape[0]:,} × {df.shape[1]}")
        ts = _detectar_timestamp(df)
        if not ts: return "### ❌ Sin columna timestamp.", *V7, _est()
        if target_col not in df.columns:
            return f"### ❌ Target `{target_col}` no encontrado.\n\nColumnas: `{', '.join(df.columns)}`", *V7, _est()

        df_p = _preprocesar(df, target_col, ts, excluir_pandemia)
        if excluir_pandemia: info(f"Pandemia {ANOS_PANDEMIA} excluida.")
        df_ch = df_p[[c for c in CONTAMINANTES_Y if c in df_p.columns]].copy()

        y = df_p[target_col]; X = df_p.drop(columns=[target_col])
        if usar_feature_engineering:
            info("Feature Engineering…")
            df_f = _agregar_features_avanzadas(pd.concat([X,y],axis=1), target_col)
            y = df_f[target_col]; X = df_f.drop(columns=[target_col])
            Xt,_,yt,_ = _dividir_cronologico(X,y,train_ratio)
            sel = _seleccionar_top_features(Xt,yt,MAX_FEATURES_SEL,log_fn=info)
            X = X[sel]; feat_cols = list(X.columns); info(f"Features: {len(feat_cols)}")
        else:
            feat_cols = [c for c in X.select_dtypes(include=[np.number]).columns if c != target_col]
            X = X[feat_cols]; info(f"Features originales: {len(feat_cols)}")

        n_nan = int(X.isna().sum().sum())
        if n_nan: warn(f"{n_nan:,} NaN → eliminando filas.")
        mask = X.notna().all(axis=1); X = X.loc[mask]; y = y.loc[mask]
        df_c = pd.concat([X,y],axis=1); info(f"Limpio: {df_c.shape[0]:,} filas")

        xp = {"max_depth":int(max_depth),"learning_rate":float(learning_rate),
              "subsample":float(subsample),"colsample_bytree":float(colsample_bytree),
              "reg_lambda":float(reg_lambda),"reg_alpha":float(reg_alpha),
              "gamma":float(gamma),"min_child_weight":float(min_child_weight)}

        info(f"TSS (K={k_splits}) · {algoritmo}")
        kl = []
        def klog(m): log.info(m); kl.append(m)
        kf_r, kf_c = _entrenar_kfold(df_c, algoritmo, k_splits, klog, xp)
        for l in kl: logs.append(l)
        if kf_r.empty: warn("TSS sin resultados.")
        fig_lc = _fig_curva(kf_c.get(target_col,{}), target_col, algoritmo, parroquia, k_splits)

        Xtr,Xte,ytr,yte = _dividir_cronologico(X,y,train_ratio)
        sc2  = int(len(Xtr)*0.80)
        Xst  = Xtr.iloc[:sc2]; Xvs = Xtr.iloc[sc2:]
        yst  = ytr.iloc[:sc2]; yvs = ytr.iloc[sc2:]
        scf  = StandardScaler()
        Xst_s = scf.fit_transform(Xst); Xvs_s = scf.transform(Xvs)
        Xte_s = scf.transform(Xte); Xtr_s = np.vstack([Xst_s,Xvs_s]); ytr_n = ytr.values

        mf = _construir_modelo(algoritmo, xp)
        if "XGBoost" in algoritmo:
            mf.fit(Xst_s,yst.values,eval_set=[(Xvs_s,yvs.values)],verbose=False)
        elif "LightGBM" in algoritmo:
            import lightgbm as lgb
            mf.fit(Xst_s,yst.values,eval_set=[(Xvs_s,yvs.values)],
                   callbacks=[lgb.early_stopping(30,verbose=False),lgb.log_evaluation(-1)])
        else: mf.fit(Xst_s,yst.values)

        yptr = mf.predict(Xtr_s); ypte = mf.predict(Xte_s)
        def _m(yt,yp): return dict(MAE=mean_absolute_error(yt,yp),
                                   RMSE=float(np.sqrt(mean_squared_error(yt,yp))),
                                   R2=r2_score(yt,yp))
        mtr = _m(ytr_n,yptr); mte = _m(yte.values,ypte)
        gap = mtr["R2"]-mte["R2"]
        if gap>0.15: warn(f"Posible overfitting: ΔR²={gap:.3f}")

        info("Generando lag_lookup…")
        lag_lk = _crear_lag_lookup(Xtr, feat_cols)
        ults   = Xtr.index.max() if hasattr(Xtr.index,"max") else None
        info(f"lag_lookup: {len(lag_lk)} cols · último: {ults}")

        hw = f"{'🟢 GPU' if GPU_DISPONIBLE else '🔵 CPU'} — {GPU_MSG}"
        pan = "🚫 2020-2021 excluidos" if excluir_pandemia else "⚠️ Pandemia incluida"

        if not kf_r.empty:
            tkf = "\n".join([
                f"| **{r['Contaminante']}** | `{int(r['Features_X'])}` "
                f"| `{r['MAE_mean']:.3f}±{r['MAE_std']:.3f}` "
                f"| `{r['RMSE_mean']:.3f}±{r['RMSE_std']:.3f}` "
                f"| {'🟢' if r['R2_mean']>=0.7 else ('🟡' if r['R2_mean']>=0.5 else '🔴')} "
                f"`{r['R2_mean']:.3f}±{r['R2_std']:.3f}` |"
                for _,r in kf_r.iterrows()
            ])
        else: tkf = "| — | — | — | — | — |"

        md = f"""
## 📊 Surrogate Model v10.5-L2 — *{parroquia}*

### TimeSeriesSplit (K={k_splits}) — 6 Contaminantes

| Contaminante | Features | MAE (μ±σ) | RMSE (μ±σ) | R² (μ±σ) |
|:---:|:---:|:---:|:---:|:---:|
{tkf}

---

### Modelo Final — `{target_col}` (Split {int(train_ratio*100)}/{int((1-train_ratio)*100)})

| Métrica | 🔵 Train | 🟠 Test ciego |
|---|:---:|:---:|
| **MAE** | `{mtr['MAE']:.4f}` | `{mte['MAE']:.4f}` |
| **RMSE** | `{mtr['RMSE']:.4f}` | `{mte['RMSE']:.4f}` |
| **R²** | `{mtr['R2']:.4f}` | `{mte['R2']:.4f}` |
| **Precisión (R² %)** | `{mtr['R2']*100:.2f}%` | `{mte['R2']*100:.2f}%` |

{"⚠️ **Posible overfitting** — ΔR² = `" + f"{gap:.3f}`" if gap>0.15 else "✅ Sin señales de overfitting."}

| Parámetro | Valor |
|---|---|
| Hardware | {hw} | Features | `{len(feat_cols)}` |
| Pandemia | {pan} | FE | {'✅ top-'+str(MAX_FEATURES_SEL) if usar_feature_engineering else '❌'} |
"""

        perm = permutation_importance(mf,Xte_s,yte.values,n_repeats=8,random_state=42,scoring="r2")
        fi = pd.DataFrame({"Feature":feat_cols,"Importance":perm.importances_mean,
                            "Std":perm.importances_std}).sort_values("Importance",ascending=False).reset_index(drop=True)

        fig_mae = _fig_mae(kf_r, target_col, parroquia)
        fig_r2  = _fig_r2(kf_r, target_col, parroquia)
        fig_pie = _fig_pie(fi, target_col, parroquia)

        tag = f"{nombre_modelo}_{parroquia.replace(' ','_')}"
        pkl = OUTPUT_DIR/f"{tag}.pkl"
        with open(pkl,"wb") as f:
            pickle.dump({"modelo":mf,"scaler":scf,"features":feat_cols,"target":target_col,
                         "parroquia":parroquia,"kfold_resumen":kf_r,"kf_curvas":kf_c,
                         "lag_lookup":lag_lk,"ultimo_timestamp":ults},f)
        info(f"PKL guardado: {pkl}")
        fi.to_csv(OUTPUT_DIR/f"{tag}_fi.csv",index=False)
        if not kf_r.empty: kf_r.to_csv(OUTPUT_DIR/f"{tag}_kfold.csv",index=False)

        SESION.update({"modelo":mf,"scaler":scf,"feat_cols":feat_cols,"target":target_col,
                       "parroquia":parroquia,
                       "feat_stats":{col:{"min":float(X[col].min()),"max":float(X[col].max()),
                                          "mean":float(X[col].mean())} for col in feat_cols},
                       "lag_lookup":lag_lk,"ultimo_timestamp":ults})
        info("Sesión lista.")

        fig_pred = _fig_pred(pd.Series(yte.values,index=Xte.index,name=target_col),
                             ypte, target_col, plot_days, parroquia)
        fig_fi   = _fig_fi(fi, target_col, parroquia)
        fig_hm   = _fig_heatmap(df_ch, target_col, parroquia)
        info("Figuras listas.")

        return md, fig_mae, fig_pred, fig_lc, fig_r2, fig_fi, fig_pie, fig_hm, _est()

    except ImportError as e:
        err(str(e)); pkg = str(e).split("'")[-2] if "'" in str(e) else str(e).split()[-1]
        return f"### ❌ Instala: `pip install {pkg}`", *V7, _est()
    except Exception as e:
        err(str(e)); return f"### ❌ Error\n```\n{traceback.format_exc()}\n```", *V7, _est()


# ──────────────────────────────────────────────────────────────────────────────
# 14. BADGE DE ESTADO
# ──────────────────────────────────────────────────────────────────────────────
def _badge(md):
    if md and "❌" not in md and "Error" not in md:
        return '<div style="display:inline-flex;align-items:center;gap:6px;padding:5px 14px;border-radius:20px;font-size:0.73rem;font-weight:600;background:#DCFCE7;color:#166534;border:1.5px solid #16A34A;">✅ Completado</div>'
    if md and ("❌" in md or "Error" in md):
        return '<div style="display:inline-flex;align-items:center;gap:6px;padding:5px 14px;border-radius:20px;font-size:0.73rem;font-weight:600;background:#FEE2E2;color:#9B1C1C;border:1.5px solid #F87171;">❌ Error</div>'
    return '<div style="display:inline-flex;align-items:center;gap:6px;padding:5px 14px;border-radius:20px;font-size:0.73rem;font-weight:600;background:#F5F3FF;color:#6750A4;border:1.5px solid #C4B8EC;">⚪ Esperando</div>'


# ──────────────────────────────────────────────────────────────────────────────
# 15. CSS — TEMA CLARO (Soft + Lavanda)
# ──────────────────────────────────────────────────────────────────────────────
CSS = """
/* ═══════════════════════════════════════════════════════════════
   TEMA CLARO — Soft + ajustes lavanda/blanco
   ═══════════════════════════════════════════════════════════════ */
body, .gradio-container {
    background: #FAF8FF !important;
    color: #1C1B1F !important;
    font-family: 'Inter', 'Segoe UI', system-ui, sans-serif !important;
}

/* ── Cards y grupos ── */
.gr-group, .gr-box, .panel, .box, [data-testid="block"] {
    background: #FFFFFF !important;
    border: 1.5px solid #E8E0F0 !important;
    border-radius: 14px !important;
    box-shadow: 0 2px 8px rgba(103,80,164,0.06) !important;
}

/* ── Tabs ── */
.tabs, .tab-nav, [role="tablist"], [role="tabpanel"], .tab-content {
    background: #FFFFFF !important;
    border-color: #E8E0F0 !important;
}
button[role="tab"], .tab-nav button {
    color: #6B5E7B !important;
    font-weight: 500 !important;
    padding: 10px 18px !important;
    transition: all 0.15s !important;
}
button[role="tab"]:hover, .tab-nav button:hover {
    color: #6750A4 !important;
    background: #F0EBFF !important;
}
button[role="tab"][aria-selected="true"], button[role="tab"].selected {
    color: #4A0E8F !important;
    border-bottom: 2.5px solid #6750A4 !important;
    font-weight: 700 !important;
}

/* ── Accordion ── */
.accordion, details, details > div, .gr-accordion {
    background: #FFFFFF !important;
    border: 1.5px solid #E8E0F0 !important;
    border-radius: 10px !important;
}
details > summary, .accordion > summary {
    background: #F0EBFF !important;
    color: #1C1B1F !important;
    padding: 11px 14px !important;
    border-radius: 10px !important;
    cursor: pointer !important;
    font-weight: 600 !important;
    font-size: 0.87rem !important;
    list-style: none !important;
}
details[open] > summary { border-radius: 10px 10px 0 0 !important; }
details > summary:hover { background: #E8E1F8 !important; }

/* ── File upload ── */
.file-preview, .upload-container, [data-testid="file"], .gr-file, .upload-box {
    background: #FDFAFF !important;
    border: 2px dashed #C8BCEC !important;
    border-radius: 12px !important;
    color: #49454F !important;
}
.gr-file:hover, [data-testid="file"]:hover {
    background: #EDE9F5 !important;
    border-color: #6750A4 !important;
}

/* ── Inputs ── */
input[type="text"], input[type="number"], input[type="search"],
input[type="email"], textarea, select {
    background: #F0EBFF !important;
    color: #1C1B1F !important;
    border: 1.5px solid #C8BCEC !important;
    border-radius: 8px !important;
    font-family: 'Inter', sans-serif !important;
}
input[type="text"]:focus, input[type="number"]:focus, textarea:focus {
    border-color: #6750A4 !important;
    box-shadow: 0 0 0 3px rgba(103,80,164,0.15) !important;
    outline: none !important;
    background: #FFFFFF !important;
}

/* ── Labels ── */
label, .gr-label, span.svelte-1e4f7tk, .label-wrap > span {
    color: #79747E !important;
    font-size: 0.72rem !important;
    font-weight: 700 !important;
    text-transform: uppercase !important;
    letter-spacing: 0.06em !important;
}

/* ── Botón primario ── */
button.primary, .gr-button-primary, button[variant="primary"] {
    background: linear-gradient(135deg, #6750A4 0%, #9163CF 100%) !important;
    color: #FFFFFF !important;
    border: none !important;
    border-radius: 8px !important;
    font-weight: 700 !important;
    font-size: 0.92rem !important;
    padding: 12px 26px !important;
    box-shadow: 0 2px 10px rgba(103,80,164,0.28) !important;
    transition: all 0.18s ease !important;
    cursor: pointer !important;
}
button.primary:hover, .gr-button-primary:hover {
    background: linear-gradient(135deg, #7965AF 0%, #9E78D4 100%) !important;
    box-shadow: 0 4px 18px rgba(103,80,164,0.38) !important;
    transform: translateY(-1px) !important;
}

/* ── Botón secundario ── */
button.secondary {
    background: #FFFFFF !important;
    color: #6750A4 !important;
    border: 1.5px solid #C8BCEC !important;
    border-radius: 8px !important;
    font-weight: 600 !important;
    transition: all 0.15s !important;
}
button.secondary:hover {
    background: #EDE9F5 !important;
    border-color: #6750A4 !important;
}

/* ── Sliders ── */
input[type="range"] { accent-color: #6750A4 !important; }
.gr-slider, [data-testid="slider"] { background: transparent !important; }

/* ── Radio buttons ── */
input[type="radio"] { accent-color: #6750A4 !important; }
.gr-radio label, [data-testid="radio"] label {
    color: #49454F !important;
    font-size: 0.87rem !important;
    text-transform: none !important;
    font-weight: 500 !important;
    background: transparent !important;
}

/* ── Checkboxes ── */
input[type="checkbox"] {
    -webkit-appearance: checkbox !important;
    appearance: checkbox !important;
    accent-color: #6750A4 !important;
    width: 15px !important;
    height: 15px !important;
    cursor: pointer !important;
    background: #FFFFFF !important;
    border: 1.5px solid #C8BCEC !important;
    border-radius: 3px !important;
}
.gr-checkbox > label, .gr-checkbox label, [data-testid="checkbox"] label {
    display: flex !important;
    align-items: center !important;
    color: #1C1B1F !important;
    font-size: 0.87rem !important;
    font-weight: 500 !important;
    text-transform: none !important;
    cursor: pointer !important;
    gap: 6px !important;
    background: transparent !important;
}

/* ── Markdown ── */
.gr-markdown, .gr-markdown p, .gr-markdown li, .gr-markdown span {
    color: #1C1B1F !important;
    background: transparent !important;
}
.gr-markdown h1, .gr-markdown h2 {
    color: #6750A4 !important;
    font-weight: 800 !important;
    border-bottom: 2px solid #EDE9F5 !important;
    padding-bottom: 6px !important;
}
.gr-markdown h3 { color: #49454F !important; font-weight: 700 !important; }
.gr-markdown code {
    background: #EDE9F5 !important;
    color: #6750A4 !important;
    padding: 2px 6px !important;
    border-radius: 4px !important;
    font-size: 0.80rem !important;
}
.gr-markdown blockquote {
    border-left: 3px solid #6750A4 !important;
    padding: 8px 12px !important;
    background: #F0EBFF !important;
    border-radius: 0 6px 6px 0 !important;
    color: #49454F !important;
}
.gr-markdown table { border-collapse: collapse !important; width: 100% !important; }
.gr-markdown th {
    background: #6750A4 !important;
    color: #FFFFFF !important;
    padding: 10px 14px !important;
    border: none !important;
    font-size: 0.68rem !important;
    text-transform: uppercase !important;
    font-weight: 800 !important;
    letter-spacing: 0.08em !important;
}
.gr-markdown td {
    color: #1C1B1F !important;
    padding: 9px 14px !important;
    border-bottom: 1px solid #F0EBFF !important;
    background: #FFFFFF !important;
}
.gr-markdown tr:nth-child(even) td { background: #FAF8FF !important; }

/* ── Plot containers ── */
.gr-plot, [data-testid="plot"], .plotly, .plot-container {
    background: #FFFFFF !important;
    border: 1.5px solid #E8E0F0 !important;
    border-radius: 10px !important;
}

/* ── Caja de logs ── */
.logs-box {
    background: #FDFAFF !important;
    border: 1.5px solid #E8E0F0 !important;
    border-radius: 8px !important;
    padding: 10px 14px !important;
    font-family: 'JetBrains Mono', 'Courier New', monospace !important;
    font-size: 0.70rem !important;
    color: #6750A4 !important;
    max-height: 140px !important;
    overflow-y: auto !important;
    line-height: 1.75 !important;
}

/* ── GPU badge ── */
.gpu-on {
    background: #DCFCE7; color: #166534; border: 1.5px solid #16A34A;
    padding: 4px 12px; border-radius: 20px; font-size: 0.71rem; font-weight: 700;
}
.gpu-off {
    background: #EDE9F5; color: #6750A4; border: 1.5px solid #C8BCEC;
    padding: 4px 12px; border-radius: 20px; font-size: 0.71rem; font-weight: 700;
}

/* ── FE badge ── */
.fe-badge {
    background: #F0EBFF;
    border: 1.5px solid #C8BCEC;
    border-radius: 8px;
    padding: 8px 12px;
    font-size: 0.73rem;
    color: #49454F;
    margin: 5px 0;
    line-height: 1.6;
}

/* ── IQCA box ── */
.iqca-box {
    background: #F0EBFF;
    border: 1.5px solid #6750A4;
    border-radius: 10px;
    padding: 12px 16px;
    font-size: 0.80rem;
    color: #4A0E8F;
    font-weight: 500;
    line-height: 1.7;
}

/* ── Scrollbar ── */
::-webkit-scrollbar { width: 5px; height: 5px; }
::-webkit-scrollbar-track { background: #FAF8FF; border-radius: 3px; }
::-webkit-scrollbar-thumb { background: #C8BCEC; border-radius: 3px; }
::-webkit-scrollbar-thumb:hover { background: #6750A4; }

/* ── Gradio Dropdown ── */
.wrap.svelte-1ipelgc, ul.options {
    background: #FFFFFF !important;
    border: 1.5px solid #E8E0F0 !important;
}
li.item { color: #1C1B1F !important; }
li.item:hover { background: #EDE9F5 !important; }
"""


# ──────────────────────────────────────────────────────────────────────────────
# 16. UI
# ──────────────────────────────────────────────────────────────────────────────
MODELOS = [
    "HistGradientBoosting (CPU — sin GPU requerida)",
    "XGBoost  (auto GPU/CPU)",
    "LightGBM (auto GPU/CPU)",
]
gpu_badge_html = (
    f'<span class="gpu-on">🟢 GPU: {GPU_MSG.split(": ")[-1]}</span>'
    if GPU_DISPONIBLE else
    f'<span class="gpu-off">🔵 CPU — {GPU_MSG}</span>'
)

def construir_app():
    with gr.Blocks(
        theme=gr.themes.Soft(
            primary_hue="violet",
            secondary_hue="purple",
            neutral_hue="slate",
            font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif"],
        ),
        css=CSS,
        title="Quito Air ML — v10.5-L2",
    ) as app:

        # ── Header ────────────────────────────────────────────────────────────
        gr.HTML(f"""
        <div style="padding:16px 24px 14px;background:#FFFFFF;
                    border-bottom:2px solid #EDE9F5;
                    display:flex;align-items:center;justify-content:space-between;
                    flex-wrap:wrap;gap:10px;">
            <div>
                <h1 style="font-size:1.60rem;font-weight:900;margin:0 0 3px;
                           background:linear-gradient(90deg,#6750A4,#9163CF);
                           -webkit-background-clip:text;-webkit-text-fill-color:transparent;">
                    🌬️ Quito Air ML
                </h1>
                <p style="color:#79747E;font-size:0.78rem;margin:0;font-weight:500;">
                    Surrogate Model · Calidad del Aire · IQCA REMMAQ · v10.5-L2
                </p>
            </div>
            <div>{gpu_badge_html}</div>
        </div>
        <div style="height:12px;background:#FAF8FF;"></div>
        """)

        with gr.Row(equal_height=False):

            # ── SIDEBAR ───────────────────────────────────────────────────────
            with gr.Column(scale=1, min_width=370):

                with gr.Group():
                    gr.Markdown("### 📂 Dataset")
                    csv_upload = gr.File(
                        label="Arrastra o sube tu CSV",
                        file_types=[".csv"], type="filepath",
                    )
                    csv_info = gr.HTML(value="")

                gr.HTML("<div style='height:10px'/>")

                with gr.Group():
                    gr.Markdown("### ⚙️ Configuración del Modelo")
                    with gr.Row():
                        target_input = gr.Dropdown(
                            label="Target (contaminante)",
                            choices=CONTAMINANTES_Y, value="PM25",
                            allow_custom_value=True, scale=3,
                        )
                        nombre_modelo_input = gr.Textbox(
                            label="Nombre modelo (.pkl)",
                            value="surrogate_calidad_aire", scale=3,
                        )
                    algoritmo_radio = gr.Radio(
                        label="Algoritmo de boosting",
                        choices=MODELOS, value=MODELOS[0],
                    )
                    with gr.Row():
                        excluir_pandemia_chk = gr.Checkbox(
                            label="🚫 Excluir pandemia 2020–2021",
                            value=True, scale=1,
                        )
                        usar_fe_ck = gr.Checkbox(
                            label="🧪 Feature Engineering",
                            value=False, scale=1,
                        )
                    gr.HTML(
                        '<div class="fe-badge">'
                        'FE: rolling stats · diffs de lags · interacciones temp/viento'
                        ' · día de semana → top-40 por RandomForest.'
                        '</div>'
                    )
                    with gr.Row():
                        train_ratio_slider = gr.Slider(
                            label="Train / Test split",
                            minimum=0.6, maximum=0.95, step=0.05, value=0.80, scale=3,
                        )
                        k_splits_slider = gr.Slider(
                            label="K — pliegues TSCV",
                            minimum=2, maximum=15, step=1, value=5, scale=2,
                        )
                        plot_days_slider = gr.Slider(
                            label="Días a graficar",
                            minimum=3, maximum=30, step=1, value=7, scale=2,
                        )

                gr.HTML("<div style='height:10px'/>")

                with gr.Accordion("🧪 Hiperparámetros XGBoost / LightGBM / HistGB", open=False):
                    gr.Markdown("_learning_rate, max_depth y reg_lambda aplican también a HistGB._")
                    with gr.Row():
                        max_depth_slider     = gr.Slider(label="max_depth",      minimum=2,    maximum=10,   step=1,    value=5)
                        learning_rate_slider = gr.Slider(label="learning_rate",  minimum=0.01, maximum=0.3,  step=0.01, value=0.05)
                    with gr.Row():
                        subsample_slider        = gr.Slider(label="subsample",        minimum=0.5, maximum=1.0, step=0.05, value=0.8)
                        colsample_bytree_slider = gr.Slider(label="colsample_bytree", minimum=0.5, maximum=1.0, step=0.05, value=0.8)
                    with gr.Row():
                        reg_lambda_slider       = gr.Slider(label="reg_lambda (L2)", minimum=0.0, maximum=10.0, step=0.5, value=2.0)
                        reg_alpha_slider        = gr.Slider(label="reg_alpha  (L1)", minimum=0.0, maximum=5.0,  step=0.1, value=0.5)
                    with gr.Row():
                        gamma_slider            = gr.Slider(label="gamma",            minimum=0.0, maximum=5.0,  step=0.1, value=0.0)
                        min_child_weight_slider = gr.Slider(label="min_child_weight", minimum=1,   maximum=20,   step=1,   value=1)

                gr.HTML("<div style='height:12px'/>")
                btn_train = gr.Button("🚀  Iniciar Entrenamiento", variant="primary", size="lg")
                status_badge = gr.HTML(value=_badge(""))
                gr.Markdown(
                    "<span style='color:#79747E;font-size:0.70rem;font-weight:700;"
                    "text-transform:uppercase;letter-spacing:0.07em;'>🖥️ Log de ejecución</span>"
                )
                estado_output = gr.Markdown(value="_Esperando…_", elem_classes=["logs-box"])

            # ── PANEL DERECHO ─────────────────────────────────────────────────
            with gr.Column(scale=2, min_width=620):
                with gr.Tabs():

                    with gr.Tab("📊 Métricas"):
                        gr.HTML("""
                        <div style="display:grid;grid-template-columns:repeat(3,1fr);gap:10px;
                                    margin:8px 0 16px;">
                            <div style="background:#F5F3FF;border:1.5px solid #DDD6EE;
                                        border-radius:10px;padding:14px 16px;">
                                <div style="font-size:0.60rem;text-transform:uppercase;
                                            letter-spacing:0.08em;color:#79747E;
                                            font-weight:700;margin-bottom:5px;">Test R²</div>
                                <div style="font-size:1.5rem;font-weight:800;color:#6750A4;
                                            font-family:'JetBrains Mono',monospace;">—</div>
                            </div>
                            <div style="background:#F5F3FF;border:1.5px solid #DDD6EE;
                                        border-radius:10px;padding:14px 16px;">
                                <div style="font-size:0.60rem;text-transform:uppercase;
                                            letter-spacing:0.08em;color:#79747E;
                                            font-weight:700;margin-bottom:5px;">Test MAE</div>
                                <div style="font-size:1.5rem;font-weight:800;color:#49454F;
                                            font-family:'JetBrains Mono',monospace;">—</div>
                            </div>
                            <div style="background:#DCFCE7;border:1.5px solid #86EFAC;
                                        border-radius:10px;padding:14px 16px;">
                                <div style="font-size:0.60rem;text-transform:uppercase;
                                            letter-spacing:0.08em;color:#166534;
                                            font-weight:700;margin-bottom:5px;">Estado</div>
                                <div style="font-size:0.82rem;font-weight:700;color:#166534;">
                                    Listo para entrenar
                                </div>
                            </div>
                        </div>
                        """)
                        metricas_output = gr.Markdown(value="*Las métricas aparecerán aquí.*")
                        gr.HTML('<div style="height:8px"></div><hr style="border:none;border-top:1.5px solid #EDE9F5;">')
                        gr.Markdown("**📊 MAE / RMSE comparativo por Contaminante**")
                        fig_mae_output = gr.Plot(label="MAE / RMSE — TSS K=5")

                    with gr.Tab("📈 Real vs Predicho"):
                        fig_pred_output = gr.Plot(label="Serie temporal — Test Ciego")

                    with gr.Tab("📉 Curva Aprendizaje"):
                        gr.Markdown(
                            "Línea **continua** = Train · Línea **discontinua** = Validación TSS.\n"
                            "Banda sombreada = ±1σ entre K pliegues."
                        )
                        fig_lc_output = gr.Plot(label="Loss / Error vs Iteración — TSS")

                    with gr.Tab("📊 Rendimiento R²"):
                        gr.Markdown(
                            "🟢 R²≥0.70 Bueno · 🟡 ≥0.50 Aceptable · 🔴 <0.50 Mejorable"
                        )
                        fig_r2_output = gr.Plot(label="R² por Contaminante")

                    with gr.Tab("🔍 Feature Importance"):
                        gr.Markdown(
                            "Importancia por permutación sobre el **Test Ciego** (20% final)."
                        )
                        fig_fi_output = gr.Plot(label="Permutation Importance (Δ R²)")

                    with gr.Tab("🥧 Distribución"):
                        gr.Markdown(
                            "Distribución **relativa** de importancia — top-10 features."
                        )
                        fig_pie_output = gr.Plot(label="Donut — Top-10 Features")

                    with gr.Tab("📊 Correlaciones"):
                        gr.Markdown(
                            "Correlación de Pearson **solo entre contaminantes**. "
                            "El rectángulo resalta el target activo."
                        )
                        fig_hm_output = gr.Plot(label="Correlación Pearson — Contaminantes")

                    with gr.Tab("🌿 Predicción IQCA"):
                        gr.Markdown(
                            "### 🌿 Predicción + IQCA REMMAQ\n\n"
                            "Solo ingresa variables meteorológicas. "
                            "Los lags se imputan automáticamente.\n\n"
                            "> Entrena primero un modelo."
                        )
                        gr.HTML(
                            '<div class="iqca-box">📌 <b>IQCA:</b> '
                            '0–50 Deseable · 51–100 Aceptable · 101–150 Precaución · '
                            '151–200 Alerta · 201–300 Alarma · 301–500 Emergencia</div>'
                        )
                        with gr.Group():
                            fecha_hora_input = gr.Textbox(
                                label="📅 Fecha y Hora (YYYY-MM-DD HH:MM)",
                                placeholder="2024-03-15 14:00", value="",
                            )
                            with gr.Row():
                                temp_input  = gr.Slider(label="🌡️ Temperatura (°C)",    minimum=-10, maximum=40,  step=0.1, value=18.0)
                                hum_input   = gr.Slider(label="💧 Humedad (%)",          minimum=0,   maximum=100, step=1,   value=70)
                            with gr.Row():
                                vvel_input  = gr.Slider(label="💨 Viento vel. (m/s)",    minimum=0,   maximum=30,  step=0.1, value=2.0)
                                vdir_input  = gr.Slider(label="🧭 Viento dir. (°)",      minimum=0,   maximum=360, step=1,   value=180)
                            precip_input = gr.Slider(
                                label="🌧️ Precipitación (mm)", minimum=0, maximum=200, step=0.1, value=0.0
                            )
                        btn_predecir   = gr.Button("🔮  Estimar Calidad del Aire + IQCA", variant="primary")
                        resultado_pred = gr.Markdown(value="_Entrena primero un modelo._")

        gr.HTML("""
        <div style="text-align:center;padding:14px 0 8px;color:#79747E;font-size:0.71rem;
                    border-top:1.5px solid #EDE9F5;margin-top:8px;background:#FFFFFF;">
            Quito Air ML · Surrogate Model v10.5-L2 · Tema Claro Soft · IQCA REMMAQ
        </div>
        """)

        # ── Eventos ───────────────────────────────────────────────────────────
        csv_upload.change(fn=_resumen_csv_html,
                          inputs=[csv_upload],
                          outputs=[target_input, csv_info])

        btn_train.click(
            fn=lambda: '<div style="display:inline-flex;align-items:center;gap:6px;padding:5px 14px;border-radius:20px;font-size:0.73rem;font-weight:600;background:#EDE9F5;color:#6750A4;border:1.5px solid #C4B8EC;">🔄 Entrenando...</div>',
            inputs=[], outputs=[status_badge], queue=False,
        )
        btn_train.click(
            fn=entrenar,
            inputs=[
                csv_upload, target_input, nombre_modelo_input,
                algoritmo_radio, plot_days_slider, train_ratio_slider,
                excluir_pandemia_chk, k_splits_slider,
                max_depth_slider, learning_rate_slider, subsample_slider,
                colsample_bytree_slider, reg_lambda_slider, reg_alpha_slider,
                gamma_slider, min_child_weight_slider, usar_fe_ck,
            ],
            outputs=[
                metricas_output, fig_mae_output, fig_pred_output,
                fig_lc_output,   fig_r2_output,  fig_fi_output,
                fig_pie_output,  fig_hm_output,  estado_output,
            ],
        ).then(fn=_badge, inputs=[metricas_output], outputs=[status_badge])

        btn_predecir.click(
            fn=predecir_simple,
            inputs=[fecha_hora_input, temp_input, hum_input, vvel_input, vdir_input, precip_input],
            outputs=[resultado_pred],
        )

    return app


# ──────────────────────────────────────────────────────────────────────────────
# 17. PUNTO DE ENTRADA
# ──────────────────────────────────────────────────────────────────────────────
def _ip_fallback():
    import socket
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8",80)); print(f"  🖥️  IP: http://{s.getsockname()[0]}:<PUERTO>"); s.close()
    except Exception: pass

if __name__ == "__main__":
    app = construir_app()
    print("\n" + "═"*66)
    print("  🌬️  Quito Air ML — Surrogate Model  v10.5-L2")
    print("  🎨  CSS White Clean — fondos blancos en todo Gradio")
    print("═"*66)
    print(f"  Hardware: {GPU_MSG} | GPU: {GPU_DISPONIBLE}")
    print(f"  FE top-N: {MAX_FEATURES_SEL} | IQCA REMMAQ activo")
    print("═"*66)
    try:
        app.launch(server_name="0.0.0.0", server_port=None, share=True,
                   max_threads=40, debug=True, show_error=True,
                   prevent_thread_lock=False, quiet=False)
    except OSError:
        app.launch(server_name="0.0.0.0", server_port=7861, share=True, max_threads=40)
    except Exception:
        _ip_fallback()
        app.launch(server_name="0.0.0.0", server_port=None, share=False, max_threads=40)